# Bitget × Bybit — Futures/Futures Spread Trading

**Стратегия:** кросс-биржевой арбитраж фьючерсов
- Лонг фьючерс на бирже A (дешевле) + Шорт на бирже B (дороже)
- Вход: `spread >= entry_threshold`, выход: `spread <= close_threshold`
- Ордера: лимитные FOK на обеих биржах одновременно
- Half-position: если одна нога не исполнилась — ждём, при таймауте закрываем по лучшей цене

## 1. Установка зависимостей

In [ ]:
!pip install requests pandas numpy websocket-client -q

## 2. Импорты и логирование

In [ ]:
import requests, hashlib, hmac, base64, time, warnings
import logging, json, threading
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from IPython.display import display
warnings.filterwarnings('ignore')

import os
from logging.handlers import TimedRotatingFileHandler

LOG_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'logs')
os.makedirs(LOG_DIR, exist_ok=True)

_fmt = logging.Formatter('%(asctime)s.%(msecs)03d [%(levelname)s] %(message)s',
                         datefmt='%H:%M:%S')

# Консоль (видно при подключении к VPS / в tmux)
_console = logging.StreamHandler()
_console.setFormatter(_fmt)

# Файл с ПОЧАСОВОЙ ротацией: каждый час новый файл logs/bot.log.YYYY-MM-DD_HH
_fileh = TimedRotatingFileHandler(
    os.path.join(LOG_DIR, 'bot.log'),
    when='H', interval=1, backupCount=0, encoding='utf-8', utc=False)
_fileh.suffix = '%Y-%m-%d_%H'          # имя ротированного файла: bot.log.2026-05-30_18
_fileh.setFormatter(_fmt)

log = logging.getLogger('pairs')
log.setLevel(logging.INFO)
log.handlers.clear()
log.addHandler(_console)
log.addHandler(_fileh)
log.propagate = False
print(f'Импорты OK. Логи пишутся в {LOG_DIR}/bot.log (ротация каждый час)')

## 3. Конфигурация

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                    BITGET API-КЛЮЧИ                          ║
BITGET_API_KEY    = ''
BITGET_API_SECRET = ''
BITGET_PASSPHRASE = ''
# ╠══════════════════════════════════════════════════════════════╣
# ║                    BYBIT API-КЛЮЧИ                           ║
BYBIT_API_KEY    = ''
BYBIT_API_SECRET = ''
# ╠══════════════════════════════════════════════════════════════╣
# ║                    РЕЖИМ РАБОТЫ                              ║
MODE = 'LIVE'   # 'SIMULATION' или 'LIVE'
# ╠══════════════════════════════════════════════════════════════╣
# ║                    ПАРАМЕТРЫ СТРАТЕГИИ                       ║
SPREAD_CLOSE    = 0.15   # % — закрытие (спред сошёлся)
MAX_POSITIONS    = 3      # одна "свежая" пара одновременно
POS_LIMIT_MINUTES  = 99999  # 0 = сразу; 99999 = слот держится до закрытия     # 0 = слот освобождается СРАЗУ после открытия (бот сразу ищет новую монету)
MAX_USDT        = 20.0   # USDT на одну монету (10 на каждой бирже)
MAX_COINS       = None   # None = без ограничения, мониторим все общие монеты
MIN_VOL_USDT    = 1_000_000  # мин. суточный оборот в USDT
POLL_SEC        = 0.2    # интервал HTTP-polling (fallback)
COMMISSION_PCT  = 0.1    # комиссия за одну ногу, %
TARGET_PROFIT_PCT = 0.2  # желаемая чистая прибыль на пару (%); close_thr = entry_spread - (4*comm + это)
FUNDING_NET_MIN   = 0.02   # мин. нетто-фандинг пары за период (%), иначе вход SKIP
MIN_ENTRY_SPREAD = 0.5   # абсолютный минимум спреда для входа, %
MAX_HALF_MINUTES = 15    # таймаут ожидания второй ноги, мин
LIMIT_WAIT       = 5     # сек ожидания исполнения FOK
# ╠══════════════════════════════════════════════════════════════╣
# ║                    СВЕРКА PnL С БИРЖЕЙ                       ║
VERIFY_PNL_ENABLED   = True   # сверять расчётный PnL с биржевым
PNL_DIFF_THRESHOLD   = 0.05   # USDT — при расхождении больше -> HALT
PNL_FETCH_RETRY      = 6      # попыток для каждого источника (6×5с = 30 сек)
PNL_FETCH_DELAY      = 5.0    # сек задержка перед запросом (биржа ещё пишет)
# ╚══════════════════════════════════════════════════════════════╝

BITGET_BASE = 'https://api.bitget.com'
BYBIT_BASE  = 'https://api.bybit.com'
HALF_USDT   = MAX_USDT / 2

# Минимальный спред должен покрывать 4 ноги комиссий + зазор
MIN_PROFITABLE_SPREAD = COMMISSION_PCT * 4 / 100 * 100 + 0.1  # %
print(f'Mode: {MODE}')
print(f'Half USDT: {HALF_USDT} на каждой бирже')
print(f'Min profitable spread: {MIN_PROFITABLE_SPREAD:.3f}%')
print(f'Close threshold: {SPREAD_CLOSE}%')

# Монеты требующие специальных условий на Bybit (товарные фьючерсы)
_blocked_coins: set = {'CL', 'XAU', 'XAG', 'NG', 'HG', 'ZN', 'ZC', 'ZW', 'ZS'}

BLACKLIST_HOURS_1 = 24    # блок на 1-й и 2-й раз (часов)
BLACKLIST_HOURS_3 = 168   # блок с 3-го раза (7 дней в часах)
# Временный чёрный список: {coin: time.time() когда заблокирована}
_blacklist: dict = {}        # {coin: timestamp}
_blacklist_hits: dict = {}   # {coin: count} — сколько раз попадала
BLACKLIST_FILE   = 'blacklist.json'  # файл для сохранения между запусками

## 4. API-клиенты

In [ ]:
# ════════════════════════════════════════
# BITGET
# ════════════════════════════════════════
BITGET_SESSION = requests.Session()
BITGET_SESSION.headers.update({'Content-Type': 'application/json', 'locale': 'en-US'})

def _bitget_sign(ts, method, path, body=''):
    msg = ts + method.upper() + path + (body or '')
    return base64.b64encode(
        hmac.new(BITGET_API_SECRET.encode(), msg.encode(), hashlib.sha256).digest()
    ).decode()

# Смещение времени Bitget
_bitget_time_offset = 0

def _sync_bitget_time():
    """Синхронизирует время с сервером Bitget."""
    global _bitget_time_offset
    try:
        t_before = int(time.time() * 1000)
        r = BITGET_SESSION.get(BITGET_BASE + '/api/v2/public/time', timeout=5)
        t_after  = int(time.time() * 1000)
        data = r.json().get('data', {})
        server_ts = int(data.get('serverTime') or data.get('timestamp') or 0)
        if server_ts > 0:
            local_mid = (t_before + t_after) // 2
            _bitget_time_offset = server_ts - local_mid
            rtt = t_after - t_before
            log.info(f'Bitget time offset: {_bitget_time_offset:+d}ms (RTT={rtt}ms)')
    except Exception as e:
        log.warning(f'Bitget time sync failed: {e}')

def _bitget_ts():
    return str(int(time.time() * 1000) + _bitget_time_offset)

def _bitget_auth(method, path, params=None, body=''):
    ts = _bitget_ts()
    qs = ('?' + '&'.join(f'{k}={v}' for k, v in (params or {}).items())) if params else ''
    return {
        'ACCESS-KEY':        BITGET_API_KEY,
        'ACCESS-SIGN':       _bitget_sign(ts, method, path + qs, body),
        'ACCESS-TIMESTAMP':  ts,
        'ACCESS-PASSPHRASE': BITGET_PASSPHRASE,
        'Content-Type':      'application/json',
        'locale':            'en-US',
    }

def bitget_get(path, params=None, signed=False):
    from urllib.parse import urlencode
    if signed and params:
        qs_str = urlencode(params)
        r = BITGET_SESSION.get(
            BITGET_BASE + path + '?' + qs_str,
            headers=_bitget_auth('GET', path, params), timeout=10)
    else:
        r = BITGET_SESSION.get(
            BITGET_BASE + path, params=params,
            headers=_bitget_auth('GET', path, params) if signed else {},
            timeout=10)
    r.raise_for_status()
    j = r.json()
    if str(j.get('code', '0')) not in ('0', '00000'):
        raise ValueError(f'Bitget GET {path} [{j.get("code")}]: {j.get("msg")}')
    d = j.get('data', [])
    if isinstance(d, dict):
        for k in ('list', 'items', 'data', 'result'):
            if k in d: return d[k]
        return [d] if d else []
    return d if isinstance(d, list) else []

def bitget_post(path, body: dict):
    body_str = json.dumps(body, ensure_ascii=True)
    r = BITGET_SESSION.post(
        BITGET_BASE + path,
        data=body_str.encode('utf-8'),
        headers=_bitget_auth('POST', path, body=body_str),
        timeout=10)
    j = r.json()
    code = str(j.get('code', '0'))
    if code not in ('0', '00000') or not r.ok:
        if code == '43001':
            log.debug(f'Bitget POST {path} 43001 (filled/not exists)')
        else:
            log.error(f'Bitget POST {path} FAILED [{code}]: {j.get("msg")}')
        raise ValueError(f'Bitget POST {path} [{code}] {j.get("msg")}')
    return j.get('data', {})

# ════════════════════════════════════════
# BYBIT
# ════════════════════════════════════════
BYBIT_SESSION = requests.Session()
BYBIT_SESSION.headers.update({'Content-Type': 'application/json'})

# Смещение локального времени относительно сервера Bybit (мс)
_bybit_time_offset = 0

def _sync_bybit_time():
    """Синхронизирует локальное время с сервером Bybit.
    Учитываем RTT: смещение = server_ts - (t_before + t_after) / 2
    """
    global _bybit_time_offset
    try:
        t_before  = int(time.time() * 1000)
        r         = BYBIT_SESSION.get(BYBIT_BASE + '/v5/market/time', timeout=5)
        t_after   = int(time.time() * 1000)
        server_ts = int(r.json().get('result', {}).get('timeNano', 0)) // 1_000_000
        if server_ts > 0:
            local_mid = (t_before + t_after) // 2
            _bybit_time_offset = server_ts - local_mid
            rtt = t_after - t_before
            log.info(f'Bybit time offset: {_bybit_time_offset:+d}ms (RTT={rtt}ms)')
    except Exception as e:
        log.warning(f'Bybit time sync failed: {e}')

def _bybit_ts():
    """Текущее время скорректированное на смещение сервера."""
    return str(int(time.time() * 1000) + _bybit_time_offset)

def _bybit_sign(payload_str, ts, recv_window=15000):
    pre = str(ts) + BYBIT_API_KEY + str(recv_window) + payload_str
    return hmac.new(BYBIT_API_SECRET.encode(), pre.encode(), hashlib.sha256).hexdigest()

def bybit_get(path, params=None):
    from urllib.parse import urlencode
    params = params or {}
    if abs(_bybit_time_offset) > 500:
        _sync_bybit_time()
    for _att in range(2):
        ts = _bybit_ts()
        rw = 15000
        qs = urlencode(params)
        headers = {
            'X-BAPI-API-KEY':     BYBIT_API_KEY,
            'X-BAPI-SIGN':        _bybit_sign(qs, ts, rw),
            'X-BAPI-TIMESTAMP':   ts,
            'X-BAPI-RECV-WINDOW': str(rw),
        }
        r = BYBIT_SESSION.get(BYBIT_BASE + path, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        j = r.json()
        if j.get('retCode', -1) == 10002 and _att == 0:
            log.warning(f'Bybit GET {path}: 10002 (время) -> пересинхронизация и повтор')
            _sync_bybit_time()
            continue
        if j.get('retCode', -1) != 0:
            raise ValueError(f'Bybit GET {path} [{j.get("retCode")}]: {j.get("retMsg")}')
        result = j.get('result', {})
        return result.get('list', result) if isinstance(result, dict) else result

def bybit_post(path, body: dict):
    # Автосинхронизация если смещение большое; при 10002 — пересинк и один повтор
    if abs(_bybit_time_offset) > 500:
        _sync_bybit_time()
    body_str = json.dumps(body)
    for _att in range(2):
        ts = _bybit_ts()
        rw = 15000
        headers = {
            'X-BAPI-API-KEY':     BYBIT_API_KEY,
            'X-BAPI-SIGN':        _bybit_sign(body_str, ts, rw),
            'X-BAPI-TIMESTAMP':   ts,
            'X-BAPI-RECV-WINDOW': str(rw),
            'Content-Type':       'application/json',
        }
        r = BYBIT_SESSION.post(BYBIT_BASE + path,
                               data=body_str.encode('utf-8'), headers=headers, timeout=10)
        r.raise_for_status()
        j = r.json()
        if j.get('retCode', -1) == 10002 and _att == 0:
            log.warning(f'Bybit POST {path}: 10002 (время) -> пересинхронизация и повтор')
            _sync_bybit_time()
            continue
        if j.get('retCode', -1) != 0:
            log.error(f'Bybit POST {path} FAILED [{j.get("retCode")}]: {j.get("retMsg")}')
            raise ValueError(f'Bybit POST {path} [{j.get("retCode")}] {j.get("retMsg")}')
        return j.get('result', {})

# Синхронизация времени + Ping
_sync_bybit_time()
try:
    import statistics as _st
    _bg = []; _bb = []
    for _ in range(3):
        t0 = time.perf_counter()
        BITGET_SESSION.get(BITGET_BASE + '/api/v2/public/time', timeout=5)
        _bg.append((time.perf_counter()-t0)*1000)
        t0 = time.perf_counter()
        BYBIT_SESSION.get(BYBIT_BASE + '/v5/market/time', timeout=5)
        _bb.append((time.perf_counter()-t0)*1000)
    print(f'Bitget ping: {_st.mean(_bg):.0f}ms')
    print(f'Bybit  ping: {_st.mean(_bb):.0f}ms')
except Exception as e:
    print(f'Ping error: {e}')

# ── Balance helpers (для логов и сверки) ─────────────────────
def get_bg_balance(detailed=False):
    """Доступный USDT на Bitget futures.
    detailed=False: возвращает min из (crossedMaxAvailable, available) -- реально
    доступно для нового ордера; float или None.
    detailed=True: возвращает dict со всеми полями для диагностики."""
    if MODE == 'SIMULATION':
        return None if not detailed else {}
    try:
        bg_acc = bitget_get('/api/v2/mix/account/accounts',
                            params={'productType': 'USDT-FUTURES'}, signed=True)
        bg_d = bg_acc[0] if isinstance(bg_acc, list) and bg_acc else {}
        info = {
            'available':           float(bg_d.get('available', 0) or 0),
            'crossedMaxAvailable': float(bg_d.get('crossedMaxAvailable', 0) or 0),
            'maxOpenPosAvailable': float(bg_d.get('maxOpenPosAvailable', 0) or 0),
            'locked':              float(bg_d.get('locked', 0) or 0),
            'usdtEquity':          float(bg_d.get('usdtEquity', 0) or 0),
            'crossedRiskRate':     float(bg_d.get('crossedRiskRate', 0) or 0),
            'marginMode':          bg_d.get('marginMode', ''),
        }
        if detailed:
            return info
        # Для нового ордера релевантно crossedMaxAvailable (минимум из available
        # и того что осталось после учёта текущих позиций). Если 0 -- fallback.
        cma = info['crossedMaxAvailable']
        return cma if cma > 0 else info['available']
    except Exception as e:
        log.error(f'Bitget balance error: {e}')
        return None

def get_bb_balance():
    """Доступный USDT на Bybit unified. Возвращает float или None."""
    if MODE == 'SIMULATION':
        return None
    try:
        bb_acc = bybit_get('/v5/account/wallet-balance', params={'accountType': 'UNIFIED'})
        bb_list = bb_acc if isinstance(bb_acc, list) else [bb_acc]
        for acc in bb_list:
            for ci in acc.get('coin', []):
                if ci.get('coin') == 'USDT':
                    return float(ci.get('availableToWithdraw', 0) or
                                 ci.get('walletBalance', 0) or 0)
        return 0.0
    except Exception as e:
        log.error(f'Bybit balance error: {e}')
        return None

def log_balances(context=''):
    """Логирует балансы обеих бирж. context -- метка."""
    bg_info = get_bg_balance(detailed=True)
    bb = get_bb_balance()
    if bg_info is None or bg_info == {}:
        bg = None
        bg_s = 'n/a'
        bg_detail = ''
    else:
        bg = bg_info.get('crossedMaxAvailable', 0) or bg_info.get('available', 0)
        bg_s = f'{bg:.4f}'
        bg_detail = (f' (avail={bg_info["available"]:.2f} '
                     f'crossedMax={bg_info["crossedMaxAvailable"]:.2f} '
                     f'maxOpenPos={bg_info["maxOpenPosAvailable"]:.2f} '
                     f'locked={bg_info["locked"]:.2f} '
                     f'equity={bg_info["usdtEquity"]:.2f})')
    bb_s = f'{bb:.4f}' if bb is not None else 'n/a'
    tag = f' [{context}]' if context else ''
    log.warning(f'BALANCE{tag}: BG={bg_s} USDT{bg_detail}, BB={bb_s} USDT, need={HALF_USDT:.2f}')
    if bg is not None and bg < HALF_USDT:
        log.warning(f'BALANCE{tag}: Bitget не хватает {HALF_USDT - bg:.2f} USDT')
    if bb is not None and bb < HALF_USDT:
        log.warning(f'BALANCE{tag}: Bybit не хватает {HALF_USDT - bb:.2f} USDT')
    return bg, bb


## 8. Минимальные требования к ордерам

In [ ]:
_min_req: dict = {}  # coin → {'bg': {...}, 'bb': {...}}

def _load_bg_req(coin):
    req = {'min_qty': 1.0, 'size_mult': 1.0, 'price_place': 5, 'qty_scale': 2}
    try:
        data = bitget_get('/api/v2/mix/market/contracts',
                          params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES'})
        d = data[0] if isinstance(data, list) and data else {}
        if d.get('minTradeNum'):   req['min_qty']     = float(d['minTradeNum'])
        if d.get('pricePlace'):    req['price_place'] = int(d['pricePlace'])
        if d.get('volumePlace'):   req['qty_scale']   = int(d['volumePlace'])
        # Логируем все поля ответа для диагностики contractSize
        size_fields = {f: d.get(f) for f in ['sizeMultiplier', 'multiplier', 'contractSize',
                                               'lotSize', 'quoteAsset', 'baseAsset',
                                               'minTradeNum', 'sizeMultNum', 'baseCoin']
                       if d.get(f) is not None}
        log.debug(f'BG req {coin}: raw fields={size_fields}')
        for f in ['sizeMultiplier', 'multiplier']:
            if d.get(f):
                v = float(d[f])
                req['size_mult'] = v if v >= 1.0 else 1.0
                if req['size_mult'] != 1.0:
                    log.info(f'BG req {coin}: size_mult={req["size_mult"]} (field={f} val={v})')
                break
        # contractSize НЕ используется -- на Bitget это стоимость контракта в USD,
        # а не количество токенов. Для линейных USDT-перпов size_mult всегда 1.
    except Exception as e:
        log.debug(f'BG req {coin}: {e}')
    return req

def _load_bb_req(coin):
    req = {'min_qty': 0.001, 'qty_step': 0.001, 'price_scale': 5}
    try:
        r = BYBIT_SESSION.get(BYBIT_BASE + '/v5/market/instruments-info',
                              params={'category': 'linear', 'symbol': f'{coin}USDT'}, timeout=10)
        r.raise_for_status()
        items = r.json().get('result', {}).get('list', [])
        d = items[0] if items else {}
        lot   = d.get('lotSizeFilter', {})
        price = d.get('priceFilter', {})
        if lot.get('minOrderQty'): req['min_qty']  = float(lot['minOrderQty'])
        if lot.get('qtyStep'):     req['qty_step'] = float(lot['qtyStep'])
        if price.get('tickSize'):
            ts = price['tickSize']
            req['price_scale'] = len(ts.rstrip('0').split('.')[1]) if '.' in ts else 0
    except Exception as e:
        log.debug(f'BB req {coin}: {e}')
    return req

def load_requirements(coins):
    print(f'Загружаем требования к ордерам для {len(coins)} монет...')
    for i, coin in enumerate(coins):
        _min_req[coin] = {'bg': _load_bg_req(coin), 'bb': _load_bb_req(coin)}
        time.sleep(0.05)
        if (i+1) % 20 == 0:
            print(f'  {i+1}/{len(coins)}...')
    print(f'Загружено: {len(_min_req)} монет')

import math as _math

def calc_bg_size(coin, usdt, price):
    r    = _min_req.get(coin, {}).get('bg', {})
    mult = float(r.get('size_mult', 1.0))
    mq   = float(r.get('min_qty',   1.0))
    n    = _math.ceil(max(mq, usdt / price / mult) / mq) * int(mq) if mq >= 1            else _math.ceil(max(mq, usdt / price / mult))
    while n * mult * price < 5.0:
        n += int(mq) if mq >= 1 else 1
    return int(n)

def calc_bb_qty(coin, usdt, price):
    r    = _min_req.get(coin, {}).get('bb', {})
    mq   = float(r.get('min_qty',  0.001))
    step = float(r.get('qty_step', mq))
    dec  = len(f'{step:.10f}'.rstrip('0').split('.')[1]) if '.' in f'{step}' else 0
    qty  = _math.floor((usdt / price) / step) * step
    return round(max(mq, qty), dec)


In [ ]:
# ══════════════════════════════════════════════════════════════
# GTC-ордера для второй ноги HalfPosition
# Лимитный ордер висит до исполнения (GTC = Good Till Cancelled)
# Выставляется один раз, мониторится до HALF_LIMIT_MINUTES
# ══════════════════════════════════════════════════════════════

HALF_LIMIT_MINUTES = 99999  # 0=сразу; 99999=слот держится до закрытия half
PHASE2_START_MIN   = 15  # на этой минуте pending перевыставляется по цене безубытка

def _safe_coid(raw, coin=''):
    """clientOid: биржи принимают только [A-Za-z0-9-_]. Тикеры с иероглифами (龙虾, 牛来, 哈基米)
    валили ордер с 40808 'clientOid is illegal' -> нелатинские символы заменяем коротким хешем тикера."""
    import re as _re, hashlib as _hl
    s = _re.sub(r'[^A-Za-z0-9\-_]', '', str(raw))
    if len(s) < 8 or s != str(raw):                       # что-то вырезали (или почти всё) -> добавим хеш тикера
        h = _hl.md5(str(coin).encode('utf-8')).hexdigest()[:6]
        s = (s or 'spk') + h
    return s[:40]


def _place_gtc_bitget(coin, side, size, price, reduce_only=False, preset_tp=None, preset_sl=None, preset_type='fill_price', client_oid=None, force='gtc'):
    """Лимитный GTC ордер на Bitget futures. Возвращает orderId или ''."""
    if MODE == 'SIMULATION':
        oid = f'sim_bg_gtc_{coin}_{side}_{int(time.time()*1000)}'
        log.info(f'[SIM BG GTC] {side.upper()} {coin} size={size} price={price:.6f}')
        return oid
    r  = _min_req.get(coin, {}).get('bg', {})
    pp = r.get('price_place', 5)
    body = {
        'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
        'marginMode': 'crossed', 'marginCoin': 'USDT',
        'size': str(int(size)), 'side': side,
        'orderType': 'limit', 'price': f'{price:.{pp}f}',
        'force': (force or 'gtc').lower(),  # gtc | ioc
    }
    if reduce_only: body['reduceOnly'] = 'YES'
    # preset TP/SL прямо на ордере входа (в one-way mode единственный рабочий способ);
    # без execPrice => закрытие ПО РЫНКУ на триггере. Для reduce-only/арб-бота не добавляется.
    if preset_tp is not None and preset_sl is not None and not reduce_only:
        body['presetStopSurplusPrice'] = f'{preset_tp:.{pp}f}'
        body['presetStopSurplusType']  = preset_type
        body['presetStopLossPrice']    = f'{preset_sl:.{pp}f}'
        body['presetStopLossType']     = preset_type
        log.info(f'[BG GTC] {coin} preset TP={preset_tp:.{pp}f} SL={preset_sl:.{pp}f} ({preset_type}, маркет)')
    # свой clientOid: если биржа не вернёт orderId (data=null при её лагах), дозапросим ордер по нему
    _coid = _safe_coid(client_oid or f'spk{int(time.time()*1000)}{coin[:6]}', coin)
    body['clientOid'] = _coid
    log.info(f'[BG GTC] {side.upper()} {coin} size={size} price={price:.{pp}f}')
    try:
        resp = bitget_post('/api/v2/mix/order/place-order', body)
        oid  = str((resp or {}).get('orderId') or '')     # 'or' отсекает None -> раньше str(None)='None' ломал сопоставление TP
        if not oid:
            # ордер мог разместиться, а ID не пришёл (VELVET 10.07, IDOL 04.08: detail индексируется с лагом
            # при тормозах биржи) -> ждём дольше + fallback на orders-pending
            for _i, _w in enumerate((0.7, 1.5, 3.0, 5.0, 8.0)):
                time.sleep(_w)
                try:
                    d = bitget_get('/api/v2/mix/order/detail',
                                   params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                           'clientOid': _coid})
                    oid = str((d or {}).get('orderId') or '')
                except Exception:
                    oid = ''
                if not oid:
                    try:
                        p = bitget_get('/api/v2/mix/order/orders-pending',
                                       params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES'})
                        for _o in (p or {}).get('entrustedList') or []:
                            if _o.get('clientOid') == _coid:
                                oid = str(_o.get('orderId') or ''); break
                    except Exception:
                        pass
                if oid:
                    log.info(f'[BG GTC] {coin}: orderId восстановлен по clientOid с {_i + 1}-й попытки')
                    break
            if not oid:
                log.warning(f'[BG GTC] {coin}: биржа не вернула orderId и дозапрос по clientOid пуст — ордер, возможно, не размещён')
        log.info(f'[BG GTC] {coin} orderId={oid}')
        return oid
    except ValueError as e:
        err = str(e)
        if '40786' in err:  # duplicate clientOid: ордер с этим coid УЖЕ на бирже -> восстановим orderId
            try:
                d = bitget_get('/api/v2/mix/order/detail',
                               params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                       'clientOid': _coid})
                _oid = str((d or {}).get('orderId') or '')
                if _oid:
                    log.info(f'[BG GTC] {coin}: duplicate clientOid -> orderId={_oid}')
                    return _oid
            except Exception:
                pass
        if '45119' in err:  # Bitget закрыла открытие позиций (пре-делистинг) -> блок монеты на bg
            _ARBITRON_BLOCKED[(coin, 'bg_fut')] = time.time()
            log.warning(f'[ARBITRON] {coin}/bg_fut: биржа отклонила открытие (45119, пре-делистинг) — монета заблокирована')
        if any(c in err for c in ['40037', '40001', '40006']):
            raise RuntimeError(f'Bitget auth error: {e}')
        if '40804' in err or '45110' in err:
            log.info(f'[BG GTC] {coin}: позиции нет ({err[:30]}) -- пропускаем')
            return 'no_position'
        if '40762' in err:
            bg = get_bg_balance()
            bg_s = f'{bg:.4f}' if bg is not None else 'n/a'
            log.warning(f'[BG GTC] {coin}: недостаточно баланса '
                        f'(BG available={bg_s} USDT, need~{HALF_USDT:.2f})')
            return 'no_balance'
        log.error(f'[BG GTC] {coin}: {e}')
        return ''
    except Exception as e:
        log.error(f'[BG GTC] {coin}: {e}')
        return ''

def _place_gtc_bybit(coin, side, qty, price, reduce_only=False, tif='GTC'):
    """Лимитный GTC ордер на Bybit linear. Возвращает orderId или ''."""
    if MODE == 'SIMULATION':
        oid = f'sim_bb_gtc_{coin}_{side}_{int(time.time()*1000)}'
        log.info(f'[SIM BB GTC] {side.upper()} {coin} qty={qty} price={price:.6f}')
        return oid
    r    = _min_req.get(coin, {}).get('bb', {})
    step = float(r.get('qty_step', 0.001))
    dec  = len(f'{step:.10f}'.rstrip('0').split('.')[1]) if '.' in f'{step}' else 0
    pp   = r.get('price_scale', 5)
    body = {
        'category': 'linear', 'symbol': f'{coin}USDT',
        'side': side, 'orderType': 'Limit',
        'qty': f'{round(qty, dec):.{dec}f}',
        'price': f'{price:.{pp}f}',
        'timeInForce': (tif or 'GTC').upper(),
    }
    if reduce_only: body['reduceOnly'] = True
    log.info(f'[BB GTC] {side.upper()} {coin} qty={qty} price={price:.{pp}f}')
    try:
        resp = bybit_post('/v5/order/create', body)
        oid  = str((resp or {}).get('orderId') or '')     # 'or' отсекает None -> раньше str(None)='None'
        log.info(f'[BB GTC] {coin} orderId={oid}')
        return oid
    except ValueError as e:
        err = str(e)
        if any(c in err for c in ['10003', '10004']):
            raise RuntimeError(f'Bybit auth error: {e}')
        if '110017' in err:
            log.info(f'[BB GTC] {coin}: позиции нет (110017) -- пропускаем')
            return 'no_position'
        if any(c in err for c in ['110007', '170131', '170033', '170034']):
            bb = get_bb_balance()
            bb_s = f'{bb:.4f}' if bb is not None else 'n/a'
            log.warning(f'[BB GTC] {coin}: недостаточно баланса '
                        f'(BB available={bb_s} USDT, need~{HALF_USDT:.2f})')
            return 'no_balance'
        log.error(f'[BB GTC] {coin}: {e}')
        return ''
    except Exception as e:
        log.error(f'[BB GTC] {coin}: {e}')
        return ''

def _cancel_order_bitget(coin, order_id):
    """Отменяет лимитный ордер Bitget."""
    if MODE == 'SIMULATION': return True
    try:
        bitget_post('/api/v2/mix/order/cancel-order', {
            'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
            'orderId': order_id,
        })
        log.warning(f'[BG CANCEL] {coin} oid={order_id}')
        return True
    except Exception as e:
        log.warning(f'[BG CANCEL] {coin}: {e}')
        return False

def _cancel_order_bybit(coin, order_id):
    """Отменяет лимитный ордер Bybit."""
    if MODE == 'SIMULATION': return True
    try:
        bybit_post('/v5/order/cancel', {
            'category': 'linear', 'symbol': f'{coin}USDT', 'orderId': order_id,
        })
        log.warning(f'[BB CANCEL] {coin} oid={order_id}')
        return True
    except Exception as e:
        log.warning(f'[BB CANCEL] {coin}: {e}')
        return False

print('GTC/cancel функции готовы')


## 10. Ожидание исполнения ордеров

In [ ]:
def _wait_filled_bg(oid, coin, timeout=LIMIT_WAIT):
    """Поллинг статуса ордера Bitget. True=исполнен, False=отменён/таймаут."""
    if MODE == 'SIMULATION': return True
    if oid == 'filled_instantly': return True
    from urllib.parse import urlencode
    deadline = time.time() + timeout
    params = {'symbol': f'{coin}USDT', 'orderId': oid, 'productType': 'usdt-futures'}
    url = BITGET_BASE + '/api/v2/mix/order/detail?' + urlencode(params)
    while time.time() < deadline:
        try:
            r = BITGET_SESSION.get(url, headers=_bitget_auth('GET', '/api/v2/mix/order/detail', params), timeout=5)
            d = r.json().get('data', {})
            if isinstance(d, list): d = d[0] if d else {}
            state = str(d.get('state', '') or d.get('status', '')).lower()
            if state in ('filled', 'full_fill', 'partially_filled'): return True
            if state in ('cancelled', 'canceled'): return False
        except ValueError as e:
            if '43001' in str(e): return True
        except: pass
        time.sleep(0.05)
    return False

def _wait_filled_bb(oid, coin, timeout=LIMIT_WAIT):
    """Поллинг статуса ордера Bybit. True=исполнен, False=отменён/таймаут."""
    if MODE == 'SIMULATION': return True
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            # Сначала realtime (активные ордера)
            data = bybit_get('/v5/order/realtime',
                             params={'category': 'linear', 'symbol': f'{coin}USDT', 'orderId': oid})
            items = data if isinstance(data, list) else []
            if not items:
                # FOK мог сразу уйти в историю
                hist = bybit_get('/v5/order/history',
                                 params={'category': 'linear', 'symbol': f'{coin}USDT',
                                         'orderId': oid, 'limit': '1'})
                items = hist if isinstance(hist, list) else []
            if items:
                st  = str(items[0].get('orderStatus', '')).lower()
                qty = float(items[0].get('qty', 0) or 0)
                cum = float(items[0].get('cumExecQty', 0) or 0)
                if st == 'filled': return True
                if st == 'partiallyfilled':
                    log.debug(f'BB partial fill {oid}: {cum}/{qty}')
                if st in ('cancelled', 'rejected'): return False
        except: pass
        time.sleep(0.05)
    return False

def _get_fill_bg(oid, coin, sim_price=None):
    """Фактическая цена исполнения Bitget."""
    if MODE == 'SIMULATION': return {'avg_price': sim_price} if sim_price else {}
    from urllib.parse import urlencode
    for attempt in range(3):
        try:
            time.sleep(0.3 * (attempt + 1))
            params = {'symbol': f'{coin}USDT', 'orderId': oid, 'productType': 'usdt-futures'}
            url = BITGET_BASE + '/api/v2/mix/order/detail?' + urlencode(params)
            r = BITGET_SESSION.get(url, headers=_bitget_auth('GET', '/api/v2/mix/order/detail', params), timeout=10)
            d = r.json().get('data', {})
            if isinstance(d, list): d = d[0] if d else {}
            for f in ['priceAvg', 'avgFillPrice', 'fillPrice']:
                v = d.get(f)
                if v:
                    try:
                        fee = abs(float(d.get('totalFee', 0) or d.get('fee', 0) or 0))
                        ts  = int(d.get('uTime', 0) or d.get('cTime', 0) or 0)
                        return {'avg_price': float(v),
                                'qty': float(d.get('baseVolume', 0) or 0),
                                'fee': fee,
                                'ts': ts}
                    except: pass
        except: pass
    return {}


def _bg_position_exists(coin) -> bool:
    """REST-проверка: есть ли открытая позиция по монете на Bitget.
    Используется как fallback к WS для детекта срабатывания SL.
    Возвращает True если позиция есть (total>0), False если нет/ошибка-неизвестно."""
    if MODE == 'SIMULATION':
        return True  # в симуляции считаем что есть
    try:
        from urllib.parse import urlencode
        params = {'symbol': f'{coin}USDT', 'productType': 'usdt-futures', 'marginCoin': 'USDT'}
        url = BITGET_BASE + '/api/v2/mix/position/single-position?' + urlencode(params)
        r = BITGET_SESSION.get(url, headers=_bitget_auth('GET', '/api/v2/mix/position/single-position', params), timeout=10)
        d = r.json().get('data', [])
        rows = d if isinstance(d, list) else [d]
        for p in rows:
            if float(p.get('total', 0) or 0) > 0:
                return True
        return False
    except Exception as e:
        log.debug(f'_bg_position_exists {coin}: {e}')
        return True  # при ошибке считаем что есть (не закрываем по ошибке связи)


def _bg_position_info(coin) -> dict:
    """Инфо по открытой позиции Bitget: {'avg': цена, 'total': токены, 'side': 'long'|'short'}.
    {} если позиции нет. {'_error': True} при ошибке связи -- чтобы НЕ строить фантом по таймауту."""
    if MODE == 'SIMULATION':
        return {}
    try:
        from urllib.parse import urlencode
        params = {'symbol': f'{coin}USDT', 'productType': 'usdt-futures', 'marginCoin': 'USDT'}
        url = BITGET_BASE + '/api/v2/mix/position/single-position?' + urlencode(params)
        r = BITGET_SESSION.get(url, headers=_bitget_auth('GET', '/api/v2/mix/position/single-position', params), timeout=10)
        d = r.json().get('data', [])
        rows = d if isinstance(d, list) else [d]
        for p in rows:
            tot = float(p.get('total', 0) or 0)
            if tot > 0:
                avg = float(p.get('openPriceAvg', 0) or p.get('averageOpenPrice', 0)
                            or p.get('openAvgPrice', 0) or 0)
                side = str(p.get('holdSide', '') or '').lower()
                return {'avg': avg, 'total': tot, 'side': side}
        return {}
    except Exception as e:
        log.debug(f'_bg_position_info {coin}: {e}')
        return {'_error': True}


def _get_fill_bb(oid, coin, sim_price=None):
    """Фактическая цена исполнения Bybit."""
    if MODE == 'SIMULATION': return {'avg_price': sim_price} if sim_price else {}
    for attempt in range(3):
        try:
            time.sleep(0.3 * (attempt + 1))
            for endpoint, params in [
                ('/v5/order/realtime', {'category': 'linear', 'symbol': f'{coin}USDT', 'orderId': oid}),
                ('/v5/order/history',  {'category': 'linear', 'symbol': f'{coin}USDT', 'orderId': oid, 'limit': '1'}),
            ]:
                data = bybit_get(endpoint, params=params)
                items = data if isinstance(data, list) else []
                if items:
                    d = items[0]
                    avg = float(d.get('avgPrice', 0) or 0)
                    _cum = float(d.get('cumExecQty', 0) or 0)      # 0 = ордер отменён/не исполнен
                    if avg > 0 and _cum > 0:
                        fee = abs(float(d.get('cumExecFee', 0) or 0))
                        ts  = int(d.get('updatedTime', 0) or d.get('createdTime', 0) or 0)
                        return {'avg_price': avg,
                                'qty': _cum,
                                'fee': fee,
                                'ts': ts}
        except: pass
    return {}

print('Order waiter функции готовы')

def _log_drawdown_8h(coin: str, direction: str, entry_price: float):
    """Считает максимальную просадку ПРОТИВ позиции за последние 8 часов и пишет в лог.
    Использует 1-минутные свечи Bitget.

    direction=A (long BG)  -> просадка = (entry - min_low)  / entry × 100  (цена падала)
    direction=B (short BG) -> просадка = (max_high - entry) / entry × 100  (цена росла)
    """
    HOURS = 8

    def _fetch():
        try:
            r  = _min_req.get(coin, {}).get('bg', {})
            pp = r.get('price_place', 5)

            end_ms   = int(time.time() * 1000)
            start_ms = end_ms - HOURS * 3600 * 1000
            data = bitget_get('/api/v2/mix/market/candles',
                              params={'symbol': f'{coin}USDT',
                                      'productType': 'USDT-FUTURES',
                                      'granularity': '1m',
                                      'startTime': str(start_ms),
                                      'endTime':   str(end_ms),
                                      'limit':     '1000'})
            if not data:
                log.info(f'[DD-8h] {coin}: свечи пусты')
                return

            # формат Bitget candles: [ts, open, high, low, close, volume, ...]
            highs, lows = [], []
            for row in data:
                try:
                    highs.append(float(row[2]))
                    lows.append(float(row[3]))
                except (IndexError, ValueError, TypeError):
                    pass
            if not highs or not lows:
                log.info(f'[DD-8h] {coin}: не удалось распарсить свечи')
                return

            max_high = max(highs)
            min_low  = min(lows)
            n = len(highs)

            if direction == 'A':
                dd_pct = (entry_price - min_low) / entry_price * 100
                worst_px = min_low
                kind = 'low'
            else:
                dd_pct = (max_high - entry_price) / entry_price * 100
                worst_px = max_high
                kind = 'high'

            log.info(f'[DD-8h] {coin} dir={direction}: entry={entry_price:.{pp}f}, '
                     f'8ч {kind}={worst_px:.{pp}f}, max drawdown против позиции = {dd_pct:.3f}% '
                     f'({n} свечей)')
        except Exception as e:
            log.warning(f'[DD-8h] {coin}: ошибка: {e}')

    threading.Thread(target=_fetch, daemon=True, name=f'dd8h-{coin}').start()


def _fetch_funding_bg(coin, start_ms, end_ms):
    """Накопленный funding по позиции Bitget (signed: + получили, - заплатили).
    ИСТОЧНИК: журнал счёта (bill, businessType contract_settle_fee) за окно
    [start_ms, end_ms]. Фандинг на Bitget лежит в биллах, а НЕ в объекте позиции.
    FALLBACK: history-position, если биллы пусты."""
    if MODE == 'SIMULATION':
        return 0.0
    sym = f'{coin}USDT'
    # ИСТОЧНИК: журнал счёта (bill) -- фандинг на Bitget лежит в биллах
    # (contract_settle_fee), а НЕ в объекте позиции. Поэтому берём из bill за окно.
    try:
        data = bitget_get('/api/v2/mix/account/bill',
            params={'productType': 'USDT-FUTURES', 'coin': 'USDT',
                    'startTime': str(start_ms), 'endTime': str(end_ms + 30000),
                    'limit': '100'}, signed=True)
        if isinstance(data, dict):
            bills = data.get('bills') or data.get('list') or []
        elif isinstance(data, list):
            bills = data
        else:
            bills = []
        total = 0.0; found = False
        for b in bills:
            if str(b.get('symbol', '')).upper() != sym.upper():
                continue
            bt = str(b.get('businessType', '')).lower()
            if 'settle' in bt or 'fund' in bt:
                total += float(b.get('amount', 0) or 0); found = True
        if found:
            return total
    except Exception as e:
        log.debug(f'_fetch_funding_bg bill {coin}: {e}')
    # ── FALLBACK 2: history-position (только закрытые) ──
    try:
        data = bitget_get('/api/v2/mix/position/history-position',
            params={'productType': 'USDT-FUTURES', 'symbol': sym,
                    'startTime': str(start_ms), 'endTime': str(end_ms + 30000),
                    'limit': '20'}, signed=True)
        items = data.get('list', []) if isinstance(data, dict) else (data if isinstance(data, list) else [])
        total = 0.0
        for pos in items:
            utime = int(pos.get('utime', 0) or 0)
            if utime < start_ms or utime > end_ms + 60000:
                continue
            total += float(pos.get('totalFunding', 0) or 0)
        return total
    except Exception as e:
        log.debug(f'_fetch_funding_bg hist {coin}: {e}')
        return 0.0


def _fetch_funding_bb(coin, start_ms, end_ms):
    """Суммарный funding fee по позиции Bybit за окно (transaction-log, type=SETTLEMENT).
    Использует bybit_get (signed). Возвращает float (signed) или 0.0."""
    if MODE == 'SIMULATION':
        return 0.0
    try:
        data = bybit_get('/v5/account/transaction-log',
            params={'category': 'linear', 'symbol': f'{coin}USDT',
                    'type': 'SETTLEMENT',
                    'startTime': str(start_ms), 'endTime': str(end_ms + 30000),
                    'limit': '50'})
        # bybit_get уже извлекает 'list' -> data это список (или dict при иной структуре)
        rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
        total = 0.0
        for row in rows:
            f = row.get('funding')
            if f is None:
                f = row.get('change', 0)
            total += float(f or 0)
        return total
    except Exception as e:
        log.debug(f'_fetch_funding_bb {coin}: {e}')
        return 0.0


def _fetch_funding_rates(coin):
    """Текущие ставки фандинга обеих бирж.
    Возвращает (rate_bg, rate_bb) -- signed доли (0.0001 = 0.01%),
    либо None по соответствующей бирже при ошибке."""
    if MODE == 'SIMULATION':
        return 0.0, 0.0
    sym = f'{coin}USDT'
    rate_bg = None
    rate_bb = None
    try:
        tk = bitget_get('/api/v2/mix/market/ticker',
                        params={'productType': 'USDT-FUTURES', 'symbol': sym})
        rows = tk if isinstance(tk, list) else [tk]
        for t in rows:
            if str(t.get('symbol', '')).upper() == sym.upper():
                fr = t.get('fundingRate')
                if fr is not None and fr != '':
                    rate_bg = float(fr)
                break
    except Exception as e:
        log.debug(f'_fetch_funding_rates BG {coin}: {e}')
    try:
        d = bybit_get('/v5/market/tickers', {'category': 'linear', 'symbol': sym})
        # bybit_get уже возвращает result['list'] (список) либо result(dict)
        if isinstance(d, list):
            lst = d
        elif isinstance(d, dict):
            lst = d.get('list', [d])
        else:
            lst = []
        for t in lst:
            if str(t.get('symbol', '')).upper() == sym.upper() or len(lst) == 1:
                fr = t.get('fundingRate')
                if fr is not None and fr != '':
                    rate_bb = float(fr)
                break
    except Exception as e:
        log.debug(f'_fetch_funding_rates BB {coin}: {e}')
    return rate_bg, rate_bb


def _fetch_achieved_bg(coin):
    """Накопленный фандинг открытой позиции Bitget (signed: + получили, - заплатили).
    Источник: журнал счёта (account/bill), записи businessType='contract_settle_fee'
    по символу, за окно [cTime позиции ... сейчас]. cTime берётся из позиции, поэтому
    не зависит от рестартов бота. achievedProfits для ОТКРЫТОЙ позиции всегда '0',
    поэтому используем bill."""
    if MODE == 'SIMULATION':
        return 0.0
    sym = f'{coin}USDT'
    # 1) время открытия позиции (cTime) -> начало окна
    _start_ms = 0
    try:
        dp = bitget_get('/api/v2/mix/position/all-position',
                        params={'productType': 'USDT-FUTURES', 'marginCoin': 'USDT'}, signed=True)
        prows = dp.get('list', []) if isinstance(dp, dict) else (dp if isinstance(dp, list) else [])
        for r in prows:
            if str(r.get('symbol', '')).upper() == sym.upper():
                _start_ms = int(r.get('cTime', 0) or 0)
                break
    except Exception as e:
        log.debug(f'_fetch_achieved_bg pos {coin}: {e}')
    if not _start_ms:
        _start_ms = int(time.time() * 1000) - 24 * 3600 * 1000   # fallback: сутки
    # 2) журнал bill -> сумма фандинга (contract_settle_fee) по символу за окно
    total = 0.0
    try:
        _now = int(time.time() * 1000)
        d = bitget_get('/api/v2/mix/account/bill',
                       params={'productType': 'USDT-FUTURES',
                               'startTime': str(_start_ms), 'endTime': str(_now),
                               'limit': '100'}, signed=True)
        # ответ может быть dict{'bills':[...]} ИЛИ list[ dict{'bills':[...]} ] ИЛИ list[bill]
        bills = []
        if isinstance(d, dict):
            bills = d.get('bills') or d.get('list') or []
        elif isinstance(d, list):
            for el in d:
                if isinstance(el, dict) and 'bills' in el:
                    bills.extend(el['bills'])
                elif isinstance(el, dict):
                    bills.append(el)
        for b in bills:
            if str(b.get('symbol', '')).upper() != sym.upper():
                continue
            if str(b.get('businessType', '')).lower() == 'contract_settle_fee':
                total += float(b.get('amount', 0) or 0)
    except Exception as e:
        log.debug(f'_fetch_achieved_bg bill {coin}: {e}')
    return total


def _fetch_funding_stats_bg(coin):
    """Как _fetch_achieved_bg, но возвращает (total, n_pos, n_neg):
    сумму знакового фандинга и количество положительных / отрицательных выплат
    (записи businessType='contract_settle_fee' по символу за окно с cTime позиции)."""
    if MODE == 'SIMULATION':
        return 0.0, 0, 0
    sym = f'{coin}USDT'
    _start_ms = 0
    try:
        dp = bitget_get('/api/v2/mix/position/all-position',
                        params={'productType': 'USDT-FUTURES', 'marginCoin': 'USDT'}, signed=True)
        prows = dp.get('list', []) if isinstance(dp, dict) else (dp if isinstance(dp, list) else [])
        for r in prows:
            if str(r.get('symbol', '')).upper() == sym.upper():
                _start_ms = int(r.get('cTime', 0) or 0)
                break
    except Exception as e:
        log.debug(f'_fetch_funding_stats_bg pos {coin}: {e}')
    if not _start_ms:
        _start_ms = int(time.time() * 1000) - 24 * 3600 * 1000
    total = 0.0; n_pos = 0; n_neg = 0
    try:
        _now = int(time.time() * 1000)
        d = bitget_get('/api/v2/mix/account/bill',
                       params={'productType': 'USDT-FUTURES',
                               'startTime': str(_start_ms), 'endTime': str(_now),
                               'limit': '100'}, signed=True)
        bills = []
        if isinstance(d, dict):
            bills = d.get('bills') or d.get('list') or []
        elif isinstance(d, list):
            for el in d:
                if isinstance(el, dict) and 'bills' in el:
                    bills.extend(el['bills'])
                elif isinstance(el, dict):
                    bills.append(el)
        for b in bills:
            if str(b.get('symbol', '')).upper() != sym.upper():
                continue
            if str(b.get('businessType', '')).lower() == 'contract_settle_fee':
                amt = float(b.get('amount', 0) or 0)
                total += amt
                if amt > 0:   n_pos += 1
                elif amt < 0: n_neg += 1
    except Exception as e:
        log.debug(f'_fetch_funding_stats_bg bill {coin}: {e}')
    return total, n_pos, n_neg


def _fetch_funding_stats_bb(coin):
    """Как _fetch_funding_stats_bg, но для Bybit: (total, n_pos, n_neg).
    Старт окна -- createdTime позиции Bybit (а не half.open_time, который
    сбрасывается при рестарте). Источник выплат: transaction-log type=SETTLEMENT."""
    if MODE == 'SIMULATION':
        return 0.0, 0, 0
    sym = f'{coin}USDT'
    _start_ms = 0
    try:
        dp = bybit_get('/v5/position/list',
                       params={'category': 'linear', 'symbol': sym})
        prows = dp if isinstance(dp, list) else (dp.get('list', []) if isinstance(dp, dict) else [])
        for r in prows:
            if str(r.get('symbol', '')).upper() == sym.upper() and float(r.get('size', 0) or 0) > 0:
                _start_ms = int(r.get('createdTime', 0) or 0)
                break
    except Exception as e:
        log.debug(f'_fetch_funding_stats_bb pos {coin}: {e}')
    if not _start_ms:
        _start_ms = int(time.time() * 1000) - 24 * 3600 * 1000
    total = 0.0; n_pos = 0; n_neg = 0
    try:
        _now = int(time.time() * 1000)
        data = bybit_get('/v5/account/transaction-log',
            params={'category': 'linear', 'symbol': sym, 'type': 'SETTLEMENT',
                    'startTime': str(_start_ms), 'endTime': str(_now), 'limit': '50'})
        rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
        for row in rows:
            f = row.get('funding')
            if f is None:
                f = row.get('change', 0)
            amt = float(f or 0)
            total += amt
            if amt > 0:   n_pos += 1
            elif amt < 0: n_neg += 1
    except Exception as e:
        log.debug(f'_fetch_funding_stats_bb bill {coin}: {e}')
    return total, n_pos, n_neg


def _funding_entry_ok(coin, direction):
    """Фильтр входа по фандингу.
    Разрешаем вход только если:
      1) BG-нога ПОЛУЧАЕТ фандинг (cashflow >= 0):
         dir A (BG long)  -> rate_bg <= 0
         dir B (BG short)  -> rate_bg >= 0
      2) НЕТТО-фандинг пары (bg_cash + bb_cash) >= FUNDING_NET_MIN %.
    Возвращает (ok: bool, reason: str, rate_bg, rate_bb).
    Если ставки недоступны -- пропускаем (ok=True), чтобы не блокировать
    торговлю на сбое API (лог предупреждения)."""
    r_bg, r_bb = _fetch_funding_rates(coin)
    if r_bg is None or r_bb is None:
        return True, f'ставки фандинга недоступны (bg={r_bg} bb={r_bb}) -- пропускаю фильтр', r_bg, r_bb
    # cashflow по ногам (long платит при rate>0; short получает при rate>0)
    if direction == 'A':       # BG long, BB short
        bg_cash =  -r_bg
        bb_cash =   r_bb
    else:                      # BG short, BB long
        bg_cash =   r_bg
        bb_cash =  -r_bb
    cond_bg = bg_cash >= 0
    net = (bg_cash + bb_cash) * 100   # нетто-фандинг пары за период, %
    cond_net = net >= 0               # нетто-фандинг пары >= 0 (ровно ноль допустим)
    ok = cond_bg and cond_net
    reason = (f'bg_rate={r_bg*100:+.4f}% (cash={bg_cash*100:+.4f}%) '
              f'bb_rate={r_bb*100:+.4f}% (cash={bb_cash*100:+.4f}%) '
              f'net={net:+.4f}% (нужно net>=0) '
              f'cond_bg={cond_bg} cond_net={cond_net}')
    return ok, reason, r_bg, r_bb


# ══════════════════════════════════════════════════════════════════
# СПОТ (Этап 1): инструменты + цены. Только данные, без торговли.
# Используется для dir B второй ноги (спот-лонг как альтернатива perp-лонгу).
# ══════════════════════════════════════════════════════════════════
_spot_specs_cache: dict = {}   # coin -> {'bb': {...}|None, 'bg': {...}|None}

def _spot_specs_bb(coin):
    """Спот-инструмент Bybit. dict со спецификой или None (нет спот-рынка)."""
    sym = f'{coin}USDT'
    try:
        d = bybit_get('/v5/market/instruments-info',
                      {'category': 'spot', 'symbol': sym})
        lst = d if isinstance(d, list) else (d.get('list', []) if isinstance(d, dict) else [])
        if not lst:
            return None
        it = lst[0]
        lot = it.get('lotSizeFilter', {}) or {}
        prc = it.get('priceFilter', {}) or {}
        base_prec = lot.get('basePrecision') or lot.get('qtyStep') or '0.000001'
        tick = prc.get('tickSize') or '0.00001'
        def _dec(x):
            x = str(x)
            return len(x.split('.')[1].rstrip('0')) if '.' in x else 0
        return {
            'min_qty':   float(lot.get('minOrderQty', 0) or 0),
            'qty_step':  float(base_prec),
            'qty_dec':   _dec(base_prec),
            'tick':      float(tick),
            'price_dec': _dec(tick),
            'min_amt':   float(lot.get('minOrderAmt', 0) or 0),  # мин. сумма в USDT
        }
    except Exception as e:
        log.debug(f'_spot_specs_bb {coin}: {e}')
        return None

def _spot_specs_bg(coin):
    """Спот-инструмент Bitget. dict со спецификой или None (нет спот-рынка)."""
    sym = f'{coin}USDT'
    try:
        d = bitget_get('/api/v2/spot/public/symbols', params={'symbol': sym})
        lst = d if isinstance(d, list) else (d.get('list', []) if isinstance(d, dict) else [])
        it = None
        for x in lst:
            if str(x.get('symbol', '')).upper() == sym.upper():
                it = x; break
        if it is None and lst:
            it = lst[0] if str(lst[0].get('symbol','')).upper()==sym.upper() else None
        if it is None:
            return None
        pp = int(it.get('pricePrecision', 5) or 5)
        qp = int(it.get('quantityPrecision', 4) or 4)
        return {
            'min_qty':   float(it.get('minTradeAmount', 0) or 0),
            'qty_dec':   qp,
            'qty_step':  10 ** (-qp),
            'price_dec': pp,
            'tick':      10 ** (-pp),
            'min_amt':   float(it.get('minTradeUSDT', 0) or 0),
        }
    except Exception as e:
        log.debug(f'_spot_specs_bg {coin}: {e}')
        return None

_VOL_SPIKE_MULT = 2.0       # порог: минутка за 5 мин > MULT × средней минутки за час
_vol_spike_cache = {}        # coin -> (ts, (is_spike, ratio, n))

def _volume_spike_bg(coin, mult=_VOL_SPIKE_MULT):
    """Всплеск объёма по 1-мин свечам Bitget futures.
    True, если объём ХОТЯ БЫ ОДНОЙ минутки за последние 5 мин превышает
    mult × средний минутный объём за последний час. Возвращает
    (is_spike, ratio, n_candles). При ошибке -> (False, ...) -- НЕ блокируем."""
    now = time.time()
    c = _vol_spike_cache.get(coin)
    if c and now - c[0] < 20:
        return c[1]
    res = (False, 0.0, 0)
    try:
        end_ms = int(now * 1000); start_ms = end_ms - 65 * 60 * 1000
        data = bitget_get('/api/v2/mix/market/candles',
                          params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                  'granularity': '1m', 'startTime': str(start_ms),
                                  'endTime': str(end_ms), 'limit': '100'})
        rows = data if isinstance(data, list) else []
        cand = []
        for r in rows:
            try:
                ts = int(r[0]); v = float(r[6]) if len(r) > 6 else float(r[5])  # quoteVol(USDT) | baseVol
                cand.append((ts, v))
            except Exception:
                pass
        if len(cand) < 10:
            res = (False, 0.0, len(cand))            # данных мало -> не блокируем
        else:
            cand.sort(key=lambda x: x[0])
            vols = [v for _, v in cand]
            hour = vols[-60:] if len(vols) >= 60 else vols
            avg  = (sum(hour) / len(hour)) if hour else 0.0
            mx5  = max(vols[-5:]) if len(vols) >= 5 else (max(vols) if vols else 0.0)
            ratio = (mx5 / avg) if avg > 0 else 0.0
            res = (ratio > mult, ratio, len(cand))
    except Exception as e:
        log.debug(f'_volume_spike_bg {coin}: {e}')
        res = (False, 0.0, 0)
    _vol_spike_cache[coin] = (now, res)
    return res


_funding_interval_cache = {}   # coin -> hours

_dd_closes_cache = {}   # coin -> (ts, [close,...])

def _minute_closes_bg(coin, n=65):
    """Закрытия 1-мин свечей Bitget futures (последние n минут). Кэш 30с."""
    now = time.time()
    c = _dd_closes_cache.get(coin)
    if c and now - c[0] < 30:
        return c[1]
    closes = []
    try:
        end_ms = int(now * 1000); start_ms = end_ms - (n + 2) * 60 * 1000
        data = bitget_get('/api/v2/mix/market/candles',
                          params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                  'granularity': '1m', 'startTime': str(start_ms),
                                  'endTime': str(end_ms), 'limit': '100'})
        rows = data if isinstance(data, list) else []
        tmp = []
        for r in rows:
            try:
                tmp.append((int(r[0]), float(r[4])))   # close = index 4
            except Exception:
                pass
        tmp.sort(key=lambda x: x[0])
        closes = [v for _, v in tmp][-n:]
    except Exception as e:
        log.debug(f'_minute_closes_bg {coin}: {e}')
    _dd_closes_cache[coin] = (now, closes)
    return closes

def _max_dd_bg(coin, is_long, window=60):
    """Макс. просадка ноги за последние window мин по 1-мин закрытиям (в %),
    в НЕБЛАГОПРИЯТНУЮ для ноги сторону: long -- падение от пика, short -- рост от дна."""
    closes = _minute_closes_bg(coin, n=window + 1)
    if len(closes) < 2:
        return 0.0
    dd = 0.0
    if is_long:
        peak = closes[0]
        for p in closes:
            if p > peak: peak = p
            if peak > 0:
                dd = max(dd, (peak - p) / peak * 100)
    else:
        trough = closes[0]
        for p in closes:
            if p < trough: trough = p
            if trough > 0:
                dd = max(dd, (p - trough) / trough * 100)
    return dd


def _funding_interval_hours(coin):
    """Интервал выплат фандинга монеты в часах (Bybit instruments-info.fundingInterval,
    минуты -> часы). Кэшируется (почти не меняется). По умолчанию 8ч при ошибке."""
    if coin in _funding_interval_cache:
        return _funding_interval_cache[coin]
    h = 8.0
    try:
        d = bybit_get('/v5/market/instruments-info',
                      {'category': 'linear', 'symbol': f'{coin}USDT'})
        lst = d if isinstance(d, list) else (d.get('list', []) if isinstance(d, dict) else [])
        if lst:
            fi = lst[0].get('fundingInterval')
            if fi:
                h = float(fi) / 60.0
    except Exception as e:
        log.debug(f'_funding_interval_hours {coin}: {e}')
    _funding_interval_cache[coin] = h
    return h


def get_spot_specs(coin):
    """Кэш спот-спецификаций обеих бирж. {'bb':..|None, 'bg':..|None}."""
    if coin not in _spot_specs_cache:
        _spot_specs_cache[coin] = {'bb': _spot_specs_bb(coin), 'bg': _spot_specs_bg(coin)}
    return _spot_specs_cache[coin]

def _fetch_spot_prices(coin):
    """REST-снимок лучших спот bid/ask обеих бирж (цены спота != перпа).
    Возвращает dict; None по стороне, если рынка/цены нет."""
    out = {'bg_spot_bid': None, 'bg_spot_ask': None,
           'bb_spot_bid': None, 'bb_spot_ask': None}
    sym = f'{coin}USDT'
    # Bybit spot
    try:
        d = bybit_get('/v5/market/tickers', {'category': 'spot', 'symbol': sym})
        lst = d if isinstance(d, list) else (d.get('list', []) if isinstance(d, dict) else [])
        if lst:
            t = lst[0]
            b = float(t.get('bid1Price') or 0); a = float(t.get('ask1Price') or 0)
            out['bb_spot_bid'] = b if b > 0 else None
            out['bb_spot_ask'] = a if a > 0 else None
    except Exception as e:
        log.debug(f'_fetch_spot_prices BB {coin}: {e}')
    # Bitget spot
    try:
        d = bitget_get('/api/v2/spot/market/tickers', params={'symbol': sym})
        lst = d if isinstance(d, list) else (d.get('list', []) if isinstance(d, dict) else [])
        for t in lst:
            if str(t.get('symbol','')).upper() == sym.upper() or len(lst) == 1:
                b = float(t.get('bidPr') or 0); a = float(t.get('askPr') or 0)
                out['bg_spot_bid'] = b if b > 0 else None
                out['bg_spot_ask'] = a if a > 0 else None
                break
    except Exception as e:
        log.debug(f'_fetch_spot_prices BG {coin}: {e}')
    return out


# ── СПОТ (Этап 2a): размещение / отмена / детект исполнения. Безопасно. ──
def _place_spot_gtc_bb(coin, side, qty, price):
    """Спот GTC на Bybit. side 'Buy'/'Sell'. qty в базовой монете. -> oid|''."""
    if MODE == 'SIMULATION':
        return f'sim_bbspot_{coin}_{side}_{int(time.time()*1000)}'
    sp = (get_spot_specs(coin) or {}).get('bb')
    if not sp:
        return ''
    qd = sp['qty_dec']; pd = sp['price_dec']
    body = {'category': 'spot', 'symbol': f'{coin}USDT', 'side': side,
            'orderType': 'Limit', 'qty': f'{round(qty, qd):.{qd}f}',
            'price': f'{round(price, pd):.{pd}f}', 'timeInForce': 'GTC'}
    log.info(f'[BB SPOT] {side.upper()} {coin} qty={qty} price={price:.{pd}f}')
    try:
        resp = bybit_post('/v5/order/create', body)
        oid = str((resp or {}).get('orderId', ''))
        log.info(f'[BB SPOT] {coin} orderId={oid}')
        return oid
    except ValueError as e:
        err = str(e)
        if any(c in err for c in ['170131','170033','170034','170136','110007']):
            log.warning(f'[BB SPOT] {coin}: недостаточно баланса/мин -- {err[:80]}')
            return 'no_balance'
        log.warning(f'[BB SPOT] {coin}: {err[:120]}')
        return ''

def _place_spot_gtc_bg(coin, side, qty, price):
    """Спот GTC на Bitget. side 'buy'/'sell'. qty в базовой монете. -> oid|''."""
    if MODE == 'SIMULATION':
        return f'sim_bgspot_{coin}_{side}_{int(time.time()*1000)}'
    sp = (get_spot_specs(coin) or {}).get('bg')
    if not sp:
        return ''
    qd = sp['qty_dec']; pd = sp['price_dec']
    body = {'symbol': f'{coin}USDT', 'side': side.lower(), 'orderType': 'limit',
            'force': 'gtc', 'price': f'{round(price, pd):.{pd}f}',
            'size': f'{round(qty, qd):.{qd}f}'}
    log.info(f'[BG SPOT] {side.upper()} {coin} qty={qty} price={price:.{pd}f}')
    try:
        resp = bitget_post('/api/v2/spot/trade/place-order', body)
        d = resp[0] if isinstance(resp, list) and resp else (resp or {})
        oid = str(d.get('orderId', '') or d.get('data', {}).get('orderId', ''))
        log.info(f'[BG SPOT] {coin} orderId={oid}')
        return oid
    except ValueError as e:
        err = str(e)
        if any(c in err for c in ['43012','43004','45110','40762']):
            log.warning(f'[BG SPOT] {coin}: недостаточно баланса/мин -- {err[:80]}')
            return 'no_balance'
        log.warning(f'[BG SPOT] {coin}: {err[:120]}')
        return ''

def _cancel_spot_bb(coin, oid):
    if MODE == 'SIMULATION' or not oid:
        return
    try:
        bybit_post('/v5/order/cancel',
                   {'category': 'spot', 'symbol': f'{coin}USDT', 'orderId': str(oid)})
    except Exception as e:
        log.debug(f'_cancel_spot_bb {coin} {oid}: {e}')

def _cancel_spot_bg(coin, oid):
    if MODE == 'SIMULATION' or not oid:
        return
    try:
        bitget_post('/api/v2/spot/trade/cancel-order',
                    {'symbol': f'{coin}USDT', 'orderId': str(oid)})
    except Exception as e:
        log.debug(f'_cancel_spot_bg {coin} {oid}: {e}')

def _get_spot_fill_bb(coin, oid):
    """Исполнение спот-ордера Bybit. {'avg_price','qty'} если исполнен, иначе {}."""
    if MODE == 'SIMULATION':
        return {}
    if not oid:
        return {}
    try:
        d = bybit_get('/v5/order/history',
                      {'category': 'spot', 'symbol': f'{coin}USDT', 'orderId': str(oid)})
        lst = d if isinstance(d, list) else (d.get('list', []) if isinstance(d, dict) else [])
        for o in lst:
            if str(o.get('orderId')) == str(oid):
                st = str(o.get('orderStatus', ''))
                ap = float(o.get('avgPrice') or 0)
                cq = float(o.get('cumExecQty') or 0)
                if st == 'Filled' and ap > 0:
                    return {'avg_price': ap, 'qty': cq}
                return {}
    except Exception as e:
        log.debug(f'_get_spot_fill_bb {coin} {oid}: {e}')
    return {}

def _get_spot_fill_bg(coin, oid):
    """Исполнение спот-ордера Bitget. {'avg_price','qty'} если исполнен, иначе {}."""
    if MODE == 'SIMULATION':
        return {}
    if not oid:
        return {}
    try:
        d = bitget_get('/api/v2/spot/trade/orderInfo',
                       params={'orderId': str(oid)}, signed=True)
        lst = d if isinstance(d, list) else (d.get('list', [d]) if isinstance(d, dict) else [])
        for o in lst:
            if str(o.get('orderId')) == str(oid):
                st = str(o.get('status', ''))
                ap = float(o.get('priceAvg') or 0)
                bv = float(o.get('baseVolume') or 0)
                if st in ('filled', 'full_fill', 'partial_fill') and ap > 0:
                    return {'avg_price': ap, 'qty': bv}
                return {}
    except Exception as e:
        log.debug(f'_get_spot_fill_bg {coin} {oid}: {e}')
    return {}


def _adopt_second_leg(coin, half):
    """Подхватывает уже выставленный на бирже ордер 2-й ноги (чтобы не дублировать
    при реконсиляции). Заполняет pending_oid (dir A) или кандидатов (dir B).
    Возвращает True, если что-то подхвачено; False -- ордеров нет (бот выставит)."""
    sym = f'{coin}USDT'
    adopted = False
    bb_perp = bb_spot = bg_spot = []
    try:
        d = bybit_get('/v5/order/realtime', {'category': 'linear', 'symbol': sym})
        bb_perp = d if isinstance(d, list) else (d.get('list', []) if isinstance(d, dict) else [])
    except Exception as e:
        log.debug(f'_adopt BBperp {coin}: {e}')
    try:
        d = bybit_get('/v5/order/realtime', {'category': 'spot', 'symbol': sym})
        bb_spot = d if isinstance(d, list) else (d.get('list', []) if isinstance(d, dict) else [])
    except Exception as e:
        log.debug(f'_adopt BBspot {coin}: {e}')
    try:
        d = bitget_get('/api/v2/spot/trade/unfilled-orders', params={'symbol': sym}, signed=True)
        bg_spot = d if isinstance(d, list) else (d.get('list', []) if isinstance(d, dict) else [])
    except Exception as e:
        log.debug(f'_adopt BGspot {coin}: {e}')

    def _side(o): return str(o.get('side', '')).lower()

    if half.direction == 'A':
        # 2-я нога = BB perp SELL
        for o in bb_perp:
            if _side(o) == 'sell':
                half.pending_oid  = str(o.get('orderId'))
                half.pending_exch = 'bybit'
                half.target_price = float(o.get('price') or 0) or half.target_price
                adopted = True
                break
    else:
        # 2-я нога = LONG: BB perp Buy / BB spot Buy / BG spot Buy
        for o in bb_perp:
            if _side(o) == 'buy':
                half.c_perp_oid = str(o.get('orderId')); half.c_perp_qty = float(o.get('qty') or 0)
                adopted = True; break
        for o in bb_spot:
            if _side(o) == 'buy':
                half.c_sbb_oid = str(o.get('orderId')); half.c_sbb_qty = float(o.get('qty') or 0)
                adopted = True; break
        for o in bg_spot:
            if _side(o) == 'buy':
                half.c_sbg_oid = str(o.get('orderId')); half.c_sbg_qty = float(o.get('size') or 0)
                adopted = True; break
        if adopted and not half.target_price:
            for o in (bb_perp + bb_spot):
                if _side(o) == 'buy' and o.get('price'):
                    half.target_price = float(o['price']); break
    if adopted:
        half._last_check = time.time()
    return adopted


def _log_funding_pair(coin, direction, open_time):
    """Логирует funding обеих ног пары за время удержания (BG + BB).
    Вызывается при закрытии пары. ТОЛЬКО лог -- основной PnL не меняет."""
    if MODE == 'SIMULATION':
        return
    def _fetch():
        try:
            start_ms = int(open_time * 1000)
            end_ms   = int(time.time() * 1000)
            held_h   = (end_ms - start_ms) / 3600000
            fund_bg = _fetch_funding_bg(coin, start_ms, end_ms)
            fund_bb = _fetch_funding_bb(coin, start_ms, end_ms)
            total   = fund_bg + fund_bb
            log.info(f'[FUNDING] {coin} dir={direction} держали {held_h:.1f}ч: '
                     f'BG={fund_bg:+.5f} BB={fund_bb:+.5f} итого={total:+.5f} USDT '
                     f'(+ получили / - заплатили; НЕ учтён в расчётном net PnL)')
        except Exception as e:
            log.warning(f'[FUNDING] {coin}: ошибка: {e}')
    threading.Thread(target=_fetch, daemon=True, name=f'funding-{coin}').start()


In [ ]:
# Снапшоты открытых позиций (нужны спайку для детекта закрытия по TP)
import time

def _bg_positions_snapshot() -> dict:
    """Возвращает {coin: {side, size, avg_price}} с Bitget USDT-FUTURES."""
    out = {}
    try:
        data = bitget_get('/api/v2/mix/position/all-position',
                          params={'productType': 'USDT-FUTURES', 'marginCoin': 'USDT'},
                          signed=True)
        for p in (data or []):
            sym  = p.get('symbol', '')
            coin = sym[:-4] if sym.endswith('USDT') else sym
            total = float(p.get('total', 0) or 0)
            if total <= 0: continue
            out[coin] = {'side': (p.get('holdSide') or '').lower(),
                         'size': total,
                         'avg_price': float(p.get('openPriceAvg', 0) or 0)}
    except Exception as e:
        log.error(f'_bg_positions_snapshot: {type(e).__name__}: {e}')
    return out

# кэш снапшота позиций Bybit: гасит рейт-лимит (1 запрос за окно вместо N на N позиций)
_bb_pos_cache = {'t': 0.0, 'data': None, 'err': None}
BB_POS_CACHE_SEC = 3.0   # как долго переиспользовать снапшот

def _bb_positions_snapshot(max_age: float = None) -> dict:
    """Снапшот позиций Bybit linear: {coin: {side, qty, avg_price}}.
    С КЭШЕМ (max_age сек) — чтобы детект закрытия не дёргал /v5/position/list по разу на КАЖДУЮ
    позицию и не ловил рейт-лимит 10006. ВАЖНО: при ошибке API НЕ возвращает {} (иначе детект
    закрытия примет это за 'позиций нет' и ЛОЖНО закроет открытую позицию — так бросило HEI):
    отдаёт последний успешный снапшот, а если его нет — пробрасывает исключение,
    чтобы вызывающий (_spike_closed) считал 'неизвестно' и слот НЕ освобождал."""
    max_age = BB_POS_CACHE_SEC if max_age is None else max_age
    now = time.time(); c = _bb_pos_cache
    if now - c['t'] < max_age:                 # недавно уже ходили (успех ИЛИ ошибка) -> не долбим API
        if c['data'] is not None:
            return c['data']
        raise (c['err'] or RuntimeError('Bybit positions unavailable'))
    c['t'] = now                               # фиксируем попытку ДО запроса -> параллельные вызовы не дублируют
    try:
        data = bybit_get('/v5/position/list',
                         params={'category': 'linear', 'settleCoin': 'USDT'})
    except Exception as e:
        c['err'] = e
        log.error(f'_bb_positions_snapshot: {type(e).__name__}: {e}')
        if c['data'] is not None:
            return c['data']                   # есть прошлый снапшот -> отдаём (НЕ закрываем по ошибке связи)
        raise                                  # совсем нет данных -> наверх как 'неизвестно'
    rows = data if isinstance(data, list) else data.get('list', [])
    out = {}
    for p in (rows or []):
        sym  = p.get('symbol', '')
        coin = sym[:-4] if sym.endswith('USDT') else sym
        size = float(p.get('size', 0) or 0)
        if size <= 0: continue
        side_raw = (p.get('side') or '').lower()
        out[coin] = {'side': 'long' if side_raw == 'buy' else 'short',
                     'qty': size,
                     'avg_price': float(p.get('avgPrice', 0) or 0)}
    c['data'] = out; c['err'] = None
    return out


## Spike-scalper (свой универсум по рынкам → всплеск ×5 → открытие → закрытие +0.7%). Свечи + REST, одна монета за раз.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SPIKE-SCALPER: всплеск объёма ×5 → лимит по лучшей цене → TP по входу ±%
#   60 мин 1m-свечей → ratio = пик минутки / среднее за час.
#   ratio > SPIKE_VOL_MULT И |изм. цены за SPIKE_PRICE_WIN_MIN мин| >= SPIKE_PRICE_CHG_PCT → вход
#   направление: SPIKE_FADE=True — против движения (памп→шорт, дамп→лонг), иначе по движению
#   лимитом по лучшей цене стакана (taker), затем reduce-only TP по входу ±%.
#   Универсум — СВОЙ на каждом рынке (НЕ пересечение бирж): bg_fut/bg_spot/bb_fut
#   независимы, монета не обязана быть на обеих биржах.
#   Одна монета одновременно. Только свечи + REST, без тиков/WS/WARMUP/COINS/секции5.
#   Спот шортить нельзя → красная свеча на споте пропускается.
# ════════════════════════════════════════════════════════════════════════
import time, threading, json

# ── Настройки ─────────────────────────────────────────────────────────────
SPIKE_DRY_RUN      = False       # True = только лог сигналов, без ордеров
SPIKE_VOL_MULT     = 5.0        # порог объёма: пик минутки / среднее за час (бэктест fade: ×5)
SPIKE_PRICE_CHG_PCT = 7.0       # И движение цены >= , % за окно SPIKE_PRICE_WIN_MIN (бэктест fade: 7%)
SPIKE_CHG_FROM_BASELINE   = True   # изменение цены считать ОТ базового уровня (среднее за окно ниже), а не за SPIKE_PRICE_WIN_MIN мин
SPIKE_BASELINE_OFFSET_MIN = 60     # базовое окно ЗАКАНЧИВАЕТСЯ за столько минут до сейчас
SPIKE_BASELINE_WIN_MIN    = 30     # длина базового окна, мин  =>  база = среднее close за [сейчас-90м .. сейчас-60м]
SPIKE_SKIP_PRETREND       = True   # пропускать сигнал, если за 30м ДО спайка уже был рост объёма И цены (продолжение тренда)
SPIKE_PRETREND_OFFSET_MIN = 5      # 'до спайка' = исключить последние N минут (сам спайк)
SPIKE_PRETREND_WIN_MIN    = 30     # окно проверки предыдущего тренда, мин
SPIKE_PRETREND_PRICE_PCT  = 3.0    # цена уже выросла на >= столько % за это окно
SPIKE_PRETREND_VOL_MULT   = 1.3    # средний объём за окно >= столько x объёма за окно до него
# --- защита от делистинга: свежая проверка ликвидности перед входом (применяется ко ВСЕМ, включая вайтлист) ---
SPIKE_DELIST_FILTER   = True       # вкл проверку 24h-оборота и открытого интереса перед входом
SPIKE_MIN_VOL_24H_USD = 1000000    # 24h-оборот >= $1M (ниже — риск делистинга)
SPIKE_MIN_OI_USD      = 200000     # открытый интерес >= $200k (около нуля = почти гарантия делистинга)
# --- фильтр 'упавшего ножа': пропуск монет, упавших ниже % от исторического максимума ---
SPIKE_ATH_FILTER     = True
SPIKE_MIN_PCT_OF_ATH = 30.0     # текущая цена должна быть >= 30% от макс за историю; ниже -> пропуск
SPIKE_ATH_INTERVAL   = 'D'      # свечи для истории: 'D'=дни (1000 шт ~2.7 года = вся история перпа), 'W'=недели
SPIKE_ATH_LIMIT      = 1000     # сколько свечей запрашивать (Bybit — одним запросом)
SPIKE_ATH_BG_PAGE    = 200      # Bitget отдаёт максимум 200 свечей за запрос -> тянем страницами по endTime
SPIKE_ATH_CACHE_SEC  = 86400    # hist_max кэшируем на сутки (история старых свечей не меняется)
# --- ARBITRON: скан календаря делистингов arbitron.app/listings ---
SPIKE_ARBITRON_ON    = True
SPIKE_ARBITRON_URL   = 'https://arbitron.app/listings?type=delisting&exchange={ex}'
SPIKE_ARBITRON_EXCH  = {'Bybit': 'bb_fut', 'Bitget': 'bg_fut'}   # только СВОИ биржи
SPIKE_ARBITRON_PAGES = 2       # страниц календаря за скан
SPIKE_ARBITRON_MIN   = 15      # период скана, минут
SPIKE_ARBITRON_TTL_D = 14      # дней держать блок после исчезновения из календаря
# === МАРТИНГЕЙЛ (порт условий из симулятора live_trading_sim_fixed) — ВЫКЛ по умолчанию ===
# Фаза Ф3: доливаем равными шагами в УБЫТОЧНУЮ позицию, усредняя вход, пока не выйдем в безубыток
# (его ловит _spike_breakeven_check) или не исчерпаем рунги. Размер долива = стандартная нога (_spike_size).
SPIKE_MART_ENABLE       = True     # вкл усреднение убыточных позиций (ОСТОРОЖНО: усиливает убытки)
SPIKE_MART_TRIG_PCT     = 10.0     # шаг против ПЕРВОГО входа: 10% -> колено 1, 20% -> колено 2 (ТЗ 14.08)
SPIKE_ENTRY_SLIP_PCT    = 0.35     # допуск проскальзывания на входе: IOC ставится на 0.35% ЗА рынок
                                   # (0.15 покрывал 1 из 14 замеров; медиана ухода книги ~0.3%)
                                   # (long: ask*1.0015, short: bid*0.9985). TP/БУ считаются от ФАКТИЧЕСКОГО
                                   # филла, поэтому чистый % сделки не меняется. 0 -> старое поведение.
SPIKE_MART_TIME_SEC     = 7200     # каскад по ВРЕМЕНИ: цель=БУ и 2ч без исполнения -> долив (и далее каждые 2ч, пока есть ступени)
SPIKE_ENTRY_TIF         = 'ioc'    # тип ордера ВХОДА: 'ioc' = исполнить сразу по стакану, остаток биржа отменяет сама
SPIKE_MART_RUNGS        = 3        # макс рунгов: вход + 2 долива (по ТЗ)
SPIKE_MART_STEP_SEC     = 30       # антиспам повторов при неудачном доливе (триггер теперь ценовой)
                                   # — не усредняем в делистинг/убегающий памп (кейсы MBOX/HEI)
# === КАТАСТРОФА-СТОП (защита счёта) — форс-закрытие позиции при ЭКСТРЕМАЛЬНОЙ просадке. ВЫКЛ по умолчанию ===
# Это НЕ торговый SL: порог далеко (-60%), холд-откаты не трогает (HEI на -31% переживёт),
# но ловит убегающие позиции типа VELVET (-200%+), которые иначе едят маржу счёта без выхода.
SPIKE_DISASTER_ENABLE = False     # катастрофа-стоп УБРАН (по ТЗ)
SPIKE_DISASTER_PCT    = 60.0      # порог: просадка глубже этого -> форс-закрытие + бан монеты
# --- ретрай постановки TP (reduce-only может отлететь 22002, если позиция ещё не зарегана после быстрого налива) ---
SPIKE_TP_RETRIES   = 4      # сколько раз повторить постановку TP при пустом oid
SPIKE_TP_RETRY_SEC = 0.7    # пауза между ретраями TP, с (даём позиции распространиться на бирже)
SPIKE_TP_RETRIES_BB   = 6   # Bybit регает позицию медленнее (110017) -> больше попыток
SPIKE_TP_RETRY_SEC_BB = 1.5 # и длиннее пауза (6 x 1.5 = 9с против 5 x 0.7 = 3.5с)
SPIKE_PRICE_WIN_MIN = 5         # окно движения цены: close N минут назад -> сейчас
SPIKE_FADE         = True       # True = вход ПРОТИВ движения (mean-reversion: памп→шорт, дамп→лонг).
                                #        Бэктест fade ×5/7% за 5 мин: +0.39%/сделку, 148 сделок, win 95% (30д).
SPIKE_TP_PCT       = 3.0        # дефолт TP, % (20.08: расчёт по 300 сделкам с учётом простоя слота -> пик на 3%)
SPIKE_STOP_PCT     = 4.5        # аварийный стоп: закрыть рыночно при движении ПРОТИВ входа на , % (0=выкл)

# ── WHITELIST: торговать ТОЛЬКО эти монеты, с пер-монетными TP/SL (из бэктеста) ──
#   ключ (МОНЕТА, биржа) -> (TP%, SL%). US и VELVET есть на обеих биржах с РАЗНЫМИ параметрами.
#   Перебивает статический блок-лист и фильтр «молодых». Рантайм-блэклист (делистинг) и пост-стоп бан сохраняются.
SPIKE_WHITELIST_ON = False   # вайтлист ВЫКЛ (по ТЗ): торгуем весь юниверс, TP=SPIKE_TP_PCT=1.0%
# TP по истории путей (spike_trade_hist.json, 300 сделок). Модель: дошло до цели -> +TP-комиссия;
# не дошло -> фактический исход (БУ/мартингейл/хвост) МИНУС штраф за занятый слот (0.6%).
# Отбор: n>=8 и подтверждение на отложенной выборке (обучение 70% / тест 30%).
# APR/bb исключён: на тесте -0.79%. Остальные 8 пар подтвердились.
SPIKE_COIN_PARAMS  = {
    ('HFT',     'bg_fut'): (4.7, 4.5),   # n=13, тест +3.31
    ('ESPORTS', 'bg_fut'): (4.0, 8.0),   # n=11, тест +2.79
    ('BICO',    'bg_fut'): (2.9, 4.5),   # n= 8, тест +2.78
    ('CYS',     'bg_fut'): (4.5, 4.5),   # n=20, тест +1.94
    ('US',      'bg_fut'): (4.5, 4.0),   # n=11, тест +1.94
    ('BLUAI',   'bb_fut'): (3.1, 4.0),   # n=17, тест +1.81
    ('CYS',     'bb_fut'): (2.8, 4.5),   # n=19, тест +1.61
    ('US',      'bb_fut'): (4.8, 8.0),   # n=13, тест +0.80
    # остальные — дефолт SPIKE_TP_PCT (3.0); SL в режиме breakeven не используется
    ('PORTAL', 'bb_fut'): (3.0, 4.5), ('BP',     'bb_fut'): (3.0, 6.0),
    ('HEI',    'bb_fut'): (3.0, 3.0), ('MBOX',   'bb_fut'): (3.0, 5.0),
    ('EVAA',   'bb_fut'): (3.0, 8.0), ('VELVET', 'bb_fut'): (3.0, 3.0),
    ('SPACE',  'bg_fut'): (3.0, 4.0), ('VELVET', 'bg_fut'): (3.0, 3.0),
    ('OPG',    'bg_fut'): (3.0, 4.0),
}
_spike_tp_override: dict = {}    # (coin, market) -> TP% для «мягких» позиций (см. SPIKE_SOFT_*), переживает рестарт
SPIKE_SOFT_FILE = 'spike_soft.json'

def _spike_soft_save():
    try:
        with open(SPIKE_SOFT_FILE, 'w') as f:
            json.dump({f'{c}|{m}': v for (c, m), v in _spike_tp_override.items()}, f)
    except Exception as e:
        log.debug(f'[SPIKE] soft save: {e}')

def _spike_soft_load():
    try:
        with open(SPIKE_SOFT_FILE) as f:
            for k, v in json.load(f).items():
                c, m = k.split('|', 1); _spike_tp_override[(c, m)] = float(v)
        if _spike_tp_override:
            log.info(f'[SPIKE] мягкие позиции из файла: {len(_spike_tp_override)}')
    except FileNotFoundError:
        pass
    except Exception as e:
        log.debug(f'[SPIKE] soft load: {e}')

def _spike_pct(coin, market):
    """(TP%, SL%) для (монета, биржа); мягкая позиция -> свой TP; иначе таблица или глобальные дефолты."""
    tp, sl = SPIKE_COIN_PARAMS.get((coin, market), (SPIKE_TP_PCT, SPIKE_STOP_PCT))
    _o = _spike_tp_override.get((coin, market))
    return (float(_o), sl) if _o else (tp, sl)
SPIKE_MAX_HOLD_SEC = 0          # таймаут-выход ВЫКЛЮЧЕН (был 3600): держим до TP или серверного SL
SPIKE_STOP_BAN_SEC = 0          # баны после стопа УБРАНЫ (по ТЗ); 0 = не банить
SPIKE_MART_BAN_DAYS = 14        # монета, потратившая ОБЕ ступени мартингейла, -> бан на N дней (обе биржи); 0 = выкл
SPIKE_MART_BAN_PERM = 2         # на СКОЛЬКОМ по счёту мартингейл-бане монета банится НАВСЕГДА; 0 = выкл
SPIKE_HOLD_BAN_SEC  = 12 * 3600 # позиция провисела дольше -> бан монеты на SPIKE_HOLD_BAN_DAYS; 0 = выкл
SPIKE_HOLD_BAN_DAYS = 7         # срок бана за долгое удержание
SPIKE_HOLD_BAN_PERM = 2         # на СКОЛЬКОМ по счёту таком бане — НАВСЕГДА; 0 = выкл
SPIKE_RUNG_BAN_N    = 3         # монета, ушедшая в долив (любое колено) N раз -> бан на SPIKE_RUNG_BAN_DAYS; 0 = выкл
SPIKE_RUNG_BAN_DAYS = 7         # срок бана за частые колена (рецидивист: UAI 4 колена за неделю, всё в ноль)
SPIKE_RUNG_BAN_WIN  = 7 * 86400 # окно, в котором считаются колена (старше — не учитываются)
_spike_rung_hist: dict = {}     # монета -> [ts колен]
_spike_holdban_cnt: dict = {}   # монета -> сколько раз висела дольше лимита
SPIKE_HOLD_STATS_FILE = 'spike_hold_stats.jsonl'   # статистика: время в позиции + были ли доливы (по каждому закрытию)
SPIKE_USDT         = 25.0       # размер ордера (нога) ПО УМОЛЧАНИЮ, USDT — отвязан от HALF_USDT арба
SPIKE_USDT_BY_MARKET = {        # нога ЛОНГА по биржам (05.10: возвращено 12.5 — 4 ноги в потолок 50 / 2 в потолок 25)
    'bg_fut': 12.5,
    'bb_fut': 12.5,
}
SPIKE_USDT_SHORT     = 5.0      # 05.10: нога ШОРТА, USDT (лонг — SPIKE_USDT_BY_MARKET). 5 = минимум ордера на Bitget и Bybit
SPIKE_MKT_NOTIONAL_CAP = {      # ПОТОЛОК суммарного номинала по бирже (все позиции + все доливы), USDT
    'bg_fut': 50.0,             #   15+15+15 = 45 -> одна позиция с полным мартингейлом умещается
    'bb_fut': 25.0,             #   15 -> одна нога; второй вход/долив уже не влезет
}


def _spike_mkt_cap(market):
    return float(SPIKE_MKT_NOTIONAL_CAP.get(market, 10 ** 9))


def _spike_mkt_notional(market, exclude=None):
    """Суммарный номинал открытых позиций на бирже, USDT (с учётом множителя контракта)."""
    tot = 0.0
    with _spike_lock:
        items = list(_spike_positions.items())
    for (c_, m_), p in items:
        if m_ != market or (exclude and c_ == exclude):
            continue
        tot += (p.get('entry_px') or 0) * (p.get('qty') or 0) * _spike_mult(c_, m_)
    return tot


def _spike_soft_allowed(market):
    """Мягкий вход разрешён: биржа в списке, свободно >= N ног, мягких позиций меньше лимита. -> (ok, свободных ног)"""
    if not SPIKE_SOFT_ENABLE or market not in SPIKE_SOFT_MARKETS:
        return (False, 0)
    leg = _spike_leg_usdt(market)
    free = int((_spike_mkt_cap(market) - _spike_mkt_notional(market) + 0.5) // leg)
    with _spike_lock:
        n_soft = sum(1 for k in _spike_positions if k[1] == market and k in _spike_tp_override)
    return (free >= SPIKE_SOFT_MIN_FREE_LEGS and n_soft < SPIKE_SOFT_MAX_POS, free)


def _spike_soft_cleanup():
    """Снять переопределение TP с закрытых позиций (после 3 мин от старта — чтобы не задеть восстановление)."""
    if time.time() - _spike_soft_started < 180 or not _spike_tp_override:
        return
    with _spike_lock:
        dead = [k for k in _spike_tp_override if k not in _spike_positions]
    if dead:
        for k in dead:
            _spike_tp_override.pop(k, None)
        _spike_soft_save()


def _spike_leg_usdt(market, direction=None):
    """Размер ноги: лонг (и без направления) — SPIKE_USDT_BY_MARKET; шорт — SPIKE_USDT_SHORT."""
    if direction == 'short':
        return float(SPIKE_USDT_SHORT)
    return float(SPIKE_USDT_BY_MARKET.get(market, SPIKE_USDT))
SPIKE_MAX_NOTIONAL_X = 2.0      # пропускать монету, если мин. лот дороже X ног этой биржи (лимит считается от ноги)
SPIKE_MAX_NOTIONAL = 50         # запасное значение, если нога биржи неизвестна
SPIKE_MAX_POSITIONS = 0         # ОБЩИЙ лимит по боту: 0 = без ограничений (лимиты ПО БИРЖАМ, ниже)
SPIKE_MAX_POS_PER_MARKET = 0    # лимит по ЧИСЛУ позиций снят: ограничение задаёт SPIKE_MKT_NOTIONAL_CAP
                                # (сумма ног на бирже: bg 50 = 4 ноги по 12.5, bb 25 = 2 ноги). 0 = без ограничения
SPIKE_ONE_COIN_ONE_LEG   = True # одна монета — только на ОДНОЙ бирже: если монета уже открыта где-то,
                                # вход по ней на второй бирже пропускается (двойная экспозиция в один риск)
SPIKE_BLOCK_NEW_WHILE_MART = True   # мартингейл на бирже -> новых входов НА ЭТОЙ бирже нет (другая биржа торгует)
SPIKE_MART_ONE_COIN_MKT    = True   # доливы одновременно только по ОДНОЙ монете на бирже:
                                    # начали доливать одну — остальные позиции этой биржи доливов не получают
SPIKE_MART_STOP_IF_MAXED   = True   # если ЛЮБАЯ позиция на бирже исчерпала все ступени (3/3) —
                                    # доливы по ОСТАЛЬНЫМ позициям этой биржи запрещены
SPIKE_ONE_POS_PER_COIN = True   # True = одна позиция на МОНЕТУ: если она уже открыта на любой бирже — новый вход запрещён
SPIKE_FUNDING_FILTER  = False   # ВЫКЛ: фандинг не влияет на вход (резал все лонги при положительном фандинге)
SPIKE_FUNDING_MIN     = 0.0     # мин. фандинг В НАШУ ПОЛЬЗУ, доля (0=не платить; 0.0001=получать >=0.01%; <0=допустить плату)
SPIKE_FUNDING_REQUIRE = False   # если ставка недоступна: True=пропустить вход, False=входить как обычно
SPIKE_EXIT_MODE = 'breakeven'   # 'tpsl'=старый TP/SL; 'breakeven'=БЕЗ стопов, выход в безубыток(+фандинг) или 12ч по рынку
SPIKE_BREAKEVEN_MARGIN  = 0.0   # порог нетто-выхода, %: 0=ровно безубыток; >0=небольшой профит сверх издержек
SPIKE_BREAKEVEN_AFTER_SEC = 7200   # без колен: через 2ч цель TP -> БЕЗУБЫТОК (GTC переставляется + добивание по net)
SPIKE_BREAKEVEN_MAX_SEC = 0        # 0 = БЕЗ таймаута (держим до TP или безубытка); >0 = закрыть по рынку через столько сек
SPIKE_FUNDING_POLL_SEC  = 300   # как часто опрашивать накопленный фандинг позиции, сек
SPIKE_LEVERAGE     = 1          # принудительно ставить это плечо перед каждым входом (0 = не трогать)
SPIKE_MARKETS      = ['bg_fut', 'bb_fut']   # UAI закрыта 03.09 -> Bybit снова в работе (нога меньше, см. SPIKE_USDT_BY_MARKET)
SPIKE_FILL_WAIT    = 5.0        # сек ждать исполнения входа, иначе отмена
SPIKE_POLL_SEC     = 5.0        # период сканирования
SPIKE_COOLDOWN_SEC = 300        # пауза по монете после входа/неудачного входа
SPIKE_FAIL_MAX     = 2          # после стольких неудачных входов подряд — монета в чёрный список (делистинг и т.п.)
SPIKE_FAIL_BAN     = False      # реактивный блэклист по отказам входа: False = ВЫКЛ (отказ != делистинг; защиту даёт фильтр ликвидности)
SPIKE_COIN_BLOCKLIST = {'PLAYSOUT', 'B', 'VELVET'}   # постоянный бан: убыточны во всех вариантах бэктеста (VELVET добавлена: таймаут -8.9%)
SPIKE_STOCK_METAL_BLOCK = {   # металлы и токенизированные акции/ETF — не крипта, другое поведение + часто нужен агримент
    # металлы / сырьё
    'XAU', 'XAG', 'XAUT', 'PAXG', 'CL', 'XPT', 'XPD', 'WTI', 'GOLD', 'SILVER',
    # токенизированные акции / ETF (видены в универсуме + крупные)
    'SOXL', 'SMCI', 'MU', 'SNDK', 'MRVL', 'AAOI', 'RKLB', 'ORCL', 'DRAM', 'CBRS', 'LITE',
    'OPENAI', 'NVDA', 'TSLA', 'AAPL', 'MSFT', 'GOOGL', 'GOOG', 'AMZN', 'META', 'AMD',
    'NFLX', 'AVGO', 'PLTR', 'INTC', 'COIN', 'MSTR', 'HOOD', 'CRCL', 'IBIT', 'SPY', 'QQQ',
    'AAOI', 'SNDK', 'CRWD', 'SHOP', 'UBER', 'BABA', 'DIS', 'PYPL', 'SQ', 'ARM', 'TSM',
    'MRNA', 'CSOPSK2LHKD',   # 22.09: прошли мягкий сигнал на Bitget, спас только лот
}
SPIKE_COIN_SIDE    = {'UB': 'short_only', 'SKYAI': 'short_only', 'SKYAI1': 'short_only'}  # запрет стороны (лонги стабильно в минус)
SPIKE_ALLOW_LONG   = True       # лонги на дампах РАЗРЕШЕНЫ (откат 11.08); False = только шорты на пампах
SPIKE_ALLOW_SHORT  = True       # 05.10: шорты включены с ногой 5 USDT (SPIKE_USDT_SHORT); 04.10 отключались после QNT
# === МЯГКИЙ СИГНАЛ (22.09): Bitget половину суток стоит пустым -> при 2+ свободных ногах один слот
#     отдаём под ослабленный порог ×4 / 6% с целью 1%. Все остальные фильтры входа — как обычно. ===
SPIKE_SOFT_ENABLE        = True
SPIKE_SOFT_MARKETS       = {'bg_fut'}   # только Bitget (Bybit и так забит)
SPIKE_SOFT_VOL_MULT      = 4.0          # объём: пик минутки / среднее за час
SPIKE_SOFT_CHG_PCT       = 6.0          # цена: |изм.| от базы, %
SPIKE_SOFT_TP_PCT        = 1.0          # цель мягкой позиции, %
SPIKE_SOFT_MIN_FREE_LEGS = 2            # свободных ног на бирже ДО входа (после входа остаётся 1 под основной сигнал)
SPIKE_SOFT_MAX_POS       = 1            # одновременно мягких позиций на бирже
_spike_soft_started = time.time()
SPIKE_MIN_AGE_DAYS = 180        # общий выключатель фильтра возраста: 0 = выкл (пороги ниже)
SPIKE_AGE_BLOCK_D  = 30         # моложе стольких дней — вход запрещён совсем
SPIKE_AGE_SHORT_D  = 30         # 22.09: было 90 (30-90д только шорт). Теперь = SPIKE_AGE_BLOCK_D: лонги разрешены с 30 дней
SPIKE_LOW_PCTL     = 10         # ЛОНГИ не открываем, если цена ниже N-го перцентиля дневных закрытий за 90д (SOPH-кейс)
SPIKE_LOW_PCTL_DAYS = 90        # окно перцентиля, дней. SPIKE_LOW_PCTL = 0 -> фильтр выкл. Кэш на сутки.
SPIKE_HIGH_PCTL    = 90         # 21.09: ЛОНГИ не открываем, если цена ВЫШЕ N-го перцентиля дневных закрытий
                                #        за те же SPIKE_LOW_PCTL_DAYS дней (откат сдувающегося пампа, кейс 龙虾).
                                #        0 = фильтр выкл. Окно и кэш общие с нижним перцентилем.
SPIKE_PCTL_MIN_CLOSES = 7       # 22.09: минимум дневных закрытий для расчёта перцентилей (было жёстко 20:
                                #        LONGXIA/bb с 9 свечами проходила верхний фильтр как «нет данных»)
SPIKE_MIN_DAILY_CANDLES = 30    # 22.09: закрытий за окно меньше -> монета моложе 30д -> вход запрещён (обе стороны).
                                #        Страховка кэша возраста: считается по тем же свечам, что и перцентили.
                                #        Обе биржи отдают до 90 дневных свечей, старые монеты не задевает. 0 = выкл.
SPIKE_FEE_PCT      = 0.12       # комиссия за круг для оценки PnL в симуляции, %
SPIKE_DRY_MAX_HOLD_SEC = 0      # DRY_RUN: симуляция закрывает по TP; >0 — ещё и таймаут-выход, сек (0=выкл)
SPIKE_STATUS_SEC   = 60         # как часто логировать статус открытой позиции (нереализованный PnL), сек
SPIKE_HEARTBEAT_MIN = 15        # «жив» в простое раз в N минут (0 = молчать в простое)
SPIKE_MIN_VOL      = 0          # фильтр универсума: 24h-оборот ≥, USDT (0 = без фильтра, все листинги)
SPIKE_MAX_COINS    = None       # топ-N по обороту на КАЖДОМ рынке (None = ВСЕ монеты рынка)
SPIKE_UNIVERSE_REFRESH_SEC = 1800   # как часто обновлять список монет, сек
SPIKE_FETCH_SLEEP  = 0.03       # пауза между запросами свечей (от всплеска REST)
# ────────────────────────────────────────────────────────────────────────

_spike_positions: dict   = {}   # (coin, market) -> открытая сделка / dry-слот
_spike_cooldown:  dict   = {}
_spike_blacklist: set    = set()   # монеты с перманентным запретом входа (делистинг и т.п.) — на сессию
_spike_fail:      dict   = {}   # монета -> число неудачных входов подряд
_spike_ban:       dict   = {}   # монета -> ts, до которого вход запрещён (бан после стопа/таймаута)
SPIKE_BAN_FILE = 'spike_ban.json'  # бан переживает рестарт (иначе недельный бан слетит при передеплое)
SPIKE_MARTBAN_FILE = 'spike_martban.json'   # счётчик мартингейл-банов по монете + перманентные баны
_spike_martban_cnt: dict = {}      # монета -> сколько раз исчерпала обе ступени мартингейла

def _spike_ban_save():
    try:
        now = time.time()
        live = {c: ts for c, ts in _spike_ban.items() if ts > now}
        with open(SPIKE_BAN_FILE, 'w') as f:
            json.dump(live, f)
    except Exception as e:
        log.debug(f'[SPIKE] ban save: {e}')

def _spike_martban_save():
    try:
        with open(SPIKE_MARTBAN_FILE, 'w') as f:
            json.dump({'cnt': _spike_martban_cnt, 'hold': _spike_holdban_cnt,
                       'rungs': _spike_rung_hist, 'perm': sorted(_spike_blacklist)}, f)
    except Exception as e:
        log.debug(f'[SPIKE] martban save: {e}')


def _spike_martban_load():
    try:
        with open(SPIKE_MARTBAN_FILE) as f:
            d = json.load(f)
        _spike_martban_cnt.update(d.get('cnt', {}))
        _spike_holdban_cnt.update(d.get('hold', {}))
        _spike_rung_hist.update(d.get('rungs', {}))
        _spike_blacklist.update(d.get('perm', []))
        if _spike_blacklist:
            log.info(f'[SPIKE] перманентный бан загружен: {len(_spike_blacklist)} монет '
                     f'({", ".join(sorted(_spike_blacklist))})')
    except FileNotFoundError:
        pass
    except Exception as e:
        log.debug(f'[SPIKE] martban load: {e}')


def _spike_ban_load():
    try:
        with open(SPIKE_BAN_FILE) as f:
            data = json.load(f)
        now = time.time()
        for c, ts in data.items():
            if ts > now:
                _spike_ban[c] = ts
        if _spike_ban:
            log.info(f'[SPIKE] загружен бан из файла: {len(_spike_ban)} монет ещё под баном')
    except FileNotFoundError:
        pass
    except Exception as e:
        log.debug(f'[SPIKE] ban load: {e}')
_spike_candles_cache:  dict = {}
_spike_universe_cache: dict = {'ts': 0, 'uni': {}}
_spike_lock   = threading.Lock()
_spike_stop   = threading.Event()
_spike_thread = None


# ── Универсум: свои монеты на КАЖДОМ рынке (без пересечения бирж) ─────────
def _bg_fut_coins():
    """Топ-N USDT-перпов Bitget по 24h-обороту (самодостаточно, без секции 5)."""
    vols = {}
    try:
        data = bitget_get('/api/v2/mix/market/tickers', params={'productType': 'USDT-FUTURES'})
        rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
        for d in rows:
            sym = str(d.get('symbol', ''))
            if not sym.endswith('USDT'):
                continue
            vol = 0.0
            for f in ('usdtVolume', 'quoteVolume', 'turnover24h'):
                try:
                    v = float(d.get(f) or 0)
                    if v > 0:
                        vol = v; break
                except Exception:
                    pass
            if vol >= SPIKE_MIN_VOL:
                vols[sym[:-4]] = vol
    except Exception as e:
        log.debug(f'_bg_fut_coins: {e}')
    return [c for c, _ in sorted(vols.items(), key=lambda kv: kv[1], reverse=True)][:SPIKE_MAX_COINS]

def _bb_fut_coins():
    """Топ-N USDT-перпов Bybit по 24h-обороту (turnover24h в USDT)."""
    vols = {}
    try:
        data = bybit_get('/v5/market/tickers', {'category': 'linear'})
        rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
        for d in rows:
            sym = str(d.get('symbol', ''))
            if not sym.endswith('USDT') or sym.endswith('USDC'):
                continue
            try:
                vol = float(d.get('turnover24h') or 0)
            except Exception:
                vol = 0.0
            if vol >= SPIKE_MIN_VOL:
                vols[sym[:-4]] = vol
    except Exception as e:
        log.debug(f'_bb_fut_coins: {e}')
    return [c for c, _ in sorted(vols.items(), key=lambda kv: kv[1], reverse=True)][:SPIKE_MAX_COINS]

def _bg_spot_coins():
    out = {}
    try:
        data = bitget_get('/api/v2/spot/market/tickers', params={})
        rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
        for t in rows:
            sym = str(t.get('symbol', ''))
            if not sym.endswith('USDT'):
                continue
            vol = 0.0
            for f in ('usdtVol', 'quoteVol', 'usdtVolume'):
                try:
                    v = float(t.get(f) or 0)
                    if v > 0:
                        vol = v; break
                except Exception:
                    pass
            if vol >= SPIKE_MIN_VOL:
                out[sym[:-4]] = vol
    except Exception as e:
        log.debug(f'_bg_spot_coins: {e}')
    return [c for c, _ in sorted(out.items(), key=lambda kv: kv[1], reverse=True)][:SPIKE_MAX_COINS]

def _spike_get_universe():
    """{market: [coins]} со своим списком на каждом рынке. Кэш SPIKE_UNIVERSE_REFRESH_SEC."""
    now = time.time()
    if _spike_universe_cache['uni'] and now - _spike_universe_cache['ts'] < SPIKE_UNIVERSE_REFRESH_SEC:
        return _spike_universe_cache['uni']
    uni = {}
    if 'bg_fut'  in SPIKE_MARKETS: uni['bg_fut']  = _bg_fut_coins()
    if 'bg_spot' in SPIKE_MARKETS: uni['bg_spot'] = _bg_spot_coins()
    if 'bb_fut'  in SPIKE_MARKETS: uni['bb_fut']  = _bb_fut_coins()
    _spike_universe_cache.update(ts=now, uni=uni)
    log.info('[SPIKE] универсум: ' + ', '.join(f'{m}={len(c)}' for m, c in uni.items()))
    return uni


def _spike_candles(coin, market):
    """60+ мин 1m-свечей -> [(ts, open, close, vol_usdt)] по возрастанию. Кэш 20с."""
    now = time.time(); key = (coin, market)
    c = _spike_candles_cache.get(key)
    if c and now - c[0] < 20:
        return c[1]
    out = []
    try:
        if market == 'bg_fut':
            end = int(now * 1000)
            data = bitget_get('/api/v2/mix/market/candles',
                              params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                      'granularity': '1m', 'startTime': str(end - 95*60*1000),
                                      'endTime': str(end), 'limit': '100'})
        elif market == 'bg_spot':
            data = bitget_get('/api/v2/spot/market/candles',
                              params={'symbol': f'{coin}USDT', 'granularity': '1min', 'limit': '100'})
        elif market == 'bb_fut':
            data = bybit_get('/v5/market/kline',
                             {'category': 'linear', 'symbol': f'{coin}USDT', 'interval': '1', 'limit': '100'})
        else:
            data = None
        rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
        for r in rows:
            try:
                vol = float(r[6]) if len(r) > 6 else float(r[5])   # quote/turnover (USDT)
                out.append((int(r[0]), float(r[1]), float(r[4]), vol))
            except Exception:
                pass
        out.sort(key=lambda x: x[0])
    except Exception as e:
        log.debug(f'_spike_candles {coin}/{market}: {e}'); out = []
    _spike_candles_cache[key] = (now, out)
    return out


def _spike_signal(coin, market, vol_mult=None, chg_pct=None):
    """Вход: всплеск объёма ×SPIKE_VOL_MULT И |изм. цены| >= SPIKE_PRICE_CHG_PCT.
    Изм. цены: если SPIKE_CHG_FROM_BASELINE — от среднего уровня за окно час назад (перегретость),
    иначе за SPIKE_PRICE_WIN_MIN мин. Если SPIKE_SKIP_PRETREND — пропуск, когда за 30м до спайка уже
    был рост объёма+цены (продолжение тренда). Направление — по знаку (с учётом SPIKE_FADE). -> (ok, ratio, dir, chg%)."""
    rows = _spike_candles(coin, market)
    if len(rows) < 10:
        return (False, 0.0, None, 0.0)
    win  = rows[-60:]                                   # последние ~60 минут
    vols = [r[3] for r in win]
    avg  = sum(vols) / len(vols) if vols else 0.0
    if avg <= 0:
        return (False, 0.0, None, 0.0)
    ratio = max(win[-5:], key=lambda x: x[3])[3] / avg  # пик минутки за 5 мин / среднее
    last = win[-1][2]
    _need = SPIKE_BASELINE_OFFSET_MIN + SPIKE_BASELINE_WIN_MIN
    if SPIKE_CHG_FROM_BASELINE and len(rows) >= _need:
        # база = среднее close за окно [сейчас-(offset+win) .. сейчас-offset] (по умолч. [-90м..-60м]);
        # ловит ПЕРЕГРЕТОСТЬ относительно уровня час назад, а не 5-мин шум
        base_slice = rows[-_need:-SPIKE_BASELINE_OFFSET_MIN]
        base = sum(r[2] for r in base_slice) / len(base_slice) if base_slice else 0.0
        chg = (last - base) / base * 100 if base > 0 else 0.0
    else:
        # фолбэк: движение за последние N минут (мало свечей / база выключена)
        pw = min(SPIKE_PRICE_WIN_MIN, len(win) - 1)
        first = win[-pw - 1][2]
        chg = (last - first) / first * 100 if first > 0 else 0.0
    _vm = SPIKE_VOL_MULT if vol_mult is None else float(vol_mult)
    _cp = SPIKE_PRICE_CHG_PCT if chg_pct is None else float(chg_pct)
    if ratio <= _vm or abs(chg) < _cp:
        return (False, ratio, None, chg)
    # фильтр продолжения тренда: пропуск, если за 30м ДО спайка уже был рост объёма И цены
    if SPIKE_SKIP_PRETREND:
        _po, _pw = SPIKE_PRETREND_OFFSET_MIN, SPIKE_PRETREND_WIN_MIN
        if len(rows) >= _po + 2 * _pw:
            _cl = [r[2] for r in rows]; _vl = [r[3] for r in rows]
            _p0 = _cl[-_po - 1]; _pp = _cl[-_po - 1 - _pw]           # цена в начале спайка vs 30м раньше
            _pchg = (_p0 - _pp) / _pp * 100 if _pp > 0 else 0.0
            _vrec = sum(_vl[-_po - _pw:-_po]) / _pw                   # средний объём за 30м до спайка
            _vpre = sum(_vl[-_po - 2 * _pw:-_po - _pw]) / _pw         # средний объём за 30м до того
            _price_rose = _pchg >= SPIKE_PRETREND_PRICE_PCT
            _vol_rose   = _vpre > 0 and _vrec >= SPIKE_PRETREND_VOL_MULT * _vpre
            if _price_rose and _vol_rose:
                return (False, ratio, None, chg)                     # рост объёма+цены до спайка -> продолжение тренда, пропуск
    direction = 'long' if chg >= 0 else 'short'
    if SPIKE_FADE:                                      # против движения (mean-reversion)
        direction = 'short' if direction == 'long' else 'long'
    return (True, ratio, direction, chg)


def _spike_book(coin, market):
    """Лучшие bid/ask по REST (вызов только при сигнале). -> (bid, ask) | (None, None)."""
    sym = f'{coin}USDT'
    try:
        if market == 'bg_fut':
            data = bitget_get('/api/v2/mix/market/tickers',
                              params={'productType': 'USDT-FUTURES', 'symbol': sym})
            rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
            for d in rows:
                if str(d.get('symbol', '')).upper() == sym.upper() or len(rows) == 1:
                    return (float(d.get('bidPr') or 0) or None, float(d.get('askPr') or 0) or None)
        elif market == 'bb_fut':
            data = bybit_get('/v5/market/tickers', {'category': 'linear', 'symbol': sym})
            rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
            if rows:
                t = rows[0]
                return (float(t.get('bid1Price') or 0) or None, float(t.get('ask1Price') or 0) or None)
        elif market == 'bg_spot':
            sp = _fetch_spot_prices(coin)
            return (sp.get('bg_spot_bid'), sp.get('bg_spot_ask'))
    except Exception as e:
        log.debug(f'_spike_book {coin}/{market}: {e}')
    return (None, None)


def _spike_ensure_req(coin):
    """Лениво грузим _min_req[coin] для перпов (размер/округление). 1 раз на монету."""
    if coin not in _min_req:
        try:
            _min_req[coin] = {'bg': _load_bg_req(coin), 'bb': _load_bb_req(coin)}
        except Exception as e:
            log.debug(f'_spike_ensure_req {coin}: {e}')


def _spike_px_dec(coin, market):
    if market == 'bg_fut':
        return _min_req.get(coin, {}).get('bg', {}).get('price_place', 5)
    if market == 'bb_fut':
        return _min_req.get(coin, {}).get('bb', {}).get('price_scale', 5)
    return ((get_spot_specs(coin) or {}).get('bg') or {}).get('price_dec', 5)


def _spike_hold_stat(coin, market, pos, label=''):
    """Статистика удержания: одна строка в SPIKE_HOLD_STATS_FILE на каждое закрытие +
    сводка в лог (медиана времени с доливами / без)."""
    try:
        now = time.time()
        held = now - (pos.get('opened') or now)
        rungs = int(pos.get('rungs') or 1)
        rec = {'ts': now, 'coin': coin, 'market': market, 'side': pos.get('direction'),
               'label': label, 'held_sec': round(held), 'held_h': round(held/3600, 2),
               'rungs': rungs - 1, 'martingale': rungs > 1,
               'entry0': pos.get('entry0'), 'entry_avg': pos.get('entry_px'), 'qty': pos.get('qty'),
               'notional': round((pos.get('entry_px') or 0) * (pos.get('qty') or 0) * _spike_mult(coin, market), 2)}
        with open(SPIKE_HOLD_STATS_FILE, 'a', encoding='utf-8') as f:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
        # сводка по всему файлу
        hm, hn = [], []
        for ln in open(SPIKE_HOLD_STATS_FILE, encoding='utf-8'):
            r = json.loads(ln)
            (hm if r.get('martingale') else hn).append(r['held_h'])
        def _med(a): a = sorted(a); return a[len(a)//2] if a else 0.0
        log.info(f'[HOLD] {coin}/{market} {label}: держал {held/3600:.1f}ч, доливов {rungs-1} | '
                 f'статистика: без доливов n={len(hn)} медиана {_med(hn):.1f}ч, '
                 f'с доливами n={len(hm)} медиана {_med(hm):.1f}ч')
    except Exception as e:
        log.debug(f'hold-stat {coin}: {e}')


def _spike_hold_ban(coin, market, pos, label=''):
    """Позиция провисела дольше SPIKE_HOLD_BAN_SEC -> бан монеты на неделю; повторно -> навсегда."""
    _spike_hold_stat(coin, market, pos, label)         # статистика пишется ВСЕГДА, бан — по условию
    if not SPIKE_HOLD_BAN_SEC:
        return
    try:
        held = time.time() - (pos.get('opened') or time.time())
        if held < SPIKE_HOLD_BAN_SEC or coin in _spike_blacklist:
            return
        cnt = _spike_holdban_cnt.get(coin, 0) + 1
        _spike_holdban_cnt[coin] = cnt
        if SPIKE_HOLD_BAN_PERM and cnt >= SPIKE_HOLD_BAN_PERM:
            _spike_blacklist.add(coin)
            _spike_ban.pop(coin, None)
            log.warning(f'[BAN] {coin}: висела {held/3600:.1f}ч (случай №{cnt}) — ПЕРМАНЕНТНЫЙ бан '
                        f'(снимается вручную из {SPIKE_MARTBAN_FILE})')
        else:
            _spike_ban[coin] = max(_spike_ban.get(coin, 0), time.time() + SPIKE_HOLD_BAN_DAYS * 86400)
            _spike_ban_save()
            log.warning(f'[BAN] {coin}: висела {held/3600:.1f}ч > {SPIKE_HOLD_BAN_SEC/3600:.0f}ч '
                        f'(случай №{cnt}) — бан на {SPIKE_HOLD_BAN_DAYS}д (обе биржи)')
        _spike_martban_save()
    except Exception as e:
        log.debug(f'hold-ban {coin}: {e}')


def _spike_mult(coin, market):
    """Множитель контракта: у Bitget size — КОНТРАКТЫ, база = size * size_mult (DOGS=100, PEPE и т.п.).
    Без него деньги (оценка PnL, безубыток, номинал) считаются в N раз меньше. Bybit -> 1."""
    if market != 'bg_fut':
        return 1.0
    try:
        return float(_min_req.get(coin, {}).get('bg', {}).get('size_mult', 1.0) or 1.0)
    except Exception:
        return 1.0


def _spike_exch_qty(coin, market):
    """Фактический размер позиции на бирже (в тех же единицах, что qty у нас). -> float | None при ошибке."""
    try:
        if market == 'bb_fut':
            _rows = bybit_get('/v5/position/list', {'category': 'linear', 'symbol': f'{coin}USDT'})
            _rows = _rows if isinstance(_rows, list) else (_rows.get('list', []) if isinstance(_rows, dict) else [])
            return sum(abs(float(r.get('size', 0) or 0)) for r in _rows)
        if market == 'bg_fut':
            _rows = bitget_get('/api/v2/mix/position/single-position',
                               params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES', 'marginCoin': 'USDT'})
            _rows = _rows if isinstance(_rows, list) else [_rows] if _rows else []
            return sum(abs(float(r.get('total', 0) or r.get('available', 0) or 0)) for r in _rows)
    except Exception as e:
        log.debug(f'exch_qty {coin}/{market}: {e}')
    return None


def _spike_wait_qty(coin, market, want, tries=8, pause=0.4):
    """Ждём, пока биржа покажет позицию >= want (долив мог ещё не отразиться) -> фактический qty.
    Лечит гонку 'закрыть больше, чем открыто' (Bitget 40804 / Bybit 110017)."""
    real = None
    for _ in range(tries):
        real = _spike_exch_qty(coin, market)
        if real is not None and real >= want * 0.999:
            return real
        time.sleep(pause)
    return real if real else want


def _spike_size(coin, market, price, direction=None):
    """-> (size_or_qty, ok). Перпы: контракты/база; спот: база. Шорт на Bybit — округление ВВЕРХ до SPIKE_USDT_SHORT."""
    try:
        _leg = _spike_leg_usdt(market, direction)
        if market == 'bg_fut':
            return (calc_bg_size(coin, _leg, price), True)          # calc_bg_size сам добивает до >= 5 USDT
        if market == 'bb_fut':
            qty = calc_bb_qty(coin, _leg, price)
            if direction == 'short':                               # floor мог дать < 5 USDT -> биржа отобьёт; шаг вверх
                import math
                r = _min_req.get(coin, {}).get('bb', {})
                step = float(r.get('qty_step', r.get('min_qty', 0.001)) or 0.001)
                dec = len(f'{step:.10f}'.rstrip('0').split('.')[1]) if '.' in f'{step}' else 0
                while qty * price < SPIKE_USDT_SHORT:
                    qty = round(qty + step, dec)
            return (qty, True)
        if market == 'bg_spot':
            sp = (get_spot_specs(coin) or {}).get('bg')
            if not sp:
                return (0, False)
            import math
            step = sp['qty_step'] or 1e-6
            qty  = round(max(math.floor((_spike_leg_usdt(market) / price) / step) * step, sp['min_qty']), sp['qty_dec'])
            return (qty, qty * price >= max(sp['min_amt'], 1.0))
    except Exception as e:
        log.debug(f'_spike_size {coin}/{market}: {e}')
    return (0, False)


def _spike_entry(coin, market, direction, size, price, tif='gtc'):
    dec = _spike_px_dec(coin, market)
    px = round(price, dec)
    if market == 'bg_fut':
        # one-way mode: позиционный TP/SL отдельным вызовом не принимается ни одним эндпоинтом.
        # Рабочий способ — preset TP/SL НА ОРДЕРЕ ВХОДА (маркет на триггере). Считаем от цены входа.
        _tpp, _slp = _spike_pct(coin, market)
        tp_off = _tpp / 100.0; sl_off = _slp / 100.0
        if direction == 'long':
            ptp = round(px * (1 + tp_off), dec); psl = round(px * (1 - sl_off), dec)
        else:
            ptp = round(px * (1 - tp_off), dec); psl = round(px * (1 + sl_off), dec)
        _psl_use = None if SPIKE_EXIT_MODE == 'breakeven' else psl   # breakeven: без SL
        _ptp_use = None if SPIKE_EXIT_MODE == 'breakeven' else ptp   # breakeven: TP ставим отдельным GTC-лимитом, не пресетом
        return _place_gtc_bitget(coin, 'buy' if direction == 'long' else 'sell', size, px,
                                 reduce_only=False, preset_tp=_ptp_use, preset_sl=_psl_use, force=tif)
    if market == 'bb_fut':
        return _place_gtc_bybit(coin, 'Buy' if direction == 'long' else 'Sell', size, px, reduce_only=False, tif=tif)
    if market == 'bg_spot':
        return _place_spot_gtc_bg(coin, 'buy' if direction == 'long' else 'sell', size, px)
    return ''


def _spike_fill(coin, market, oid):
    if market == 'bg_fut':
        return _get_fill_bg(oid, coin) if _wait_filled_bg(oid, coin, timeout=SPIKE_FILL_WAIT) else {}
    if market == 'bb_fut':
        return _get_fill_bb(oid, coin) if _wait_filled_bb(oid, coin, timeout=SPIKE_FILL_WAIT) else {}
    if market == 'bg_spot':
        dl = time.time() + SPIKE_FILL_WAIT
        while time.time() < dl:
            f = _get_spot_fill_bg(coin, oid)
            if f:
                return f
            time.sleep(0.4)
    return {}


def _spike_cancel(coin, market, oid):
    if market == 'bg_fut':   _cancel_order_bitget(coin, oid)
    elif market == 'bb_fut': _cancel_order_bybit(coin, oid)
    elif market == 'bg_spot': _cancel_spot_bg(coin, oid)


def _spike_tp(coin, market, direction, qty, entry_px, price=None):
    if price is not None:
        tp = price                                   # явная цена (безубыток после мартингейл-долива)
    else:
        off = _spike_pct(coin, market)[0] / 100.0
        tp  = entry_px * (1 + off) if direction == 'long' else entry_px * (1 - off)
    tp  = round(tp, _spike_px_dec(coin, market))
    _coid = _safe_coid(f'spk{int(time.time()*1000)}{coin[:6]}', coin)   # один clientOid на ВСЕ попытки; без нелатиницы (40808)
    def _place():
        if market == 'bg_fut':
            try:
                return _place_gtc_bitget(coin, 'sell' if direction == 'long' else 'buy', qty, tp,
                                         reduce_only=True, client_oid=_coid)
            except Exception as _e:
                if '40786' not in str(_e):
                    raise
                for _w in (0.4, 1.0, 2.0):            # ордер уже на бирже -> вытаскиваем его orderId
                    try:
                        d = bitget_get('/api/v2/mix/order/detail',
                                       params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                               'clientOid': _coid})
                        _o = str((d or {}).get('orderId') or '')
                        if _o:
                            log.info(f'[SPIKE] {coin}/{market}: TP уже стоял (dup clientOid) -> orderId={_o}')
                            return _o
                    except Exception:
                        pass
                    try:
                        p = bitget_get('/api/v2/mix/order/orders-pending',
                                       params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES'})
                        for _e2 in (p or {}).get('entrustedList') or []:
                            if _e2.get('clientOid') == _coid:
                                _o = str(_e2.get('orderId') or '')
                                if _o:
                                    log.info(f'[SPIKE] {coin}/{market}: TP найден в открытых ордерах -> {_o}')
                                    return _o
                    except Exception:
                        pass
                    time.sleep(_w)
                return ''
        if market == 'bb_fut':
            try:
                return _place_gtc_bybit(coin, 'Sell' if direction == 'long' else 'Buy', qty, tp, reduce_only=True)
            except Exception as _e:
                if '110017' in str(_e) or 'position is zero' in str(_e).lower():
                    return 'no_position'      # позиция ещё не зарегана на Bybit -> ретрай
                raise
        if market == 'bg_spot':
            return _place_spot_gtc_bg(coin, 'sell', qty, tp)     # лонг закрываем продажей базы
        return ''
    _tries = SPIKE_TP_RETRIES_BB if market == 'bb_fut' else SPIKE_TP_RETRIES
    _pause = SPIKE_TP_RETRY_SEC_BB if market == 'bb_fut' else SPIKE_TP_RETRY_SEC
    oid = _place()
    # ретрай: reduce-only TP отлетает (22002/40804/110017 "нет позиции"), если позиция ещё не зарегана
    # на бирже после быстрого налива. 'no_position' — НЕ успех: ждём и пробуем снова.
    for _i in range(_tries):
        if oid and oid not in ('no_position', 'no_balance'):
            break
        time.sleep(_pause)
        oid = _place()
        if oid and oid not in ('no_position', 'no_balance'):
            log.info(f'[SPIKE] {coin}/{market}: TP поставлен с {_i + 2}-й попытки (позиция зарегалась)')
    if not oid or oid in ('no_position', 'no_balance'):
        log.warning(f'[SPIKE] {coin}/{market}: TP НЕ поставлен после {_tries + 1} попыток '
                    f'({_pause}с пауза) — позиция без TP, выход по безубытку')
        oid = ''
    return oid, tp


_spike_lev_set: set = set()   # (coin, market) с уже выставленным плечом SPIKE_LEVERAGE


def _spike_set_leverage(coin, market):
    """Принудительно выставить SPIKE_LEVERAGE перед входом (один раз на монету/рынок за сессию)."""
    if not SPIKE_LEVERAGE:
        return
    try:
        if MODE == 'SIMULATION':
            return
    except NameError:
        pass
    key = (coin, market)
    if key in _spike_lev_set:
        return
    lev = str(int(SPIKE_LEVERAGE))
    try:
        if market == 'bb_fut':                                  # Bybit отвечает ошибкой 110043, если плечо уже такое ->
            try:                                                # сперва читаем текущее и молча выходим при совпадении
                _rows = bybit_get('/v5/position/list', {'category': 'linear', 'symbol': f'{coin}USDT'})
                _rows = _rows if isinstance(_rows, list) else (_rows.get('list', []) if isinstance(_rows, dict) else [])
                if _rows and all(str(int(float(r.get('leverage') or 0))) == lev for r in _rows):
                    _spike_lev_set.add(key)
                    return
            except Exception:
                pass
            bybit_post('/v5/position/set-leverage', {
                'category': 'linear', 'symbol': f'{coin}USDT',
                'buyLeverage': lev, 'sellLeverage': lev})
        elif market == 'bg_fut':
            bitget_post('/api/v2/mix/account/set-leverage', {
                'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                'marginCoin': 'USDT', 'leverage': lev})
        else:
            return
        _spike_lev_set.add(key)
        log.info(f'[SPIKE] плечо {coin}/{market} -> {lev}x')
    except Exception as e:
        s = str(e)
        if any(x in s for x in ('110043', '45117', 'no change', 'not modified', 'leverage not modified')):
            _spike_lev_set.add(key)              # уже стоит нужное — считаем выполненным
        else:
            log.warning(f'[SPIKE] плечо {coin}/{market}: {e}')   # не блокируем вход


def _spike_server_tpsl(coin, market, direction, entry_px, qty):
    """Серверный TP/SL на бирже (биржа сама держит и закрывает позицию).
       Возвращает (tp_px, sl_px, ok). ok=False -> вызвать фолбэк (TP-лимит + вебсокет-стоп)."""
    tp_pct, sl_pct = _spike_pct(coin, market)
    off_tp = tp_pct / 100.0
    off_sl = sl_pct / 100.0
    if direction == 'long':
        tp = entry_px * (1 + off_tp); sl = entry_px * (1 - off_sl)
    else:
        tp = entry_px * (1 - off_tp); sl = entry_px * (1 + off_sl)
    dec = _spike_px_dec(coin, market)
    tp = round(tp, dec); sl = round(sl, dec)
    tps = f'{tp:.{dec}f}'; sls = f'{sl:.{dec}f}'
    try:
        if market == 'bb_fut':
            body = {'category': 'linear', 'symbol': f'{coin}USDT',
                    'takeProfit': tps,
                    'tpslMode': 'Full', 'tpTriggerBy': 'LastPrice',
                    'tpOrderType': 'Market', 'positionIdx': 0}
            if SPIKE_EXIT_MODE != 'breakeven':                       # breakeven: только TP, без SL
                body.update({'stopLoss': sls, 'slTriggerBy': 'LastPrice', 'slOrderType': 'Market'})
            resp = bybit_post('/v5/position/trading-stop', body)
            _sllog = sls if SPIKE_EXIT_MODE != 'breakeven' else '—нет'
            log.info(f'[SPIKE] серверный TP{"/SL" if SPIKE_EXIT_MODE!="breakeven" else " (без SL)"} Bybit {coin}: TP={tps} SL={_sllog} | resp={resp}')
            return tp, sl, True
        elif market == 'bg_fut':
            # TP/SL уже выставлены preset'ом НА ОРДЕРЕ ВХОДА (_spike_entry) — в one-way это
            # единственный рабочий способ. Биржа держит маркет-TP/SL сама, тут ничего не шлём.
            _sllog = sls if SPIKE_EXIT_MODE != 'breakeven' else '—нет'
            log.info(f'[SPIKE] серверный TP{"/SL" if SPIKE_EXIT_MODE!="breakeven" else " (без SL)"} Bitget {coin}: TP={tps} SL={_sllog} (preset на входе, маркет)')
            return tp, sl, True
        else:
            # спот: позиционный серверный TP/SL недоступен -> фолбэк на TP-лимит
            log.info(f'[SPIKE] {coin}/{market}: спот — серверный TP/SL недоступен, фолбэк на TP-лимит')
            return tp, sl, False
    except Exception as e:
        log.warning(f'[SPIKE] серверный TP/SL {coin}/{market} ОТКЛОНЁН биржей: {type(e).__name__}: {e} '
                    f'— ФОЛБЭК на TP-лимит + вебсокет-стоп (позиция НЕ останется без стопа)')
        return tp, sl, False


def _spike_open(coin, market, direction, ratio, chg):
    bid, ask = _spike_book(coin, market)
    if not bid or not ask:
        return
    entry_px = ask if direction == 'long' else bid             # вход через стакан (taker)
    ref_px = entry_px
    if SPIKE_ENTRY_SLIP_PCT > 0:                               # допуск: лимит ЗА рынок, чтобы IOC не срывался
        entry_px = entry_px * (1 + SPIKE_ENTRY_SLIP_PCT/100.0) if direction == 'long' \
                   else entry_px * (1 - SPIKE_ENTRY_SLIP_PCT/100.0)
    if market in ('bg_fut', 'bb_fut'):
        _spike_ensure_req(coin)
    size, ok = _spike_size(coin, market, entry_px, direction)
    if not ok or not size:
        log.warning(f'[SPIKE] {coin}/{market}: размер ниже минимума -> skip'); return

    # отсекаем высокономинальные монеты: мин. лот дороже лимита
    mult = float(_min_req.get(coin, {}).get('bg', {}).get('size_mult', 1.0)) if market == 'bg_fut' else 1.0
    notional = size * entry_px * mult
    _cap = _spike_mkt_cap(market)                                  # потолок биржи: уже занятое + эта позиция
    _busy = _spike_mkt_notional(market)
    if _busy + notional > _cap:
        log.info(f'[SPIKE] {coin}/{market}: занято {_busy:.0f} + {notional:.0f} > потолка биржи {_cap:.0f} USDT -> skip')
        return (0, False)
    _lim = SPIKE_MAX_NOTIONAL_X * _spike_leg_usdt(market)          # 2x ноги ЭТОЙ биржи
    if notional > _lim:
        log.info(f'[SPIKE] {coin}/{market}: мин. номинал {notional:.1f} USDT > лимита {_lim:.0f} (2x ноги) -> skip')
        _spike_cooldown[coin] = time.time(); return

    arrow = 'LONG' if direction == 'long' else 'SHORT'
    tag = ' fade' if SPIKE_FADE else ''
    _stp, _ssl = _spike_pct(coin, market)
    log.info(f'[SPIKE] СИГНАЛ{tag} {coin}/{market} {arrow} ×{ratio:.1f} Δ{chg:+.1f}% вход@{entry_px} size={size} TP={_stp}% SL={_ssl}%')

    # ── ФИЛЬТР ФАНДИНГА: входим только если фандинг НАМ платится (или не дороже порога) ──
    if SPIKE_FUNDING_FILTER:
        _rbg, _rbb = _fetch_funding_rates(coin)
        _rate = _rbg if market == 'bg_fut' else (_rbb if market == 'bb_fut' else None)
        if _rate is None:
            if SPIKE_FUNDING_REQUIRE:
                log.info(f'[SPIKE] {coin}/{market} {arrow}: ставка фандинга недоступна -> пропуск (require)')
                with _spike_lock: _spike_cooldown[coin] = time.time()
                return
        else:
            # фандинг В НАШУ ПОЛЬЗУ: short получает при ставке>0, long — при ставке<0
            _f_us = _rate if direction == 'short' else -_rate
            if _f_us < SPIKE_FUNDING_MIN:
                log.info(f'[SPIKE] {coin}/{market} {arrow}: фандинг ПРОТИВ нас '
                         f'(нам {_f_us*100:+.4f}%, ставка {_rate*100:+.4f}%, порог {SPIKE_FUNDING_MIN*100:+.4f}%) -> пропуск входа')
                with _spike_lock: _spike_cooldown[coin] = time.time()
                return
            log.info(f'[SPIKE] {coin}/{market} {arrow}: фандинг в нашу пользу (нам {_f_us*100:+.4f}%, ставка {_rate*100:+.4f}%) -> ок')

    if SPIKE_DRY_RUN:
        tp_px = entry_px * (1 + _spike_pct(coin, market)[0] / 100) if direction == 'long' else entry_px * (1 - _spike_pct(coin, market)[0] / 100)
        with _spike_lock:
            _spike_positions[(coin, market)] = {'coin': coin, 'market': market, 'direction': direction,
                                                'entry_px': entry_px, 'tp_px': tp_px, 'qty': size,
                                                'opened': time.time(), 'dry': True}
            _spike_cooldown[coin] = time.time()
        log.info(f'[SPIKE] СИМУЛЯЦИЯ открыто {coin}/{market} {arrow} @{entry_px} qty={size} -> TP@{tp_px:.8g}'); return

    _spike_set_leverage(coin, market)
    log.info(f'[SPIKE] вход → {arrow} лимит @{entry_px} (книга {ref_px}, допуск {SPIKE_ENTRY_SLIP_PCT}%) '
             f'size={size} (taker), отправляю')
    oid = _spike_entry(coin, market, direction, size, entry_px, tif=SPIKE_ENTRY_TIF)
    if not oid or oid in ('no_balance', 'no_position'):
        log.warning(f'[SPIKE] {coin}/{market}: вход не размещён ({oid!r})')
        if oid == 'no_balance':
            _spike_notify_no_balance(coin, market, direction)   # письмо: нет денег
        with _spike_lock:
            _spike_cooldown[coin] = time.time()              # пауза, чтобы не долбиться сразу снова
            _spike_fail[coin] = _spike_fail.get(coin, 0) + 1
            if SPIKE_FAIL_BAN and _spike_fail[coin] >= SPIKE_FAIL_MAX:
                _spike_blacklist.add(coin)
                log.warning(f'[SPIKE] {coin}: {_spike_fail[coin]} отказа входа подряд -> чёрный список на сессию (вероятно делистинг)')
        return
    log.info(f'[SPIKE] вход принят → oid={oid} ({SPIKE_ENTRY_TIF.upper()}), жду исполнения ≤{SPIKE_FILL_WAIT:.0f}с')
    fill = _spike_fill(coin, market, oid)
    if not fill or not fill.get('avg_price'):
        # IOC мог исполниться ЧАСТИЧНО и уйти в canceled — дочитываем филл принудительно
        try:
            fill = _get_fill_bg(oid, coin) if market == 'bg_fut' else (
                   _get_fill_bb(oid, coin) if market == 'bb_fut' else {})
        except Exception:
            fill = {}
    if not fill or not fill.get('avg_price'):
        try:
            _b2, _a2 = _spike_book(coin, market)
            _now_px = _a2 if direction == 'long' else _b2
            _gap = ((_now_px - ref_px)/ref_px*100) if direction == 'long' else ((ref_px - _now_px)/ref_px*100)
            _tail = f' | книга была {ref_px}, стала {_now_px}, ушла на {_gap:+.3f}% (допуск {SPIKE_ENTRY_SLIP_PCT}%)'
        except Exception:
            _tail = ''
        log.warning(f'[SPIKE] {coin}/{market}: вход {SPIKE_ENTRY_TIF.upper()} не исполнен (стакан ушёл) -> пропуск сигнала{_tail}')
        if (SPIKE_ENTRY_TIF or 'gtc').lower() != 'ioc':
            _spike_cancel(coin, market, oid)
        _spike_cooldown[coin] = time.time()    # кулдаун и при неисполненном входе -> нет дубль-сигнала на след. цикле
        return

    entry = float(fill['avg_price']); qty = float(fill.get('qty') or 0) or size
    # СВЕРКА С БИРЖЕЙ: отменённый IOC мог выглядеть исполненным -> фантомная позиция (APR/bb 15.08).
    try:
        _real = 0.0
        if market == 'bb_fut':
            _rows = bybit_get('/v5/position/list', {'category': 'linear', 'symbol': f'{coin}USDT'})
            _rows = _rows if isinstance(_rows, list) else (_rows.get('list', []) if isinstance(_rows, dict) else [])
            _real = sum(abs(float(r.get('size', 0) or 0)) for r in _rows)
        elif market == 'bg_fut':
            _rows = bitget_get('/api/v2/mix/position/single-position',
                               params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                       'marginCoin': 'USDT'})
            _rows = _rows if isinstance(_rows, list) else [_rows] if _rows else []
            _real = sum(abs(float(r.get('total', 0) or r.get('available', 0) or 0)) for r in _rows)
        else:
            _real = qty
        if _real <= 0:
            log.warning(f'[SPIKE] {coin}/{market}: ФАНТОМ — биржа не показывает позицию после входа '
                        f'(ордер отменён), запись отменена, сигнал пропущен')
            _spike_cooldown[coin] = time.time()
            return
        if abs(_real - qty) / max(qty, 1e-12) > 0.02:      # биржа знает точнее нашего расчёта
            log.info(f'[SPIKE] {coin}/{market}: qty по бирже {_real:.8g} (у нас {qty:.8g}) — берём биржевое')
            qty = _real
    except Exception as _e:
        log.debug(f'сверка позиции {coin}/{market}: {_e}')
    if SPIKE_EXIT_MODE == 'breakeven':
        # TP — GTC reduce-only ЛИМИТ в стакане (исполнится МЕЙКЕРОМ при достижении цены), БЕЗ SL.
        # Серверный пресет/маркет на Bitget часто не триггерится по торговой цене -> ставим лимит на цене TP.
        tp_oid, tp_px = _spike_tp(coin, market, direction, qty, entry)
        sl_px = None; server_ok = False
        _to_txt = ("иначе %dч по рынку" % (SPIKE_BREAKEVEN_MAX_SEC//3600)) if SPIKE_BREAKEVEN_MAX_SEC else "таймаута нет — держим до безубытка"
        log.info(f'[SPIKE] ОТКРЫТО {coin}/{market} {arrow} @{entry} qty={qty} -> GTC TP-лимит@{tp_px} oid={tp_oid} БЕЗ SL: '
                 f'если за {SPIKE_BREAKEVEN_AFTER_SEC//60}мин TP не исполнился -> безубыток(+фандинг), {_to_txt} — слот занят')
    else:
        log.info(f'[SPIKE] вход исполнен → @{entry} qty={qty}, ставлю серверный TP/SL (TP +{_spike_pct(coin, market)[0]}% / SL -{_spike_pct(coin, market)[1]}%)')
        # СЕРВЕРНЫЙ TP/SL: биржа сама держит и закрывает. Если биржа отклонит -> ФОЛБЭК на TP-лимит + вебсокет-стоп.
        tp_px, sl_px, server_ok = _spike_server_tpsl(coin, market, direction, entry, qty)
        tp_oid = None
        if server_ok:
            log.info(f'[SPIKE] ОТКРЫТО {coin}/{market} {arrow} @{entry} qty={qty} -> серверные TP@{tp_px} SL@{sl_px} — слот занят')
        else:
            # фолбэк: старая схема — TP-лимит + вебсокет/REST стоп
            tp_oid, tp_px = _spike_tp(coin, market, direction, qty, entry)
            log.warning(f'[SPIKE] ОТКРЫТО {coin}/{market} {arrow} @{entry} qty={qty} -> ФОЛБЭК TP-лимит@{tp_px} oid={tp_oid} + вебсокет-стоп — слот занят')
    with _spike_lock:
        _spike_positions[(coin, market)] = {'coin': coin, 'market': market, 'direction': direction,
                                            'entry_px': entry, 'entry0': entry, 'qty': qty, 'tp_oid': tp_oid,
                                            'tp_px': tp_px, 'sl_px': sl_px,
                                            'server_tpsl': server_ok, 'opened': time.time()}
        _spike_pos_save()
        _spike_cooldown[coin] = time.time()
        _spike_fail.pop(coin, None)             # успех -> сбросить счётчик отказов
    _spike_track_init(coin, market, _spike_positions[(coin, market)])
    _spike_entry_mail(coin, market, _spike_positions[(coin, market)])   # письмо+график на вход
    if (not server_ok) and SPIKE_EXIT_MODE != 'breakeven':
        _spike_ws_watch(coin, market)           # фолбэк: тики активной позиции -> точный стоп
def _spike_closed(coin, market, pos):
    if pos.get('dry'):
        bid, ask = _spike_book(coin, market)
        if not bid and not ask:
            return False
        d = pos['direction']; tp = pos['tp_px']
        px = (bid if d == 'long' else ask) or ask or bid      # сторона исполнения reduce-only TP
        hit = (px >= tp) if d == 'long' else (px <= tp)
        if hit:
            pos['close_px'], pos['close_reason'] = tp, 'TP'
            return True
        if SPIKE_DRY_MAX_HOLD_SEC and time.time() - pos['opened'] >= SPIKE_DRY_MAX_HOLD_SEC:
            pos['close_px'], pos['close_reason'] = px, 'таймаут'
            return True
        return False
    try:
        if market == 'bg_fut':   return not _bg_position_exists(coin)
        if market == 'bb_fut':   return coin not in _bb_positions_snapshot()
        if market == 'bg_spot':  return bool(_get_spot_fill_bg(coin, pos.get('tp_oid')))
    except Exception as e:
        log.warning(f'[SPIKE] {coin}/{market}: позиция НЕ проверена ({type(e).__name__}) — слот НЕ освобождаю (защита от ложного закрытия)')
    return False    # при сомнении слот НЕ освобождаем


def _spike_force_close(coin, market, pos):
    """Закрыть позицию рыночно: снять висящий TP + агрессивный reduce-only лимит через стакан. -> exit_px|None."""
    d = pos['direction']; qty = pos.get('qty') or 0
    try:
        if pos.get('tp_oid'):
            _spike_cancel(coin, market, pos['tp_oid'])          # снять висящий TP
    except Exception as e:
        log.debug(f'[SPIKE] cancel TP {coin}/{market}: {e}')
    bid, ask = _spike_book(coin, market)
    px = (bid if d == 'long' else ask) or ask or bid            # пересекаем стакан -> taker
    if not px:
        return None
    px = round(px, _spike_px_dec(coin, market))
    try:
        if market == 'bg_fut':
            oid = _place_gtc_bitget(coin, 'sell' if d == 'long' else 'buy', qty, px, reduce_only=True)
        elif market == 'bb_fut':
            oid = _place_gtc_bybit(coin, 'Sell' if d == 'long' else 'Buy', qty, px, reduce_only=True)
        elif market == 'bg_spot':
            oid = _place_spot_gtc_bg(coin, 'sell', qty, px)
        else:
            return None
        f = _spike_fill(coin, market, oid)
        return float(f['avg_price']) if f and f.get('avg_price') else px
    except Exception as e:
        log.warning(f'[SPIKE] force-close {coin}/{market}: {e}')
        return None


# ════════ WS-тики активной позиции: точный стоп по тикам (фолбэк на REST) ════════
SPIKE_WS_ENABLED   = True       # вебсокет-тики для точного стопа (False = только REST)
SPIKE_WS_FRESH_SEC = 2.0        # тик «свежий», если обновился не позже; иначе REST-фолбэк
SPIKE_WS_POLL_SEC  = 0.1        # частота быстрой проверки стопа по тикам

_spike_tick: dict = {}                       # (coin, market) -> (price, ts)
_spike_tick_lock = threading.Lock()
_spike_path: dict = {}                       # (coin, market) -> list[(ts, px)] путь цены откр. позиции
_spike_path_lock = threading.Lock()
SPIKE_PATH_SAMPLE_SEC = 0.5                  # шаг записи тика в путь (даунсемпл по времени)
SPIKE_PATH_MAX_PTS    = 4000                 # потолок точек пути на позицию
_spike_ws: dict = {}                         # 'bg'/'bb' -> {'app':WebSocketApp|None, 'sub':coin|None}
_spike_ws_lock = threading.Lock()
_BG_WS_URL = "wss://ws.bitget.com/v2/ws/public"
_BB_WS_URL = "wss://stream.bybit.com/v5/public/linear"


def _spike_ws_on_msg(ex, msg):
    if msg == "pong":
        return
    try:
        d = json.loads(msg)
        if isinstance(d, dict) and d.get("op") == "ping":
            slot = _spike_ws.get(ex)
            if slot and slot.get("app"):
                try: slot["app"].send(json.dumps({"op": "pong"}))
                except Exception: pass
            return
        if ex == 'bg':
            arg = d.get("arg", {})
            if arg.get("channel") == "trade" and d.get("data"):
                coin = arg.get("instId", "")[:-4]
                row = max(d["data"], key=lambda r: int(r[0]))
                with _spike_tick_lock:
                    _spike_tick[(coin, 'bg_fut')] = (float(row[1]), time.time())
        else:
            topic = d.get("topic", "")
            if topic.startswith("publicTrade.") and d.get("data"):
                coin = topic.split(".")[1][:-4]
                with _spike_tick_lock:
                    _spike_tick[(coin, 'bb_fut')] = (float(d["data"][-1]["p"]), time.time())
    except Exception:
        pass


def _spike_ws_ping(w, ex):
    while True:
        try:
            w.send("ping" if ex == 'bg' else json.dumps({"op": "ping"}))
        except Exception:
            break
        time.sleep(20)


def _spike_ws_run(ex):
    try:
        import websocket
        url = _BG_WS_URL if ex == 'bg' else _BB_WS_URL
        def _open(w):
            with _spike_ws_lock:
                if ex in _spike_ws:
                    _spike_ws[ex]['app'] = w
            threading.Thread(target=_spike_ws_ping, args=(w, ex), daemon=True).start()
        while not _spike_stop.is_set():
            try:
                websocket.WebSocketApp(url, on_message=lambda w, m: _spike_ws_on_msg(ex, m), on_open=_open).run_forever(ping_interval=20, ping_timeout=10)
            except Exception as e:
                log.debug(f'[SPIKE] ws {ex} обрыв, переподключение: {e}')
            # соединение упало -> сбросить app, чтобы самопроверка/стоп видели реальное состояние
            with _spike_ws_lock:
                if ex in _spike_ws:
                    _spike_ws[ex]['app'] = None
            if not _spike_stop.is_set():
                time.sleep(2)
    except Exception as e:
        log.warning(f'[SPIKE] WS-поток {ex} АВАРИЙНО завершился: {type(e).__name__}: {e}')


def _spike_ws_start():
    if not SPIKE_WS_ENABLED:
        log.info('[SPIKE] WS-тики ВЫКЛ (SPIKE_WS_ENABLED=False) — работает REST-стоп')
        return
    # защита от двойного запуска (напр. повторный вызов вручную)
    alive = [t for t in threading.enumerate() if t.name in ('spike-ws-bg', 'spike-ws-bb', 'spike-fast-stop')]
    if alive:
        log.info(f'[SPIKE] WS уже запущен ({len(alive)} потоков) — повторный старт пропущен')
        return
    # проверка пакета
    try:
        import websocket
        _ver = getattr(websocket, '__version__', '?')
        if not hasattr(websocket, 'WebSocketApp'):
            log.warning('[SPIKE] установлен пакет websocket (НЕ websocket-client) — точный стоп ВЫКЛ, REST-фолбэк. '
                        'Исправить: pip uninstall websocket && pip install websocket-client')
            return
    except Exception:
        log.warning('[SPIKE] websocket-client не установлен — точный стоп ВЫКЛ, работает REST-стоп (pip install websocket-client)')
        return
    for ex in ('bg', 'bb'):
        with _spike_ws_lock:
            _spike_ws[ex] = {'app': None, 'sub': None}
        threading.Thread(target=_spike_ws_run, args=(ex,), daemon=True, name=f'spike-ws-{ex}').start()
    threading.Thread(target=_spike_fast_stop, daemon=True, name='spike-fast-stop').start()
    # САМОПРОВЕРКА: ждать подключения до 8с, опрашивая статус (Bybit поднимается дольше Bitget)
    st = {'bg': 'НЕТ соединения', 'bb': 'НЕТ соединения'}
    _deadline = time.time() + 8.0
    while time.time() < _deadline:
        time.sleep(0.5)
        with _spike_ws_lock:
            st = {ex: ('подключён' if _spike_ws.get(ex, {}).get('app') else 'НЕТ соединения') for ex in ('bg', 'bb')}
        if 'НЕТ соединения' not in st.values():
            break                                  # обе биржи поднялись — не ждём дальше
    threads = sorted(t.name for t in threading.enumerate() if 'spike-ws' in t.name or t.name == 'spike-fast-stop')
    log.info(f'[SPIKE] WS-тики включены | websocket-client {_ver} | bg={st["bg"]} bb={st["bb"]} | потоки={threads}')
    if 'НЕТ соединения' in st.values():
        log.warning('[SPIKE] часть WS НЕ подключилась — по этой бирже стоп уйдёт в REST-фолбэк '
                    '(возможна блокировка WS-эндпоинта на VPS: фаервол/гео)')


def _spike_ws_watch(coin, market):
    """Подписаться на тики монеты в позиции (сняв предыдущую подписку)."""
    ex = 'bg' if market == 'bg_fut' else ('bb' if market == 'bb_fut' else None)
    if ex is None:
        return
    with _spike_ws_lock:
        slot = _spike_ws.get(ex)
        if not slot or not slot.get('app'):
            return
        app = slot['app']; prev = slot.get('sub'); slot['sub'] = coin
    try:
        if ex == 'bg':
            if prev:
                app.send(json.dumps({"op": "unsubscribe", "args": [{"instType": "USDT-FUTURES", "channel": "trade", "instId": f"{prev}USDT"}]}))
            app.send(json.dumps({"op": "subscribe", "args": [{"instType": "USDT-FUTURES", "channel": "trade", "instId": f"{coin}USDT"}]}))
        else:
            if prev:
                app.send(json.dumps({"op": "unsubscribe", "args": [f"publicTrade.{prev}USDT"]}))
            app.send(json.dumps({"op": "subscribe", "args": [f"publicTrade.{coin}USDT"]}))
    except Exception as e:
        log.debug(f'[SPIKE] ws watch {coin}/{market}: {e}')


def _spike_ws_unwatch(market):
    ex = 'bg' if market == 'bg_fut' else ('bb' if market == 'bb_fut' else None)
    if ex is None:
        return
    with _spike_ws_lock:
        slot = _spike_ws.get(ex)
        if not slot or not slot.get('app'):
            return
        app = slot['app']; prev = slot.get('sub'); slot['sub'] = None
    if not prev:
        return
    try:
        if ex == 'bg':
            app.send(json.dumps({"op": "unsubscribe", "args": [{"instType": "USDT-FUTURES", "channel": "trade", "instId": f"{prev}USDT"}]}))
        else:
            app.send(json.dumps({"op": "unsubscribe", "args": [f"publicTrade.{prev}USDT"]}))
    except Exception:
        pass


def _spike_fast_stop():
    """Быстрый цикл: стоп по ЖИВОМУ тику (если свежий); иначе REST-стоп в _spike_release сработает как фолбэк."""
    try:
        while not _spike_stop.is_set():
            time.sleep(SPIKE_WS_POLL_SEC)
            if not SPIKE_STOP_PCT:
                continue
            with _spike_lock:
                items = [(k, v) for k, v in _spike_positions.items() if not v.get('dry')]
            now = time.time()
            for (coin, market), pos in items:
                if market not in ('bg_fut', 'bb_fut'):
                    continue
                with _spike_tick_lock:
                    tk = _spike_tick.get((coin, market))
                if not tk or now - tk[1] > SPIKE_WS_FRESH_SEC:
                    continue                                   # нет свежего тика -> REST-фолбэк
                px = tk[0]; d = pos['direction']; e = pos.get('entry_px') or 0
                with _spike_path_lock:                       # запись пути цены (пост-анализ TP/SL)
                    _buf = _spike_path.setdefault((coin, market), [])
                    if (not _buf) or (now - _buf[-1][0] >= SPIKE_PATH_SAMPLE_SEC):
                        _buf.append((now, px))
                        if len(_buf) > SPIKE_PATH_MAX_PTS:
                            del _buf[0:len(_buf) - SPIKE_PATH_MAX_PTS]
                if not e:
                    continue
                chg = (px - e) / e * 100 if d == 'long' else (e - px) / e * 100
                if SPIKE_DISASTER_ENABLE and chg <= -SPIKE_DISASTER_PCT:   # КАТАСТРОФА-СТОП: защита счёта, независимо от режима
                    with _spike_lock:
                        if (coin, market) not in _spike_positions:
                            continue                                       # уже закрыта
                        _spike_positions.pop((coin, market), None); _spike_pos_save()
                        _spike_cooldown[coin] = now
                        _spike_ban[coin] = now + SPIKE_STOP_BAN_SEC
                        _spike_ban_save()
                    xp = _spike_force_close(coin, market, pos) or px
                    g = (xp - e) / e * 100 if d == 'long' else (e - xp) / e * 100
                    usdt = e * (pos.get('qty') or 0) * _spike_mult(coin, market) * (g - SPIKE_FEE_PCT) / 100
                    log.warning(f'[SPIKE] КАТАСТРОФА-СТОП {coin}/{market} {d} вход@{e} выход@{xp:.8g} '
                                f'({g:+.1f}% / {usdt:+.3f} USDT) — просадка глубже -{SPIKE_DISASTER_PCT:.0f}%, '
                                f'слот свободен, {coin} в бан на {SPIKE_STOP_BAN_SEC // 86400}д')
                    _spike_ws_unwatch(market)
                    _spike_track_close(coin, market, pos, usdt, 'КАТАСТРОФА-СТОП')
                    _spike_hold_ban(coin, market, pos, 'КАТАСТРОФА-СТОП')
                    _spike_verify_pnl(coin, market, pos['opened'], usdt, 'КАТАСТРОФА-СТОП')
                    continue
                if SPIKE_EXIT_MODE != 'breakeven' and chg <= -_spike_pct(coin, market)[1]:
                    with _spike_lock:                          # критическая секция: захватить позицию
                        if (coin, market) not in _spike_positions:
                            continue                            # уже закрыта (REST-веткой) — пропустить
                        _spike_positions.pop((coin, market), None); _spike_pos_save()
                        _spike_cooldown[coin] = now
                        _spike_ban[coin] = now + SPIKE_STOP_BAN_SEC
                        _spike_ban_save()
                    xp = _spike_force_close(coin, market, pos) or px
                    gross = (xp - e) / e * 100 if d == 'long' else (e - xp) / e * 100
                    net = gross - SPIKE_FEE_PCT; usdt = e * (pos.get('qty') or 0) * _spike_mult(coin, market) * net / 100
                    log.warning(f'[SPIKE] СТОП(тик) {coin}/{market} {d} вход@{e} выход@{xp:.8g} '
                                f'({net:+.2f}% / {usdt:+.3f} USDT, держал {now - pos["opened"]:.0f}с) '
                                f'— слот свободен, {coin} в бан на {SPIKE_STOP_BAN_SEC//86400}д')
                    _spike_ws_unwatch(market)
                    _spike_track_close(coin, market, pos, usdt, 'СТОП(тик)')
                    _spike_hold_ban(coin, market, pos, 'СТОП(тик)')
                    _spike_verify_pnl(coin, market, pos['opened'], usdt, 'СТОП(тик)')   # сверка с реальным PnL биржи
                    _spike_record_close(coin, market, pos, net, usdt, 'СТОП(тик)')   # путь + анализ TP/SL

    except Exception as e:
        log.warning(f'[SPIKE] поток быстрого стопа АВАРИЙНО завершился: {type(e).__name__}: {e}')


def _spike_pos_funding_pct(coin, market, pos, now):
    """Накопленный фандинг открытой позиции в % от номинала (знак: + получили нам). Кэш + троттлинг."""
    if (now - pos.get('_fund_ts', 0)) < SPIKE_FUNDING_POLL_SEC and '_fund_pct' in pos:
        return pos['_fund_pct']
    e = pos.get('entry_px') or 0; qty = (pos.get('qty') or 0) * _spike_mult(coin, market); notl = e * qty
    fpct = pos.get('_fund_pct', 0.0)
    if notl > 0 and not pos.get('dry'):
        try:
            sm = int(pos['opened'] * 1000); em = int(now * 1000)
            f = _fetch_funding_bb(coin, sm, em) if market == 'bb_fut' else _fetch_funding_bg(coin, sm, em)
            if f is not None:
                fpct = float(f) / notl * 100.0
        except Exception as e2:
            log.debug(f'[SPIKE] funding {coin}/{market}: {e2}')
    pos['_fund_ts'] = now; pos['_fund_pct'] = fpct
    return fpct

def _spike_martingale_check(coin, market, pos, px, now):
    """Мартингейл ПО ЦЕНЕ (ТЗ): уход против ПЕРВОГО входа на 5% -> долив (= нога), на 10% -> второй
    долив. После долива выход ТОЛЬКО в безубыток: GTC-лимит на цене безубытка + _spike_breakeven_check.
    Фазы времени и предохранители убраны по ТЗ."""
    if pos.get('dry'):
        return
    e = pos.get('entry_px') or 0; d = pos['direction']; age = now - pos['opened']
    if not (px and e):
        return
    if pos.get('rungs', 1) >= SPIKE_MART_RUNGS:                    # рунги исчерпаны (вход + 2 долива)
        return
    if SPIKE_MART_ONE_COIN_MKT:                                    # уже доливаем другую монету на этой бирже?
        with _spike_lock:
            _busy_coin = next((c_ for (c_, m_), p in _spike_positions.items()
                               if m_ == market and c_ != coin and (p.get('rungs', 1) or 1) > 1), None)
        if _busy_coin:
            if not pos.get('_one_noavg'):
                log.warning(f'[SPIKE] {coin}/{market}: мартингейл уже идёт по {_busy_coin} — '
                            f'долив отменён (одна монета на биржу), доторговываем до БУ')
                pos['_one_noavg'] = True
            return
    _add_notional = _spike_leg_usdt(market, pos.get('direction'))  # долив = ещё одна нога (у шорта — 5 USDT)
    if _spike_mkt_notional(market) + _add_notional > _spike_mkt_cap(market):
        if not pos.get('_cap_noavg'):
            log.warning(f'[SPIKE] {coin}/{market}: долив отменён — потолок биржи '
                        f'{_spike_mkt_cap(market):.0f} USDT (занято {_spike_mkt_notional(market):.0f}), '
                        f'доторговываем до БУ')
            pos['_cap_noavg'] = True
        return
    if SPIKE_MART_STOP_IF_MAXED:                                   # на бирже есть позиция с исчерпанным мартингейлом?
        with _spike_lock:
            _maxed = [c_ for (c_, m_), p in _spike_positions.items()
                      if m_ == market and c_ != coin and (p.get('rungs', 1) or 1) >= SPIKE_MART_RUNGS]
        if _maxed:
            if not pos.get('_mkt_noavg'):
                log.warning(f'[SPIKE] {coin}/{market}: {_maxed[0]} на этой бирже исчерпала мартингейл — '
                            f'доливы по остальным позициям запрещены, доторговываем до БУ')
                pos['_mkt_noavg'] = True
            return
    if now < pos.get('next_avg', 0):                              # антиспам повторов при неудачном доливе
        return
    if (coin, market) in _ARBITRON_BLOCKED:                        # Arbitron: делистинг -> НЕ доливаем
        if not pos.get('_arb_noavg'):
            log.warning(f'[ARBITRON] {coin}/{market}: помечена к делистингу — долив отменён, ждём выхода')
            pos['_arb_noavg'] = True
        return
    e0 = pos.get('entry0') or e                                    # ПЕРВЫЙ вход (усреднение его не двигает)
    gross = (px - e) / e * 100 if d == 'long' else (e - px) / e * 100
    adverse = (e0 - px) / e0 * 100 if d == 'long' else (px - e0) / e0 * 100
    need = pos.get('rungs', 1) * SPIKE_MART_TRIG_PCT               # колено 1: 10% от входа, колено 2: 20%
    # долив ПО ВРЕМЕНИ: цель уже БЕЗУБЫТОК (2ч без TP или было колено), но БУ не исполняется ещё 2ч
    _since = pos.get('_last_rung_ts') or pos.get('_be_ts') or 0
    time_due = SPIKE_MART_TIME_SEC > 0 and _since > 0 and (now - _since) >= SPIKE_MART_TIME_SEC and gross < 0
    if adverse < need and not time_due:
        return
    # ── ДОЛИВ: равный шаг = стандартная нога (_spike_size), по текущей цене ──
    try:
        add_size, ok = _spike_size(coin, market, px, pos.get('direction'))
        if not ok or not add_size or add_size <= 0:
            pos['next_avg'] = now + SPIKE_MART_STEP_SEC; return
        rungs = pos.get('rungs', 1)
        oid = _spike_entry(coin, market, d, add_size, px)
        if not oid or oid in ('no_balance', 'no_position'):
            log.warning(f'[SPIKE] {coin}/{market}: МАРТИНГЕЙЛ — долив не размещён ({oid!r}), позже')
            pos['next_avg'] = now + SPIKE_MART_STEP_SEC; return
        fill = _spike_fill(coin, market, oid)
        if not fill or not fill.get('avg_price'):
            log.warning(f'[SPIKE] {coin}/{market}: МАРТИНГЕЙЛ — долив не исполнился за {SPIKE_FILL_WAIT}с, отмена')
            _spike_cancel(coin, market, oid)
            pos['next_avg'] = now + SPIKE_MART_STEP_SEC; return
        fill_px = float(fill['avg_price']); fill_qty = float(fill.get('qty') or 0) or add_size
        old_qty = pos.get('qty') or 0; new_qty = old_qty + fill_qty
        new_entry = (e * old_qty + fill_px * fill_qty) / new_qty if new_qty > 0 else e
        with _spike_lock:
            pos['entry_px'] = new_entry; pos['qty'] = new_qty
            pos['rungs'] = rungs + 1; pos['next_avg'] = now + SPIKE_MART_STEP_SEC; pos['_last_rung_ts'] = now
        _spike_pos_save()
        if pos['rungs'] >= SPIKE_MART_RUNGS and SPIKE_MART_BAN_DAYS > 0:      # обе ступени израсходованы
            _cnt = _spike_martban_cnt.get(coin, 0) + 1
            _spike_martban_cnt[coin] = _cnt
            if SPIKE_MART_BAN_PERM and _cnt >= SPIKE_MART_BAN_PERM:           # повторный мартингейл-бан -> навсегда
                _spike_blacklist.add(coin)
                _spike_ban.pop(coin, None)
                log.warning(f'[BAN] {coin}: мартингейл-бан №{_cnt} — ПЕРМАНЕНТНЫЙ бан (обе биржи, снимается только '
                            f'вручную из {SPIKE_MARTBAN_FILE}; позиция доторговывается до БУ)')
            else:
                _spike_ban[coin] = now + SPIKE_MART_BAN_DAYS * 86400
                _spike_ban_save()                                             # бан ПЕРЕЖИВАЕТ рестарт (раньше не сохранялся)
                log.warning(f'[BAN] {coin}: обе ступени мартингейла израсходованы (бан №{_cnt}) — бан на '
                            f'{SPIKE_MART_BAN_DAYS}д (обе биржи, позиция доторговывается до БУ)')
            _spike_martban_save()
            pos.pop('_fund_pct', None); pos.pop('_fund_ts', None)   # сбросить кэш фандинга (нога выросла)
        log.warning(f'[SPIKE] МАРТИНГЕЙЛ {coin}/{market} долив #{rungs} @{fill_px:.8g} '
                    f'-> нов.средняя {new_entry:.8g}, qty={new_qty:.8g}, ступень {rungs + 1}/{SPIKE_MART_RUNGS} '
                    f'(было {gross:.1f}%)')
        # после долива цель = БЕЗУБЫТОК от новой средней (комиссия круга), TP больше не ставим
        if pos.get('tp_oid'):
            _spike_cancel(coin, market, pos['tp_oid'])
        be_px = new_entry * (1 + SPIKE_FEE_PCT / 100.0) if d == 'long' else new_entry * (1 - SPIKE_FEE_PCT / 100.0)
        _real_qty = _spike_wait_qty(coin, market, new_qty)            # биржа могла ещё не учесть долив -> 40804/110017
        if abs(_real_qty - new_qty) / max(new_qty, 1e-12) > 0.005:
            log.info(f'[SPIKE] {coin}/{market}: qty по бирже {_real_qty:.8g} (у нас {new_qty:.8g}) — БУ ставим на биржевой')
            with _spike_lock:
                pos['qty'] = _real_qty
            new_qty = _real_qty
        tp_oid, tp_px = _spike_tp(coin, market, d, new_qty, new_entry, price=be_px)
        with _spike_lock:
            pos['tp_oid'] = tp_oid; pos['tp_px'] = tp_px
        log.info(f'[SPIKE] {coin}/{market}: выход переставлен на БЕЗУБЫТОК @{tp_px:.8g} (qty={new_qty:.8g})')
        _spike_pos_save()
        _spike_track_rung(coin, market, pos, new_entry)
        if SPIKE_RUNG_BAN_N:                                   # рецидивист: N колен за окно -> бан
            try:
                _h = [t for t in _spike_rung_hist.get(coin, []) if now - t < SPIKE_RUNG_BAN_WIN]
                _h.append(now); _spike_rung_hist[coin] = _h
                if len(_h) >= SPIKE_RUNG_BAN_N and coin not in _spike_blacklist:
                    _spike_ban[coin] = max(_spike_ban.get(coin, 0), now + SPIKE_RUNG_BAN_DAYS * 86400)
                    _spike_ban_save()
                    _spike_rung_hist[coin] = []                # счётчик обнуляем, чтобы не банить каждым следующим коленом
                    log.warning(f'[BAN] {coin}: {len(_h)} колен за {SPIKE_RUNG_BAN_WIN//86400}д — бан на '
                                f'{SPIKE_RUNG_BAN_DAYS}д (обе биржи, позиция доторговывается)')
                _spike_martban_save()
            except Exception as _e:
                log.debug(f'rung-ban {coin}: {_e}')
        _spike_mart_mail(coin, market, pos)
    except Exception as ex:
        log.error(f'[SPIKE] {coin}/{market}: МАРТИНГЕЙЛ ошибка долива: {type(ex).__name__}: {ex}')
        pos['next_avg'] = now + SPIKE_MART_STEP_SEC


def _spike_breakeven_check(coin, market, pos, px, now):
    """Режим breakeven: TP держит биржа. Здесь — выход в безубыток ПОСЛЕ грейс-периода (если TP не сработал).
    Рыночный таймаут — ТОЛЬКО если SPIKE_BREAKEVEN_MAX_SEC>0 (0 = без таймаута, держим до TP/безубытка).
    net% = ход_цены% - комиссия% + полученный_фандинг%."""
    e = pos.get('entry_px') or 0; d = pos['direction']; age = now - pos['opened']
    # ДОБИВАНИЕ TP ПО РЫНКУ: серверный TP на Bitget (preset) часто не триггерится по торговой цене —
    # цена пробивает TP, а позиция висит. Если цена ДОСТИГЛА уровня TP, а позиция ещё открыта —
    # закрываем сами по рынку (метка TP, бан не ставится). Работает с 1-й секунды, грейс не нужен. SL нет.
    tp_px = pos.get('tp_px') or 0
    if px and tp_px and not pos.get('tp_oid'):     # только если GTC TP-лимит не стоит (не поставился) -> добиваем по рынку; стоит -> исполнится мейкером сам
        hit_tp = (px >= tp_px) if d == 'long' else (px <= tp_px)
        if hit_tp:
            return ('TP', px)
    timed_out = bool(SPIKE_BREAKEVEN_MAX_SEC) and age >= SPIKE_BREAKEVEN_MAX_SEC
    to_label = f'ТАЙМАУТ{SPIKE_BREAKEVEN_MAX_SEC//3600}ч'
    if not (px and e):
        return (to_label, None) if timed_out else None
    if pos.get('rungs', 1) > 1 or age >= SPIKE_BREAKEVEN_AFTER_SEC:    # безубыток: после колена ИЛИ через 2ч без TP
        gross = (px - e) / e * 100 if d == 'long' else (e - px) / e * 100
        fund = _spike_pos_funding_pct(coin, market, pos, now)
        net = gross - SPIKE_FEE_PCT + fund
        if net >= SPIKE_BREAKEVEN_MARGIN:
            return ('БЕЗУБЫТОК', px)
    if timed_out:
        return (to_label, px)
    return None


def _spike_release():
    with _spike_lock:
        items = list(_spike_positions.items())
    for (coin, market), pos in items:
        if _spike_closed(coin, market, pos):
            with _spike_lock:
                _spike_positions.pop((coin, market), None); _spike_pos_save()
                _spike_cooldown[coin] = time.time()
            _spike_ws_unwatch(market)
            if pos.get('dry'):
                d = pos['direction']; e = pos['entry_px']; cp = pos.get('close_px', e)
                gross = (cp - e) / e * 100 if d == 'long' else (e - cp) / e * 100
                net = gross - SPIKE_FEE_PCT
                net_usdt = e * (pos.get('qty') or 0) * _spike_mult(coin, market) * net / 100
                held = time.time() - pos['opened']
                log.info(f'[SPIKE] СИМУЛЯЦИЯ закрыто {coin}/{market} {pos.get("close_reason","?")} вход@{e} выход@{cp:.8g} '
                         f'({net:+.2f}% / {net_usdt:+.3f} USDT, держал {held:.0f}с) — слот свободен')
            else:
                d = pos['direction']; e = pos.get('entry_px') or 0; qty = pos.get('qty') or 0
                held = time.time() - pos['opened']; sv = pos.get('server_tpsl')
                exitpx = None; net = None; net_usdt = None; real_known = False
                if sv:
                    # серверный TP/SL: позиция закрыта БИРЖЕЙ. ВАЖНО: НЕ оцениваем выход по книге —
                    # в быстром рынке книга уезжает на 2-3% от реального филла, и прибыльный TP
                    # помечался как СТОП и монета банилась по ошибке (инцидент BP +0.026 -> "стоп" -2.43%).
                    # Берём РЕАЛЬНЫЙ PnL с биржи (пара попыток с задержкой), метку/бан — по его знаку.
                    rp = {'count': 0, 'pnl': 0.0, 'exit': None}
                    for _try in range(3):
                        try:
                            rp = (spike_real_pnl_bb if market == 'bb_fut' else spike_real_pnl_bg)(coin, pos['opened'])
                        except Exception as ex:
                            log.debug(f'[SPIKE] real-pnl {coin}: {ex}'); rp = {'count': 0, 'pnl': 0.0, 'exit': None}
                        if rp.get('count'):
                            break
                        time.sleep(2)
                    if rp.get('count'):
                        net_usdt = rp['pnl']; exitpx = rp.get('exit')
                        net = (net_usdt / (e * qty * _spike_mult(coin, market)) * 100) if (e and qty) else 0.0
                        real_known = True
                    else:
                        # реал ещё недоступен -> оценка по книге ТОЛЬКО для лога, БЕЗ бана
                        bid, ask = _spike_book(coin, market)
                        exitpx = (bid if d == 'long' else ask) or ask or bid
                else:
                    # фолбэк: выход по TP-лимиту -> берём фактический филл
                    try:
                        oid = pos.get('tp_oid')
                        if   market == 'bg_fut': f = _get_fill_bg(oid, coin)
                        elif market == 'bb_fut': f = _get_fill_bb(oid, coin)
                        else:                    f = _get_spot_fill_bg(coin, oid)
                        if f and f.get('avg_price'):
                            exitpx = float(f['avg_price'])
                    except Exception as ex:
                        log.debug(f'[SPIKE] exit-fill {coin}/{market}: {ex}')
                if (net is None) and exitpx and e:
                    gross = (exitpx - e) / e * 100 if d == 'long' else (e - exitpx) / e * 100
                    net = gross - SPIKE_FEE_PCT
                    net_usdt = e * qty * _spike_mult(coin, market) * net / 100
                if net is not None:
                    # метка/бан ТОЛЬКО по подтверждённому реальному убытку:
                    #  - sv + реал известен и убыток -> СТОП(сервер) + бан
                    #  - sv + реал НЕ известен -> не баним (уточнится в PNL-CHECK), помечаем нейтрально
                    #  - фолбэк (TP-лимит) с убытком -> как стоп
                    do_ban = ((sv and real_known and net_usdt < 0) or ((not sv) and net < 0)) and SPIKE_STOP_BAN_SEC > 0
                    if sv:
                        label = 'СТОП(сервер)' if (real_known and net_usdt < 0) else ('TP' if real_known else 'закрыт(реал уточняется)')
                    else:
                        label = 'СТОП(сервер)' if net < 0 else 'TP'
                    if do_ban:
                        with _spike_lock:
                            _spike_ban[coin] = time.time() + SPIKE_STOP_BAN_SEC
                            _spike_ban_save()
                    fn = log.warning if do_ban else log.info
                    ban_note = f', {coin} в бан на {SPIKE_STOP_BAN_SEC//86400}д' if do_ban else ''
                    src_note = '' if real_known or not sv else ' [оценка по книге]'
                    fn(f'[SPIKE] закрыто {coin}/{market} {label} вход@{e} выход@{(exitpx or 0):.8g} '
                       f'({net:+.2f}% / {net_usdt:+.3f} USDT, держал {held:.0f}с) — слот свободен{ban_note}{src_note}')
                    _spike_track_close(coin, market, pos, net_usdt, label)
                    _spike_hold_ban(coin, market, pos, label)
                    _spike_verify_pnl(coin, market, pos['opened'], net_usdt, label)   # сверка с реальным PnL биржи
                    _spike_record_close(coin, market, pos, net, net_usdt, label)   # путь + анализ TP/SL при минусе
                else:
                    _spike_hold_ban(coin, market, pos, 'ЗАКРЫТ(цена неизвестна)')
                    _spike_track_close(coin, market, pos, 0.0, 'ЗАКРЫТ(цена неизвестна)')
                    log.info(f'[SPIKE] закрыто {coin}/{market} — выход по серверному TP/SL / цена неизвестна '
                             f'(держал {held:.0f}с) — слот свободен')
        else:
            now = time.time(); d = pos['direction']; e = pos.get('entry_px') or 0
            bid, ask = _spike_book(coin, market)
            px = (bid if d == 'long' else ask) or ask or bid
            chg = ((px - e) / e * 100 if d == 'long' else (e - px) / e * 100) if (px and e) else None
            # стоп-лосс: движение против входа на SPIKE_STOP_PCT
            # для серверного TP/SL стоп держит БИРЖА -> REST-стоп НЕ дёргаем (иначе двойное закрытие)
            forced = None   # ('СТОП'|'ТАЙМАУТ'|'БЕЗУБЫТОК'|'ТАЙМАУТ12ч', exit_px)
            if SPIKE_EXIT_MODE == 'breakeven':
                # 2ч без TP и без колен -> цель = БЕЗУБЫТОК: снимаем TP, ставим GTC на цене безубытка
                if (not pos.get('dry')) and pos.get('rungs', 1) == 1 and not pos.get('_be_gtc') \
                        and SPIKE_BREAKEVEN_AFTER_SEC > 0 and (now - pos['opened']) >= SPIKE_BREAKEVEN_AFTER_SEC:
                    try:
                        _e0 = pos.get('entry_px') or 0
                        if _e0:
                            _be = _e0 * (1 + SPIKE_FEE_PCT / 100.0) if d == 'long' else _e0 * (1 - SPIKE_FEE_PCT / 100.0)
                            if pos.get('tp_oid'):
                                _spike_cancel(coin, market, pos['tp_oid'])
                            _oid, _px = _spike_tp(coin, market, d, pos.get('qty') or 0, _e0, price=_be)
                            with _spike_lock:
                                pos['tp_oid'] = _oid; pos['tp_px'] = _px; pos['_be_gtc'] = True; pos['_be_ts'] = now
                            log.info(f'[SPIKE] {coin}/{market}: 2ч без TP — выход переставлен на БЕЗУБЫТОК @{_px:.8g}')
                            _spike_pos_save()
                    except Exception as _ex:
                        log.warning(f'[SPIKE] {coin}/{market}: перестановка на БУ не удалась: {_ex}')
                if SPIKE_MART_ENABLE:
                    _spike_martingale_check(coin, market, pos, px, now)   # может долить (меняет среднюю/qty/TP) ДО проверки безубытка
                be = _spike_breakeven_check(coin, market, pos, px, now)
                if be:
                    btag, bxp = be
                    if bxp is None: bxp = px
                    if (not pos.get('dry')) and bxp:
                        bxp = _spike_force_close(coin, market, pos) or bxp
                    forced = (btag, bxp)
            else:
                _slpct = _spike_pct(coin, market)[1]
                if (not pos.get('server_tpsl')) and chg is not None and SPIKE_STOP_PCT and chg <= -_slpct:
                    if pos.get('dry'):
                        xp = e * (1 - _slpct / 100) if d == 'long' else e * (1 + _slpct / 100)
                    else:
                        xp = _spike_force_close(coin, market, pos) or px
                    forced = ('СТОП', xp)
                # таймаут-выход: не дошло до TP за SPIKE_MAX_HOLD_SEC
                elif SPIKE_MAX_HOLD_SEC and now - pos['opened'] >= SPIKE_MAX_HOLD_SEC and px:
                    if pos.get('dry'):
                        xp = px
                    else:
                        xp = _spike_force_close(coin, market, pos) or px
                    forced = ('ТАЙМАУТ', xp)
            if forced:
                tag, xp = forced
                gross = (xp - e) / e * 100 if d == 'long' else (e - xp) / e * 100
                net = gross - SPIKE_FEE_PCT; usdt = e * (pos.get('qty') or 0) * _spike_mult(coin, market) * net / 100
                do_ban = (tag in ('СТОП', 'ТАЙМАУТ') or (tag.startswith('ТАЙМАУТ') and tag != 'ТАЙМАУТ' and net < 0)) and SPIKE_STOP_BAN_SEC > 0
                with _spike_lock:
                    _spike_positions.pop((coin, market), None); _spike_pos_save()
                    _spike_cooldown[coin] = now
                    if do_ban:
                        _spike_ban[coin] = now + SPIKE_STOP_BAN_SEC
                        _spike_ban_save()
                ban_note = f', {coin} в бан на {SPIKE_STOP_BAN_SEC//86400}д' if do_ban else ''
                tagc = ('СИМУЛЯЦИЯ ' if pos.get('dry') else '') + tag
                log.warning(f'[SPIKE] {tagc} {coin}/{market} {d} вход@{e} выход@{xp:.8g} '
                            f'({net:+.2f}% / {usdt:+.3f} USDT, держал {now - pos["opened"]:.0f}с) — слот свободен{ban_note}')
                if not pos.get('dry'):
                    _spike_track_close(coin, market, pos, usdt, tag)
                    _spike_hold_ban(coin, market, pos, tag)
                    _spike_verify_pnl(coin, market, pos['opened'], usdt, tag)   # сверка с реальным PnL биржи
                    _spike_record_close(coin, market, pos, net, usdt, tag)   # путь + анализ TP/SL при минусе
                _spike_ws_unwatch(market)
                continue
            _spike_track_tick(coin, market, pos, px, now)
            # статус раз в SPIKE_STATUS_SEC (по умолчанию каждую минуту)
            if SPIKE_STATUS_SEC and chg is not None and now - pos.get('last_status', 0) >= SPIKE_STATUS_SEC:
                pos['last_status'] = now
                log.info(f'[SPIKE] держу {coin}/{market} {d} вход@{e} тек@{px:.8g} '
                         f'({chg:+.2f}%, {now - pos["opened"]:.0f}с) -> TP@{pos.get("tp_px")}')


# ── ВОЗРАСТ МОНЕТЫ ПО ПЕРВОЙ ДНЕВНОЙ СВЕЧЕ своей биржи (launchTime у Bitget пустой -> CAP-кейс).
#    Лениво, на СИГНАЛЕ, кэш в файл навсегда (первая свеча не меняется). Fail-open при ошибке API. ──
SPIKE_AGE_CACHE_FILE = 'spike_coin_age_v2.json'   # v2: кэш v46 отравлен 90д-капом Bitget, пересбор
_spike_age_cache: dict = {}      # 'bg:COIN'/'bb:COIN' -> ts_ms первой свечи | 0 = старше окна выборки
_spike_ath_cache: dict = {}      # 'bg:COIN'/'bb:COIN' -> (ts_расчёта, hist_max)
_spike_age_loaded = False
_spike_young_ready = True        # совместимость: сборка списков больше не нужна


def _spike_build_young():
    """Оставлено для совместимости вызовов: проверка возраста теперь ленивая, по первой свече."""
    global _spike_young_ready
    if _spike_young_ready:
        return
    _spike_young_ready = True


def _spike_age_load():
    global _spike_age_loaded, _spike_age_cache
    if _spike_age_loaded:
        return
    _spike_age_loaded = True
    try:
        with open(SPIKE_AGE_CACHE_FILE) as f:
            _spike_age_cache = {str(k): int(v) for k, v in json.load(f).items()}
        log.info(f'[SPIKE] кэш возраста монет: {len(_spike_age_cache)} записей')
    except Exception:
        _spike_age_cache = {}


def _spike_age_save():
    try:
        with open(SPIKE_AGE_CACHE_FILE, 'w') as f:
            json.dump(_spike_age_cache, f)
    except Exception as e:
        log.debug(f'age-cache save: {e}')


SPIKE_POS_STATE_FILE = 'spike_pos_state.json'   # entry0/ступени/oid позиций — переживают рестарт


def _spike_pos_save():
    """Снимок открытых позиций на диск: рестарт больше не обнуляет мартингейл и не теряет oid."""
    try:
        snap = {}
        for (c, m), p in list(_spike_positions.items()):
            if p.get('dry'):
                continue
            snap[f'{c}|{m}'] = {k: p.get(k) for k in
                ('direction', 'entry_px', 'entry0', 'qty', 'rungs', 'tp_oid', 'tp_px',
                 'opened', '_be_ts', '_be_gtc', '_last_rung_ts', 'server_tpsl')}
        with open(SPIKE_POS_STATE_FILE, 'w') as f:
            json.dump(snap, f)
    except Exception as e:
        log.debug(f'pos-state save: {e}')


def _spike_pos_load():
    try:
        with open(SPIKE_POS_STATE_FILE) as f:
            return json.load(f) or {}
    except Exception:
        return {}


def _spike_free_usdt(market):
    """Свободный USDT на бирже позиции; None при ошибке."""
    try:
        if market == 'bb_fut':
            res = bybit_get('/v5/account/wallet-balance', {'accountType': 'UNIFIED'})
            row = (res[0] if isinstance(res, list) and res else
                   (res.get('list', [{}])[0] if isinstance(res, dict) else {})) or {}
            for c in row.get('coin', []) or []:
                if c.get('coin') == 'USDT':
                    v = c.get('availableToWithdraw') or c.get('walletBalance') or 0
                    return float(v or 0)
            return float(row.get('totalAvailableBalance') or 0)
        data = bitget_get('/api/v2/mix/account/accounts', params={'productType': 'USDT-FUTURES'})
        for a in (data or []):
            if a.get('marginCoin') == 'USDT':
                return float(a.get('crossedMaxAvailable') or a.get('available') or 0)
    except Exception as e:
        log.debug(f'free_usdt {market}: {e}')
    return None


def _spike_daily_closes(coin, market, days=365):
    """[(ts_ms, close)] по возрастанию за N дней (дневные свечи своей биржи). Пусто при ошибке."""
    lim = int(days) + 1
    try:
        if market == 'bb_fut':
            data = bybit_get('/v5/market/kline', {'category': 'linear', 'symbol': f'{coin}USDT',
                                                  'interval': 'D', 'limit': min(lim, 1000)})
            rows = data.get('list', data) if isinstance(data, dict) else data
            out = [(int(r[0]), float(r[4])) for r in (rows or [])]
        elif market == 'bg_fut':
            data = bitget_get('/api/v2/mix/market/candles',
                              params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                      'granularity': '1D', 'limit': str(min(lim, 200))})
            out = [(int(r[0]), float(r[4])) for r in (data or [])]
        else:
            return []
        out = [x for x in out if x[1] > 0]
        out.sort(key=lambda x: x[0])
        return out
    except Exception as e:
        log.debug(f'daily_closes {coin}/{market}: {e}')
        return []


def _spike_pos_chart_png(coin, market, pos, title=None, be_label='безубыток'):
    """PNG: год дневок (лог-шкала) + 30 дней крупно; уровни входа/колен/средней/безубытка. None при ошибке."""
    try:
        import io
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        import matplotlib.dates as mdates
        from datetime import datetime as _dt
    except Exception as e:
        log.debug(f'chart: matplotlib недоступен: {e}')
        return None
    try:
        rows = _spike_daily_closes(coin, market, 365)
        if len(rows) < 5:
            return None
        ds = [_dt.fromtimestamp(t / 1000) for t, _ in rows]; cs = [c for _, c in rows]
        d = pos.get('direction', '')
        entry0 = float(pos.get('entry0') or pos.get('entry_px') or 0)
        avg = float(pos.get('entry_px') or 0); be = float(pos.get('tp_px') or 0)
        rungs = int(pos.get('rungs', 1) or 1); qty = float(pos.get('qty') or 0)
        cur = cs[-1]; pnl_pct = ((cur - avg) / avg * 100 * (1 if d == 'long' else -1)) if avg else 0.0
        ath = max(cs); dd = (cur / ath - 1) * 100 if ath else 0.0
        opened = pos.get('opened')
        fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 5.6), dpi=110,
                                      gridspec_kw={'width_ratios': [2.1, 1], 'wspace': 0.28})
        def _levels(a):
            if entry0: a.axhline(entry0, color='#eb6834', lw=1.2, ls='--')
            if avg:    a.axhline(avg, color='#8b8b84', lw=1.2)
            if be:     a.axhline(be, color='#1baf7a', lw=1.5)
            if opened: a.axvline(_dt.fromtimestamp(float(opened)), color='#eb6834', lw=0.9, alpha=.6)
            a.grid(alpha=.25); a.set_facecolor('#fbfaf7'); a.tick_params(labelsize=8)
        ax.plot(ds, cs, color='#2a78d6', lw=1.4); _levels(ax)
        if min(cs) > 0 and max(cs) / min(cs) > 4:
            ax.set_yscale('log')
        ax.set_title(f'{coin}/USDT · {len(rows)}д дневок · от максимума окна {dd:+.0f}%', fontsize=9.5, loc='left')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        _h = [plt.Line2D([], [], color='#2a78d6', lw=1.4, label='цена (1D close)')]
        if entry0: _h.append(plt.Line2D([], [], color='#eb6834', ls='--', label=f'вход 1 @ {entry0:g}'))
        if avg:    _h.append(plt.Line2D([], [], color='#8b8b84', label=f'средняя {avg:g}'))
        if be:     _h.append(plt.Line2D([], [], color='#1baf7a', lw=1.5, label=f'{be_label} @ {be:g}'))
        ax.legend(handles=_h, fontsize=7.5, loc='best', framealpha=.92)
        n30 = min(30, len(rows))
        ax2.plot(ds[-n30:], cs[-n30:], color='#2a78d6', lw=1.4); _levels(ax2)
        lo = min(cs[-n30:] + [x for x in (entry0, avg, be) if x]); hi = max(cs[-n30:] + [x for x in (entry0, avg, be) if x])
        pad = (hi - lo) * 0.08 or hi * 0.02
        ax2.set_ylim(lo - pad, hi + pad)
        ax2.set_title('последние 30 дней', fontsize=9.5, loc='left')
        ax2.xaxis.set_major_locator(mdates.DayLocator(interval=7)); ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
        for y, txt, c in [(entry0, 'вход', '#eb6834'), (be, be_label[:2].upper(), '#1baf7a')]:
            if y: ax2.annotate(txt, xy=(ds[-1], y), xytext=(5, 0), textcoords='offset points', fontsize=8, color=c, va='center')
        fig.suptitle(title or f'[spike] МАРТ {coin}/{market} колено #{rungs - 1} · {d} · qty {qty:g} · тек. {pnl_pct:+.1f}% от средней',
                     fontsize=10.5, x=0.01, ha='left', y=0.99)
        fig.subplots_adjust(top=0.88, bottom=0.12)
        buf = io.BytesIO(); fig.savefig(buf, format='png', bbox_inches='tight'); plt.close(fig)
        return buf.getvalue()
    except Exception as e:
        log.debug(f'chart {coin}/{market}: {e}')
        return None



# ── TRACK: ретроспективное сравнение трёх исходов по каждой РЕАЛЬНОЙ позиции:
#    факт / без мартингейла / март+TP0 (доливы, но выход по первоначальной цене TP). Никаких ордеров. Касания веток ловятся ИСТОРИЧЕСКИ
#    по 1m-свечам high/low (без пропусков между опросами): подметание раз в ~10 мин у живых позиций,
#    раз в 5 мин в дозоре после факт-закрытия — до срабатывания СВОЕГО условия, но не дольше
#    3 суток от входа; дальше — MtM с меткой «не сработал(3д)». ──
SPIKE_TRACK_FILE       = 'spike_track.jsonl'
SPIKE_TRACK_PEND_FILE  = 'spike_track_pending.json'
SPIKE_TRACK_REPORT_SEC = 6 * 3600
SPIKE_TRACK_MAX_SEC    = 3 * 86400
SPIKE_TRACK_SWEEP_SEC  = 600            # период исторического подметания у живой позиции
SPIKE_TRACK_TAIL_SEC   = 300            # период дозора после факт-закрытия
_spike_track_closed: list = []          # финализированные с прошлого отчёта
_spike_track_pending: dict = {}         # 'COIN|market|opened' -> st
_spike_track_lock = threading.Lock()

def _spike_track_init(coin, market, pos):
    try:
        pos['track'] = {'coin': coin, 'market': market, 'side': pos['direction'],
                        'opened': pos['opened'], 'hist_ts': pos['opened'],
                        'e0': float(pos.get('entry_px') or 0), 'q0': float(pos.get('qty') or 0),
                        'tp_pct': _spike_pct(coin, market)[0], 'tp0': float(pos.get('tp_px') or 0),
                        'rungs': 1, 'last_rung_ts': None,
                        'v2': None, 'v3': None, 'v3_tp': float(pos.get('tp_px') or 0),
                        'v4': None, 'v4_tp': float(pos.get('tp_px') or 0), 'last_px': None}
    except Exception as ex:
        log.debug(f'track-init {coin}: {ex}')

def _spike_track_rung(coin, market, pos, new_entry):
    t = pos.get('track')
    if not t: return
    t['rungs'] = pos.get('rungs', 1); t['last_rung_ts'] = time.time()
    # v3_tp НЕ трогаем: ветка март+TP0 держит первоначальный TP от первого входа.
    # v4_tp — ветка март+TPср: цель переезжает на TP% от новой средней.
    t['v4_tp'] = new_entry * (1 + t['tp_pct']/100.0) if pos['direction'] == 'long' else new_entry * (1 - t['tp_pct']/100.0)

def _spike_track_klines(coin, market, t_from, t_to):
    """1m-свечи [t_from, t_to] (сек) -> [(ts_ms, high, low)] по возрастанию, пагинация по 200."""
    out = []
    try:
        start = int(t_from * 1000); end = int(t_to * 1000)
        while start < end:
            if market == 'bg_fut':
                data = bitget_get('/api/v2/mix/market/candles',
                                  params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                          'granularity': '1m', 'startTime': str(start),
                                          'endTime': str(min(end, start + 200*60*1000)), 'limit': '200'})
            elif market == 'bb_fut':
                data = bybit_get('/v5/market/kline',
                                 {'category': 'linear', 'symbol': f'{coin}USDT', 'interval': '1',
                                  'start': str(start), 'end': str(min(end, start + 200*60*1000)), 'limit': '200'})
            else:
                break
            rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
            batch = []
            for r in rows:
                try:
                    batch.append((int(r[0]), float(r[2]), float(r[3])))     # оба API: [2]=high, [3]=low
                except Exception:
                    pass
            if not batch:
                break
            out.extend(batch)
            nxt = max(ts for ts, _, _ in batch) + 60*1000
            if nxt <= start: break
            start = nxt
            time.sleep(0.15)
    except Exception as e:
        log.debug(f'track-klines {coin}/{market}: {e}')
    out.sort(key=lambda x: x[0])
    return out

def _spike_track_sweep(st, t_to=None):
    """Прогон 1m high/low от st['hist_ts'] до t_to — ловит касания без пропусков.
    без_март: до 2ч от входа цель TP0, после — БУ0 (e0 +/- комиссия круга, как у бота).
    март+TP0: доливы как в факте, цель остаётся ПЕРВОНАЧАЛЬНОЙ (tp0 от первого входа).
    март+TPср: доливы как в факте, цель = TP% от НОВОЙ СРЕДНЕЙ, ловится с последнего колена.
    Обе — на итоговом qty от итоговой средней."""
    t_to = t_to or time.time()
    t_from = st.get('hist_ts') or st['opened']
    if t_to - t_from < 30: return
    kl = _spike_track_klines(st['coin'], st['market'], t_from, t_to)
    if not kl:
        st['hist_ts'] = t_to; return
    up = (st['side'] == 'long')
    be0 = st['e0'] * (1 + SPIKE_FEE_PCT/100.0) if up else st['e0'] * (1 - SPIKE_FEE_PCT/100.0)
    sw2h = (st['opened'] + SPIKE_BREAKEVEN_AFTER_SEC) * 1000
    rung_ms = (st.get('last_rung_ts') or st['opened']) * 1000
    for ts, hi, lo in kl:
        if st['v2'] is None:
            if ts < sw2h:
                if (hi >= st['tp0']) if up else (lo <= st['tp0']):
                    st['v2'] = {'lbl': 'TP', 'px': st['tp0'], 'ts': ts/1000}
            else:
                if (hi >= be0) if up else (lo <= be0):
                    st['v2'] = {'lbl': 'БУ', 'px': be0, 'ts': ts/1000}
        if st.get('rungs', 1) > 1:
            if st['v3'] is None and ((hi >= st['v3_tp']) if up else (lo <= st['v3_tp'])):
                st['v3'] = {'lbl': 'TP', 'px': st['v3_tp'], 'ts': ts/1000}
            if st['v4'] is None and ts >= rung_ms and ((hi >= st['v4_tp']) if up else (lo <= st['v4_tp'])):
                st['v4'] = {'lbl': 'TP', 'px': st['v4_tp'], 'ts': ts/1000}
        if st['v2'] is not None and (st.get('rungs', 1) <= 1 or (st['v3'] is not None and st['v4'] is not None)):
            break
    st['last_px'] = kl[-1][1] if up else kl[-1][2]
    st['hist_ts'] = t_to

def _spike_track_tick(coin, market, pos, px, now):
    t = pos.get('track')
    if not t: return
    t['last_px'] = px; t['rungs'] = pos.get('rungs', 1)
    if now - (t.get('hist_ts') or t['opened']) >= SPIKE_TRACK_SWEEP_SEC:
        try:
            _spike_track_sweep(t, now)
        except Exception as ex:
            log.debug(f'track-tick sweep {coin}: {ex}')

def _spike_track_pnl(e, q, xp, up):
    if not (e and q and xp): return None
    g = (xp - e)/e*100 if up else (e - xp)/e*100
    return e*q*(g - SPIKE_FEE_PCT)/100

def _spike_track_close(coin, market, pos, fact_usdt, label):
    """Факт закрылся: добираем историю до этой секунды; нерешённые ветки -> дозор до своего триггера/3 суток."""
    t = pos.get('track')
    if not t: return
    try:
        st = dict(t)
        st.update({'avg': float(pos.get('entry_px') or 0), 'qty': float(pos.get('qty') or 0),
                   'rungs': pos.get('rungs', 1), 'fact': fact_usdt, 'label': label,
                   'fact_closed': time.time()})
        _spike_track_sweep(st)
        if st['rungs'] <= 1:
            st['v3'] = {'lbl': '=факт', 'px': None, 'ts': st['fact_closed']}
            st['v4'] = {'lbl': '=факт', 'px': None, 'ts': st['fact_closed']}
        if st['v2'] is not None and st['v3'] is not None and st['v4'] is not None:
            _spike_track_final(st); return
        key = f"{coin}|{market}|{pos['opened']:.0f}"
        with _spike_track_lock:
            _spike_track_pending[key] = st
        _spike_track_pend_save()
        log.info(f'[TRACK] {coin}/{market}: факт закрыт ({label}), дозор веток '
                 f'{"без_март " if st["v2"] is None else ""}{"март+TP0 " if st["v3"] is None else ""}'
                 f'{"март+TPср " if st["v4"] is None else ""}'
                 f'до {SPIKE_TRACK_MAX_SEC//86400} суток от входа (1m-свечи)')
    except Exception as ex:
        log.debug(f'track-close {coin}: {ex}')

def _spike_track_final(st):
    up = (st['side'] == 'long'); lp = st.get('last_px') or 0
    if st.get('v2'):
        v2, l2 = _spike_track_pnl(st['e0'], st['q0'], st['v2']['px'], up), st['v2']['lbl']
    else:
        v2, l2 = _spike_track_pnl(st['e0'], st['q0'], lp, up), 'не сработал(3д)'
    if st.get('v3') and st['v3']['lbl'] == '=факт':
        v3, l3 = st['fact'], '=факт'
    elif st.get('v3'):
        v3, l3 = _spike_track_pnl(st['avg'], st['qty'], st['v3']['px'], up), st['v3']['lbl']
    else:
        v3, l3 = _spike_track_pnl(st['avg'], st['qty'], lp, up), 'не сработал(3д)'
    if st.get('v4') and st['v4']['lbl'] == '=факт':
        v4, l4 = st['fact'], '=факт'
    elif st.get('v4'):
        v4, l4 = _spike_track_pnl(st['avg'], st['qty'], st['v4']['px'], up), st['v4']['lbl']
    else:
        v4, l4 = _spike_track_pnl(st['avg'], st['qty'], lp, up), 'не сработал(3д)'
    rec = {'ts': time.time(), 'coin': st['coin'], 'market': st['market'], 'side': st['side'],
           'label': st['label'], 'fact': st['fact'], 'v2': v2, 'v2_lbl': l2, 'v3': v3, 'v3_lbl': l3, 'v4': v4, 'v4_lbl': l4,
           'rungs': st['rungs'] - 1, 'held_fact': st['fact_closed'] - st['opened'],
           'v2_ts': (st.get('v2') or {}).get('ts'), 'v3_ts': (st.get('v3') or {}).get('ts')}
    with _spike_track_lock:
        _spike_track_closed.append(rec)
    try:
        with open(SPIKE_TRACK_FILE, 'a', encoding='utf-8') as f:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    except Exception as ex:
        log.debug(f'track-final write: {ex}')
    def _fm(v): return '—' if v is None else f'{v:+.3f}'
    log.info(f"[TRACK] итог {st['coin']}/{st['market']}: факт {_fm(st['fact'])} ({st['label']}) | "
             f"без_март {_fm(v2)} ({l2}) | март+TP0 {_fm(v3)} ({l3}) | март+TPср {_fm(v4)} ({l4})")

def _spike_track_pend_save():
    try:
        with _spike_track_lock:
            snap = dict(_spike_track_pending)
        with open(SPIKE_TRACK_PEND_FILE, 'w', encoding='utf-8') as f:
            json.dump(snap, f, ensure_ascii=False)
    except Exception as ex:
        log.debug(f'track-pend save: {ex}')

try:                                   # дозор переживает рестарт
    with open(SPIKE_TRACK_PEND_FILE, encoding='utf-8') as _f:
        _spike_track_pending.update(json.load(_f))
except Exception:
    pass

def _spike_track_tail():
    """Дозор: после факт-закрытия досматривает нерешённые ветки по 1m-свечам до их триггера / 3 суток от входа."""
    while True:
        time.sleep(SPIKE_TRACK_TAIL_SEC)
        try:
            with _spike_track_lock:
                items = list(_spike_track_pending.items())
            done = []
            now = time.time()
            for key, st in items:
                deadline = st['opened'] + SPIKE_TRACK_MAX_SEC
                try:
                    _spike_track_sweep(st, min(now, deadline))
                except Exception as ex:
                    log.debug(f'track-tail {key}: {ex}')
                if (st['v2'] is not None and st['v3'] is not None and st['v4'] is not None) or now >= deadline:
                    done.append(key)
                    _spike_track_final(st)
            if done:
                with _spike_track_lock:
                    for k in done: _spike_track_pending.pop(k, None)
                _spike_track_pend_save()
        except Exception as ex:
            log.debug(f'track-tail: {ex}')

def _spike_track_report():
    def _f(v): return '—' if v is None else f'{v:+.3f}'
    while True:
        time.sleep(SPIKE_TRACK_REPORT_SEC)
        try:
            with _spike_track_lock:
                closed, _spike_track_closed[:] = list(_spike_track_closed), []
                pend = list(_spike_track_pending.values())
            lines = [f'Сравнение за {SPIKE_TRACK_REPORT_SEC//3600}ч (финализированные): факт / без_март / март+TP0 / март+TPср',
                     'Касания веток — по 1m high/low, без пропусков; альтернативы без фандинга.', '']
            s = {'f': 0.0, 'v2': 0.0, 'v3': 0.0, 'v4': 0.0}
            for r in closed:
                lines.append(f"{r['coin']}/{r['market']} {r['side']} {r['label']} колен={r['rungs']}: "
                             f"факт {_f(r['fact'])} | без_март {_f(r['v2'])} ({r['v2_lbl']}) | "
                             f"март+TP0 {_f(r['v3'])} ({r['v3_lbl']}) | март+TPср {_f(r['v4'])} ({r['v4_lbl']})")
                s['f'] += r['fact'] or 0; s['v2'] += r['v2'] or 0; s['v3'] += r['v3'] or 0; s['v4'] += r.get('v4') or 0
            lines += ['', f"ИТОГ периода: факт {s['f']:+.3f} | без_март {s['v2']:+.3f} | март+TP0 {s['v3']:+.3f} | март+TPср {s['v4']:+.3f} USDT"]
            if pend:
                lines += ['', 'Дозор (факт закрыт, ветки ждут свой триггер, до 3 суток):']
                for st in pend:
                    lp = st.get('last_px') or 0; up = (st['side'] == 'long')
                    lines.append(f"{st['coin']}/{st['market']} {st['side']}: факт {_f(st['fact'])} ({st['label']}) | "
                                 f"без_март {'✔' if st['v2'] else _f(_spike_track_pnl(st['e0'], st['q0'], lp, up)) + ' MtM'} | "
                                 f"март+TP0 {'✔' if st['v3'] else _f(_spike_track_pnl(st['avg'], st['qty'], lp, up)) + ' MtM'} | "
                                 f"март+TPср {'✔' if st['v4'] else _f(_spike_track_pnl(st['avg'], st['qty'], lp, up)) + ' MtM'} | "
                                 f"осталось {max(0.0, (st['opened'] + SPIKE_TRACK_MAX_SEC - time.time()))/3600:.1f}ч")
            with _spike_lock:
                items = list(_spike_positions.items())
            if items:
                lines += ['', 'Открытые позиции (три MtM):']
                for (cn, mk), pos in items:
                    t = pos.get('track')
                    if not t: continue
                    up = (pos['direction'] == 'long'); lp = t.get('last_px') or 0
                    fac = _spike_track_pnl(pos.get('entry_px'), pos.get('qty'), lp, up)
                    v2 = _spike_track_pnl(t['e0'], t['q0'], t['v2']['px'], up) if t['v2'] else _spike_track_pnl(t['e0'], t['q0'], lp, up)
                    l2 = t['v2']['lbl'] if t['v2'] else 'MtM'
                    if pos.get('rungs', 1) <= 1:
                        v3, l3 = fac, '=факт'; v4, l4 = fac, '=факт'
                    else:
                        v3, l3 = ((_spike_track_pnl(pos.get('entry_px'), pos.get('qty'), t['v3']['px'], up), t['v3']['lbl'])
                                  if t['v3'] else (fac, 'MtM'))
                        v4, l4 = ((_spike_track_pnl(pos.get('entry_px'), pos.get('qty'), t['v4']['px'], up), t['v4']['lbl'])
                                  if t.get('v4') else (fac, 'MtM'))
                    lines.append(f'{cn}/{mk} {pos["direction"]} колен={pos.get("rungs",1)-1}: '
                                 f'факт {_f(fac)} | без_март {_f(v2)} ({l2}) | март+TP0 {_f(v3)} ({l3}) | '
                                 f'март+TPср {_f(v4)} ({l4})')
            tot = {'f': 0.0, 'v2': 0.0, 'v3': 0.0, 'v4': 0.0, 'n': 0}
            try:
                for ln in open(SPIKE_TRACK_FILE, encoding='utf-8'):
                    r = json.loads(ln)
                    tot['f'] += r.get('fact') or 0; tot['v2'] += r.get('v2') or 0
                    tot['v3'] += r.get('v3') or 0; tot['v4'] += r.get('v4') or 0; tot['n'] += 1
                lines += ['', f"НАКОПИТЕЛЬНО ({tot['n']}): факт {tot['f']:+.3f} | "
                              f"без_март {tot['v2']:+.3f} | март+TP0 {tot['v3']:+.3f} | март+TPср {tot['v4']:+.3f} USDT"]
            except FileNotFoundError:
                pass
            if closed or pend or items:
                spike_send_email('[spike] TRACK 6ч: факт vs без_март vs март+TP0 vs март+TPср', '\n'.join(lines))
        except Exception as ex:
            log.debug(f'track-report: {ex}')

threading.Thread(target=_spike_track_tail, daemon=True, name='spike-track-tail').start()
threading.Thread(target=_spike_track_report, daemon=True, name='spike-track-report').start()


def _spike_entry_mail(coin, market, pos):
    """Письмо + график на ПЕРВЫЙ вход в позицию (доливы шлёт _spike_mart_mail). В фоне."""
    try:
        p = dict(pos)
    except Exception:
        return

    def _job():
        try:
            d = p.get('direction'); avg = float(p.get('entry_px') or 0)
            qty = float(p.get('qty') or 0); notional = qty * avg
            ex = 'Bybit' if market == 'bb_fut' else 'Bitget'
            tp = float(p.get('tp_px') or 0); sl = p.get('sl_px')
            lines = [f'ВХОД {coin}/{market} {d} @{avg:.8g}',
                     f'qty {qty:.8g} (~{notional:.1f} USDT, {ex})',
                     (f'TP @{tp:.8g}' if tp else 'TP не выставлен') +
                     (f' / SL @{float(sl):.8g}' if sl else ' / без SL (выход в безубыток)'),
                     f'режим выхода: {SPIKE_EXIT_MODE}, серверный TP/SL: {"да" if p.get("server_tpsl") else "нет"}']
            _png = _spike_pos_chart_png(
                coin, market, p, be_label='TP',
                title=f'[spike] ВХОД {coin}/{market} · {d} · qty {qty:g} @ {avg:g}')
            spike_send_email(f'[spike] ВХОД {coin}/{market} {d}', '\n'.join(lines),
                             attachments=[(f'{coin}_{market}_entry.png', _png)] if _png else None)
        except Exception as e:
            log.debug(f'entry-mail: {e}')

    threading.Thread(target=_job, daemon=True, name='spike-entry-mail').start()


def _spike_mart_mail(coin, market, pos):
    """Письмо на каждое колено; на последнем — расчёт 200% маржи и призыв довнести (в фоне)."""
    try:
        p = dict(pos)
    except Exception:
        return

    def _job():
        try:
            n = int(p.get('rungs', 1)) - 1
            avg = float(p.get('entry_px') or 0); qty = float(p.get('qty') or 0)
            notional = qty * avg
            ex = 'Bybit' if market == 'bb_fut' else 'Bitget'
            lines = [f'Колено #{n} {coin}/{market} {p.get("direction")}',
                     f'средняя {avg:.8g}, qty {qty:.8g} (~{notional:.1f} USDT, {ex})',
                     f'выход в безубыток @{float(p.get("tp_px") or 0):.8g}']
            subj = f'[spike] МАРТ {coin}/{market} колено #{n}'
            if int(p.get('rungs', 1)) >= SPIKE_MART_RUNGS:                 # последняя ступень
                free = _spike_free_usdt(market)
                need = notional                    # цель 200% маржи: свободно >= номинал позиции
                if free is not None:
                    lines.append(f'свободно на {ex} ~{free:.1f} USDT; цель 200% маржи: свободно >= {need:.1f}')
                    if free < need:
                        subj = f'[spike] МАРЖА! {coin}/{market}: довнеси ~{need - free:.0f} USDT на {ex}'
                        lines.append(f'>>> ДОВНЕСИ ~{need - free:.0f} USDT на {ex} <<<')
                else:
                    lines.append('свободный баланс получить не удалось')
            _png = _spike_pos_chart_png(coin, market, p)
            spike_send_email(subj, '\n'.join(lines),
                             attachments=[(f'{coin}_{market}_mart{n}.png', _png)] if _png else None)
        except Exception as e:
            log.debug(f'mart-mail: {e}')

    threading.Thread(target=_job, daemon=True, name='spike-mart-mail').start()


def _spike_first_candle_ms(coin, market):
    """ts первой ДНЕВНОЙ свечи; 0 = монета заведомо старше порога.
    Bitget: candles капится ~90 дневными свечами (из-за чего v46 считала ВСЕ старые монеты
    90-дневными и резала их) -> сперва history-candles на (порог+20)д назад: есть свеча = старая."""
    LIM = 200                                        # 200 дневных свечей > 180д — хватает для вердикта
    if market == 'bb_fut':
        data = bybit_get('/v5/market/kline', {'category': 'linear', 'symbol': f'{coin}USDT',
                                              'interval': 'D', 'limit': LIM})
        rows = data.get('list', data) if isinstance(data, dict) else data
        if not rows:
            return None
        return 0 if len(rows) >= LIM else int(rows[-1][0])       # bybit: новые -> старые
    cutoff_ms = int((time.time() - (SPIKE_MIN_AGE_DAYS + 1) * 86400) * 1000)   # ровно порог+1д: монеты 180-200д не режутся зря
    try:
        old = bitget_get('/api/v2/mix/market/history-candles',
                         params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                 'granularity': '1D', 'endTime': str(cutoff_ms), 'limit': '1'})
        if old:
            return 0                                             # свеча старше порога есть -> старая
    except Exception:
        pass                                                     # history недоступен -> нижняя оценка ниже
    data = bitget_get('/api/v2/mix/market/candles',
                      params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                              'granularity': '1D', 'limit': str(LIM)})
    if not data:
        return None
    return 0 if len(data) >= LIM else int(data[0][0])            # моложе порога: ts первой (ниж. оценка)


def _spike_age_ok(coin, market):
    """-> (ok, age_d). ok = возраст >= SPIKE_AGE_BLOCK_D (или фильтр выкл).
    Ошибка API -> True (fail-open), без кэша. Сторону решает вызывающий по age_d."""
    if not SPIKE_MIN_AGE_DAYS:
        return (True, None)
    _spike_age_load()
    key = ('bg:' if market == 'bg_fut' else 'bb:') + coin
    ts = _spike_age_cache.get(key)
    if ts is None:
        try:
            ts = _spike_first_candle_ms(coin, market)
        except Exception as e:
            log.debug(f'age {coin}/{market}: {e}')
            return (True, None)
        if ts is None:
            return (True, None)
        _spike_age_cache[key] = int(ts)
        _spike_age_save()
    if ts == 0:
        return (True, None)
    age_d = (time.time() - ts / 1000) / 86400
    return (age_d >= SPIKE_AGE_BLOCK_D, age_d)


_spike_age_thread = None
_spike_age_warm_last = 0.0


def _spike_age_warmup():
    """Проверка возраста ВСЕГО юниверсума по первой дневной свече, в фоне (торговлю не блокирует).
    1 запрос на новую монету, результат в кэш навсегда; повторный старт = 0 запросов."""
    global _spike_age_thread
    if not SPIKE_MIN_AGE_DAYS:
        return
    if _spike_age_thread and _spike_age_thread.is_alive():
        return

    def _job():
        _spike_age_load()
        try:
            uni = _spike_get_universe()
        except Exception as e:
            log.warning(f'[SPIKE] возраст: юниверс недоступен: {e}')
            return
        key = lambda c, m: ('bg:' if m == 'bg_fut' else 'bb:') + c
        todo = [(c, m) for m, coins in uni.items() for c in coins if key(c, m) not in _spike_age_cache]
        fetched = 0; errs = 0
        for i, (c, m) in enumerate(todo, 1):
            try:
                ts = _spike_first_candle_ms(c, m)
                if ts is not None:
                    _spike_age_cache[key(c, m)] = int(ts); fetched += 1
            except Exception:
                errs += 1
            if i % 100 == 0:
                _spike_age_save()
                log.info(f'[SPIKE] возраст: проверено {i}/{len(todo)} новых монет…')
            time.sleep(0.06)                      # ~16 зап/с суммарно на обе биржи — в лимитах
        _spike_age_save()
        yb = yg = 0; nowt = time.time()
        for m, coins in uni.items():
            for c in coins:
                ts = _spike_age_cache.get(key(c, m))
                if ts and (nowt - ts / 1000) / 86400 < SPIKE_MIN_AGE_DAYS:
                    yg += (m == 'bg_fut'); yb += (m == 'bb_fut')
        log.info(f'[SPIKE] фильтр возраста <{SPIKE_MIN_AGE_DAYS}д: юниверс проверен '
                 f'(новых запросов {fetched}, ошибок {errs}), молодых bg={yg} bb={yb}')

    _spike_age_thread = threading.Thread(target=_job, daemon=True, name='spike-age')
    _spike_age_thread.start()


def _spike_age_maybe_warmup():
    """Раз в SPIKE_UNIVERSE_REFRESH_SEC догревает кэш возраста для НОВЫХ листингов."""
    global _spike_age_warm_last
    now = time.time()
    if now - _spike_age_warm_last < SPIKE_UNIVERSE_REFRESH_SEC:
        return
    _spike_age_warm_last = now
    _spike_age_warmup()


# ── ЛОГ-МЕТРИКИ ДЕТЕКТА: раз в SPIKE_REPORT_SEC печатать объём× и Δ% по просканированным монетам ──
SPIKE_REPORT_SEC  = 0      # период сброса метрик в лог, сек (0 = выкл) — выключено
SPIKE_REPORT_TOPN = 0      # 0 = все монеты в области скана; N = только топ-N по объём× (для полного юниверса)
_spike_metrics:     dict = {}      # (coin, market) -> (ts, ratio, chg) — заполняется в скане, без лишних запросов
_spike_report_last: list = [0.0]

def _spike_metrics_report():
    """Раз в SPIKE_REPORT_SEC печатает в лог объём× и Δ% по каждой просканированной монете.
       Данные берутся из последнего прохода скана (новых запросов к биржам не делает)."""
    if not SPIKE_REPORT_SEC:
        return
    now = time.time()
    if now - _spike_report_last[0] < SPIKE_REPORT_SEC:
        return
    _spike_report_last[0] = now
    items = list(_spike_metrics.items())
    if not items:
        log.info('[METRICS] данных пока нет (скан ещё не обошёл монеты или свечи пусты)')
        return
    items.sort(key=lambda kv: kv[1][1], reverse=True)   # по объём× убыв.
    shown = items[:SPIKE_REPORT_TOPN] if SPIKE_REPORT_TOPN else items
    log.info(f'[METRICS] объём×/Δ% за {SPIKE_PRICE_WIN_MIN}м по {len(shown)} монетам '
             f'(порог ×{SPIKE_VOL_MULT} и |Δ|>={SPIKE_PRICE_CHG_PCT}%):')
    for (cn, mk), (ts, ratio, chg) in shown:
        flag  = ' <-- СИГНАЛ' if (ratio > SPIKE_VOL_MULT and abs(chg) >= SPIKE_PRICE_CHG_PCT) else ''
        stale = f' (стар {now-ts:.0f}с)' if now - ts > SPIKE_REPORT_SEC * 3 else ''
        log.info(f'   {cn}/{mk} ×{ratio:5.1f}  Δ{chg:+6.2f}%{flag}{stale}')


_spike_pctl_cache: dict = {}     # (coin, market) -> (ts, отсортированные дневные закрытия)

def _spike_pctl_closes(coin, market):
    """Отсортированные дневные ЗАКРЫТИЯ за SPIKE_LOW_PCTL_DAYS дней (кэш на сутки, один запрос
    на оба перцентиля). None = нет данных / мало истории (fail-open)."""
    key = (coin, market); now = time.time()
    c = _spike_pctl_cache.get(key)
    if c and now - c[0] < 86400:
        return c[1]
    lim = int(SPIKE_LOW_PCTL_DAYS) + 1
    closes = []
    try:
        if market == 'bb_fut':
            data = bybit_get('/v5/market/kline', {'category': 'linear', 'symbol': f'{coin}USDT',
                                                  'interval': 'D', 'limit': lim})
            rows = data.get('list', data) if isinstance(data, dict) else data
            closes = [float(r[4]) for r in (rows or [])[1:]]              # [0] = текущий день — пропускаем
        elif market == 'bg_fut':
            data = bitget_get('/api/v2/mix/market/candles',
                              params={'symbol': f'{coin}USDT', 'productType': 'USDT-FUTURES',
                                      'granularity': '1D', 'limit': str(lim)})
            closes = [float(r[4]) for r in (data or [])[:-1]]
        else:
            return None
    except Exception as e:
        log.debug(f'pctl {coin}/{market}: {e}')
        return None
    closes = [x for x in closes if x > 0]
    closes.sort()                              # короткая история тоже кэшируется: по её длине судим о возрасте
    _spike_pctl_cache[key] = (now, closes)
    return closes


def _spike_pctl_price(coin, market, pct):
    """Цена pct-го перцентиля дневных закрытий. None = нет данных (fail-open)."""
    closes = _spike_pctl_closes(coin, market)
    if not closes or len(closes) < SPIKE_PCTL_MIN_CLOSES:     # мало истории — не судим
        return None
    idx = max(0, min(len(closes) - 1, int(len(closes) * float(pct) / 100.0)))
    return closes[idx]


def _spike_low_pctl_price(coin, market):
    """Нижний порог: ниже него монета у ДНА собственного диапазона (SOPH), в отличие от боковика (HANA)."""
    return _spike_pctl_price(coin, market, SPIKE_LOW_PCTL) if SPIKE_LOW_PCTL else None


def _spike_high_pctl_price(coin, market):
    """Верхний порог: выше него монета у ВЕРХА диапазона — откат тут чаще продолжение сдувания пампа."""
    return _spike_pctl_price(coin, market, SPIKE_HIGH_PCTL) if SPIKE_HIGH_PCTL else None


_spike_vol7_thread = None
_spike_vol7_warm_last = 0.0


def _spike_liquidity_ok(coin, market):
    """Защита от делистинга: свежая проверка 24h-оборота и открытого интереса ПЕРЕД входом.
    Применяется ко ВСЕМ монетам (вайтлист тоже — MBOX был в вайтлисте и делистнут).
    -> (ok, vol24h_usd, oi_usd, reason). При ошибке данных — fail-open (не блокируем)."""
    if not SPIKE_DELIST_FILTER:
        return (True, 0.0, 0.0, '')
    sym = f'{coin}USDT'
    vol24h = oi_usd = 0.0
    try:
        if market == 'bg_fut':
            data = bitget_get('/api/v2/mix/market/tickers', params={'productType': 'USDT-FUTURES', 'symbol': sym})
            rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
            d = {}
            for r in rows:
                if str(r.get('symbol', '')).upper() == sym.upper() or len(rows) == 1:
                    d = r; break
            vol24h = float(d.get('usdtVolume') or d.get('quoteVolume') or 0)
            lp = float(d.get('lastPr') or 0)
            oi_usd = float(d.get('holdingAmount') or 0) * lp
        elif market == 'bb_fut':
            data = bybit_get('/v5/market/tickers', {'category': 'linear', 'symbol': sym})
            rows = data if isinstance(data, list) else (data.get('list', []) if isinstance(data, dict) else [])
            d = rows[0] if rows else {}
            vol24h = float(d.get('turnover24h') or 0)
            oi_usd = float(d.get('openInterestValue') or 0)
            if oi_usd <= 0:
                oi_usd = float(d.get('openInterest') or 0) * float(d.get('lastPrice') or 0)
        elif market == 'bg_spot':
            return (True, 0.0, 0.0, 'спот: OI н/д')          # для спота OI неприменим
    except Exception as e:
        log.info(f'[SPIKE] {coin}/{market}: ликвидность не проверена ({type(e).__name__}) — фильтр пропущен')
        return (True, 0.0, 0.0, 'нет данных')                # fail-open
    if vol24h < SPIKE_MIN_VOL_24H_USD:
        return (False, vol24h, oi_usd, f'оборот24ч ${vol24h:,.0f} < ${SPIKE_MIN_VOL_24H_USD:,.0f}')
    if oi_usd < SPIKE_MIN_OI_USD:
        return (False, vol24h, oi_usd, f'OI ${oi_usd:,.0f} < ${SPIKE_MIN_OI_USD:,.0f}')
    return (True, vol24h, oi_usd, '')


def _spike_above_floor(coin, market):
    """Фильтр 'упавшего ножа': пропуск входа, если текущая цена < SPIKE_MIN_PCT_OF_ATH% от
    исторического максимума (монета упала >70% от пика — умирающий проект / риск делистинга).
    Максимум берём по дневным свечам (1000 шт ~2.7 года = вся история перпа). Fail-open при ошибке.
    -> (ok, cur_px, hist_max, pct)."""
    if not SPIKE_ATH_FILTER:
        return (True, 0.0, 0.0, 100.0)
    _ck = ('bg:' if market != 'bb_fut' else 'bb:') + coin
    _hit = _spike_ath_cache.get(_ck)
    if _hit and time.time() - _hit[0] < SPIKE_ATH_CACHE_SEC:
        try:
            _b, _a = _spike_book(coin, market)
            _px = (_b + _a) / 2 if (_b and _a) else (_b or _a or 0)
            if _px > 0:
                _p = _px / _hit[1] * 100.0
                return (_p >= SPIKE_MIN_PCT_OF_ATH, _px, _hit[1], _p)
        except Exception:
            pass
    sym = f'{coin}USDT'
    ckey = ('bg:' if market != 'bb_fut' else 'bb:') + coin
    try:
        if market == 'bb_fut':
            data = bybit_get('/v5/market/kline',
                             {'category': 'linear', 'symbol': sym,
                              'interval': SPIKE_ATH_INTERVAL, 'limit': SPIKE_ATH_LIMIT})
            rows = data.get('list', []) if isinstance(data, dict) else (data or [])
        else:  # bg_fut / bg_spot — Bitget отдаёт максимум 200 свечей, тянем страницами назад по endTime
            gran = {'D': '1D', 'W': '1W'}.get(SPIKE_ATH_INTERVAL, '1D')
            step_ms = (7 if gran == '1W' else 1) * 86400 * 1000
            rows, end_ms, guard = [], int(time.time() * 1000), 0
            while len(rows) < SPIKE_ATH_LIMIT and guard < 12:
                guard += 1
                # ВСЕГДА history-candles: обычный candles отдаёт ~90 дневных свечей и обрывает пагинацию
                # (LSK 12.09: макс 0.4 вместо 2.9 -> фильтр пропустил вход, счёт ликвидирован)
                _ep = '/api/v2/mix/market/history-candles'
                page = bitget_get(_ep,
                                  params={'symbol': sym, 'productType': 'USDT-FUTURES',
                                          'granularity': gran, 'endTime': str(end_ms),
                                          'limit': str(SPIKE_ATH_BG_PAGE)})
                page = page if isinstance(page, list) else (page.get('data', []) if isinstance(page, dict) else [])
                page = [r for r in page if len(r) > 4]
                if not page:
                    break
                rows.extend(page)
                oldest = min(float(r[0]) for r in page)
                if len(page) < max(2, SPIKE_ATH_BG_PAGE // 4):   # короткая страница -> начало истории
                    break
                end_ms = int(oldest) - step_ms              # следующая страница — строго раньше
                time.sleep(0.12)
        if not rows:
            return (True, 0.0, 0.0, 100.0)
        highs  = [float(r[2]) for r in rows if len(r) > 4]
        newest = max(rows, key=lambda r: float(r[0]))      # свежая свеча (устойчиво к порядку выдачи)
        cur    = float(newest[4])
        if not highs or cur <= 0:
            return (True, cur, 0.0, 100.0)
        hist_max = max(highs)
        _prev = _spike_ath_cache.get(ckey)                 # максимум только растёт: не теряем пик при обрезанной выдаче
        if _prev and _prev[1] > hist_max:
            hist_max = _prev[1]
        _spike_ath_cache[ckey] = (time.time(), hist_max)
        log.info(f'[SPIKE] ATH {coin}/{market}: свечей={len(rows)} макс={hist_max:.8g} цена={cur:.8g} ({cur/hist_max*100:.1f}%)')
        if hist_max <= 0:
            return (True, cur, 0.0, 100.0)
        pct = cur / hist_max * 100.0
    except Exception as e:
        log.info(f'[SPIKE] {coin}/{market}: ATH-фильтр не проверен ({type(e).__name__}) — пропускаю')
        return (True, 0.0, 0.0, 100.0)                      # fail-open
    if pct < SPIKE_MIN_PCT_OF_ATH:
        return (False, cur, hist_max, pct)
    return (True, cur, hist_max, pct)


# ═══ ARBITRON: блок монет, помеченных к делистингу (arbitron.app) ═══
SPIKE_WATCH_FILE = 'spike_watchlist.txt'   # ручной список риска: Binance Monitoring Tag и т.п.,
                                           # по одному тикеру в строке (#комментарии игнорируются).
                                           # Перечитывается на каждом скане ARBITRON — правится без рестарта.
SPIKE_WATCH_SEED = ['LSK', 'ACX', 'STX']   # Binance Monitoring Tag от 24.07.2026 (стартовый набор)
SPIKE_BINANCE_TAG_ON   = True              # автоподтягивание Binance Monitoring Tag
SPIKE_BINANCE_TAG_MIN  = 360               # раз в сколько минут (6ч — метки меняются редко)
SPIKE_BINANCE_TAG_FILE = 'spike_watch_binance.json'   # кэш: переживает рестарт, работает при недоступности Binance
_ARBITRON_BLOCKED: dict = {}     # (coin, market) -> ts последнего обнаружения
_binance_tag_last = 0.0
_binance_tag_set: set = set()


def _binance_monitoring_tag():
    """Тикеры под Binance Monitoring Tag. Два источника, любой из двух достаточен:
    1) список продуктов Binance: у помеченных монет в tags есть 'Monitoring';
    2) анонсы: заголовки 'Monitoring Tag' -> тикеры в скобках; 'Remove ... Monitoring Tag' — снимаем.
    Результат кэшируется в SPIKE_BINANCE_TAG_FILE; при ошибке сети берём кэш (fail-safe, не fail-open)."""
    global _binance_tag_last, _binance_tag_set
    if not SPIKE_BINANCE_TAG_ON:
        return set()
    now = time.time()
    if _binance_tag_set and now - _binance_tag_last < SPIKE_BINANCE_TAG_MIN * 60:
        return _binance_tag_set
    import re as _re
    hdr = {'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'}
    tagged, ok = set(), False
    # --- источник 1: продукты с тегами ---
    try:
        r = requests.get('https://www.binance.com/bapi/asset/v1/public/asset-service/product/get-products',
                         headers=hdr, timeout=15)
        if r.status_code == 200:
            for p in (r.json().get('data') or []):
                tags = [str(t).lower() for t in (p.get('tags') or [])]
                if any('monitor' in t for t in tags) and str(p.get('q', '')).upper() == 'USDT':
                    tagged.add(str(p.get('b', '')).upper())
            ok = True
    except Exception as e:
        log.debug(f'binance-tag products: {e}')
    # --- источник 2: анонсы (добавления и снятия) ---
    try:
        added, removed = set(), set()
        for cat in (161, 48):        # 161 = Delisting, 48 = Latest Binance News (id могут меняться — пробуем оба)
            r = requests.get('https://www.binance.com/bapi/composite/v1/public/cms/article/list/query',
                             params={'type': 1, 'pageNo': 1, 'pageSize': 50, 'catalogId': cat},
                             headers=hdr, timeout=15)
            if r.status_code != 200:
                continue
            arts = []
            for blk in (r.json().get('data', {}).get('catalogs') or []):
                arts += blk.get('articles') or []
            for a in arts:
                t = str(a.get('title', ''))
                if 'monitoring tag' not in t.lower():
                    continue
                ticks = set(x.upper() for x in _re.findall(r'\(([A-Z0-9]{2,12})\)', t))
                (removed if _re.search(r'remove|removal', t, _re.I) else added).update(ticks)
            ok = True
        tagged |= (added - removed)
    except Exception as e:
        log.debug(f'binance-tag announcements: {e}')
    if ok:
        _binance_tag_set = tagged; _binance_tag_last = now
        try:
            with open(SPIKE_BINANCE_TAG_FILE, 'w') as f:
                json.dump({'ts': now, 'tags': sorted(tagged)}, f)
        except Exception:
            pass
        log.info(f'[WATCH] Binance Monitoring Tag: {len(tagged)} монет' +
                 (f' ({", ".join(sorted(tagged))})' if tagged else ''))
        return tagged
    # сеть недоступна -> кэш из файла
    try:
        with open(SPIKE_BINANCE_TAG_FILE) as f:
            d = json.load(f)
        _binance_tag_set = set(d.get('tags', [])); _binance_tag_last = now
        log.warning(f'[WATCH] Binance недоступен — беру кэш от {time.strftime("%d.%m %H:%M", time.localtime(d.get("ts", 0)))}: '
                    f'{len(_binance_tag_set)} монет')
    except Exception:
        log.warning('[WATCH] Binance недоступен и кэша нет — работает только ручной список')
    return _binance_tag_set


def _spike_watchlist():
    """Тикеры под ручным наблюдением: файл SPIKE_WATCH_FILE + SPIKE_WATCH_SEED. Блокируются на ОБЕИХ биржах."""
    out = set(x.strip().upper() for x in SPIKE_WATCH_SEED if x.strip())
    try:
        with open(SPIKE_WATCH_FILE, encoding='utf-8') as f:
            for ln in f:
                ln = ln.split('#')[0].strip().upper()
                if ln:
                    out.add(ln)
    except FileNotFoundError:
        pass
    except Exception as e:
        log.debug(f'watchlist: {e}')
    out |= _binance_monitoring_tag()
    return out
_arbitron_last_scan = 0.0
_arbitron_warned: set = set()

def _arbitron_scan_once():
    """Раз в SPIKE_ARBITRON_MIN мин тянет календарь Arbitron ПО СВОИМ биржам (серверный фильтр
    exchange=Bybit/Bitget), тикеры из data-search. Блок пер-биржевой: делистинг HFT на Bitget
    не трогает HFT на Bybit. Fail-open, но статус скана пишется в лог КАЖДЫЙ цикл."""
    global _arbitron_last_scan
    if not SPIKE_ARBITRON_ON:
        return
    now = time.time()
    if now - _arbitron_last_scan < SPIKE_ARBITRON_MIN * 60:
        return
    _arbitron_last_scan = now
    import re                       # локальный импорт: в торговой ячейке re не импортирован (баг v41-43)
    found = {}                      # (coin, market) -> None
    stat = []
    try:
        for ex, mkt in SPIKE_ARBITRON_EXCH.items():
            ticks = set()
            for pg in range(1, SPIKE_ARBITRON_PAGES + 1):
                url = SPIKE_ARBITRON_URL.format(ex=ex) + (f'&page={pg}' if pg > 1 else '')
                r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
                if r.status_code != 200:
                    continue
                ticks |= set(re.findall(r'data-search="([A-Z0-9]{2,20})-USDT', r.text))
            for c in ticks:
                found[(c, mkt)] = None
            stat.append(f'{ex}={len(ticks)}' + (f' ({", ".join(sorted(ticks))})' if ticks else ''))
    except Exception as e:
        log.warning(f'[ARBITRON] скан не удался: {e}')   # видно в логе, торговля не тронута
        return
    try:
        uni = _spike_get_universe()
        found = {(c, m): None for (c, m) in found if c in uni.get(m, ())}
    except Exception:
        pass
    _watch = _spike_watchlist()                      # ручной список риска (Binance Monitoring Tag и пр.)
    if _watch:
        try:
            uni = _spike_get_universe()
            for _c in _watch:
                for _m in SPIKE_MARKETS:
                    if _c in uni.get(_m, ()):
                        found[(_c, _m)] = None
            stat.append(f'watchlist={len(_watch)}')
        except Exception as e:
            log.debug(f'watchlist-merge: {e}')
    log.info(f'[ARBITRON] скан: {" | ".join(stat)}; в юниверсе под блоком: {len(found)}')
    _new = []
    for key in found:
        fresh = key not in _ARBITRON_BLOCKED
        _ARBITRON_BLOCKED[key] = now
        if fresh and key not in _arbitron_warned:
            _arbitron_warned.add(key)
            c, m = key
            _src = 'в списке риска (Monitoring Tag / ручной)' if c in _watch else 'помечена к делистингу'
            log.warning(f'[ARBITRON] {c}/{m}: {_src} — вход и доливы ЗАБЛОКИРОВАНЫ')
            _new.append(key)
    if _new:                                  # ОДНО письмо на скан (было по письму на монету — спам)
        try:
            _lines = []; _alarm = []
            for c, m in _new:
                _op = [f'{cc}/{mm}' for (cc, mm) in list(_spike_positions) if cc == c and mm == m]
                _lines.append(f'{c}/{m}' + ('   <<< ОТКРЫТА ПОЗИЦИЯ — закрой руками!' if _op else ''))
                if _op:
                    _alarm.append(f'{c}/{m}')
            _subj = (f'ARBITRON делистинг: {len(_new)} монет'
                     + (f' — ОТКРЫТЫ: {", ".join(_alarm)}' if _alarm else ''))
            spike_send_email(_subj, 'Помечены к делистингу на arbitron.app (вход и доливы заблокированы):\n'
                                    + '\n'.join(_lines))
        except Exception:
            pass
    ttl = SPIKE_ARBITRON_TTL_D * 86400
    for key in [k for k, ts in _ARBITRON_BLOCKED.items() if now - ts > ttl]:
        _ARBITRON_BLOCKED.pop(key, None)
        _arbitron_warned.discard(key)
        log.info(f'[ARBITRON] {key[0]}/{key[1]}: блок снят (устарел, {SPIKE_ARBITRON_TTL_D}д)')


def spike_scan_once():
    _spike_metrics_report()
    _spike_maybe_report()        # e-mail отчёт раз в 3 часа (в фоне, не блокирует)
    _spike_maybe_daily_report()  # раз в сутки: PnL с бирж за 24ч/72ч/7д + список сделок (как при старте)
    _spike_release()
    _arbitron_scan_once()        # календарь делистингов Arbitron (раз в SPIKE_ARBITRON_MIN мин)
    _spike_age_maybe_warmup()    # догрев кэша возраста для новых листингов (раз в 30 мин)
    _spike_soft_cleanup()        # снять TP 1% с уже закрытых мягких позиций
    with _spike_lock:
        busy = bool(SPIKE_MAX_POSITIONS) and len(_spike_positions) >= SPIKE_MAX_POSITIONS   # 0 = без ограничений
    if busy:
        return
    _spike_build_young()
    now = time.time()
    for market, coins in _spike_get_universe().items():
        for coin in coins:
            # WHITELIST: торгуем ТОЛЬКО монеты из таблицы (на своей бирже)
            if SPIKE_WHITELIST_ON and (coin, market) not in SPIKE_COIN_PARAMS:
                continue
            _wl = SPIKE_WHITELIST_ON and (coin, market) in SPIKE_COIN_PARAMS
            # рантайм-блэклист (делистинг) уважаем всегда; статический блок-лист/акции и фильтр молодых — игнор для whitelisted
            if (coin, market) in _ARBITRON_BLOCKED:       # Arbitron: помечена к делистингу на ЭТОЙ бирже
                continue
            if coin in _spike_blacklist or ((coin in SPIKE_COIN_BLOCKLIST or coin in SPIKE_STOCK_METAL_BLOCK) and not _wl):
                continue

            if coin in _spike_blacklist:                      # перманентный бан: whitelist его НЕ отменяет
                continue
            if (not _wl) and _spike_ban.get(coin, 0) > now:   # whitelist игнорирует авто-бан
                continue
            if (coin, market) in _spike_positions:        # эта пара уже открыта -> без повторного входа
                continue
            if SPIKE_ONE_POS_PER_COIN and any(c == coin for (c, _m) in list(_spike_positions)):
                continue                                  # монета уже торгуется на ДРУГОЙ бирже -> запрет (одна позиция на монету)
            key = coin
            if now - _spike_cooldown.get(key, 0) < SPIKE_COOLDOWN_SEC:
                continue
            is_spike, ratio, direction, chg = _spike_signal(coin, market)
            _soft = False
            if not is_spike:
                _sok, _free = _spike_soft_allowed(market)
                if _sok:                                          # свечи в кэше 20с — второй расчёт без запроса
                    is_spike, ratio, direction, chg = _spike_signal(coin, market, SPIKE_SOFT_VOL_MULT, SPIKE_SOFT_CHG_PCT)
                    _soft = bool(is_spike)
            if SPIKE_REPORT_SEC:
                _spike_metrics[(coin, market)] = (time.time(), ratio, chg)
            if SPIKE_FETCH_SLEEP:
                time.sleep(SPIKE_FETCH_SLEEP)
            if not is_spike:
                continue
            if market == 'bg_spot' and direction == 'short':    # спот шортить нельзя
                _spike_cooldown[key] = now; continue
            # ── ЛИМИТЫ ПО БИРЖЕ: <=2 позиций на биржу; мартингейл на бирже -> новых входов на ней нет ──
            with _spike_lock:
                _mk_pos = [p for (cc, mm), p in _spike_positions.items() if mm == market]
            if SPIKE_MAX_POS_PER_MARKET and len(_mk_pos) >= SPIKE_MAX_POS_PER_MARKET:
                continue                                          # биржа занята (2 позиции)
            if SPIKE_ONE_COIN_ONE_LEG:                            # монета уже торгуется на другой бирже?
                with _spike_lock:
                    _other = [mm for (cc, mm) in _spike_positions if cc == coin and mm != market]
                if _other:
                    log.info(f'[SPIKE] {coin}/{market}: монета уже открыта на {_other[0]} — '
                             f'вход пропущен (одна монета = одна нога)')
                    _spike_cooldown[key] = now
                    continue
            if SPIKE_BLOCK_NEW_WHILE_MART and any((p.get('rungs', 1) or 1) > 1 for p in _mk_pos):
                continue                                          # на бирже идёт мартингейл
            if direction == 'long' and not SPIKE_ALLOW_LONG:
                log.info(f'[SPIKE] {coin}/{market}: сигнал на ДАМПЕ (Δ{chg:+.1f}%) — лонги отключены -> пропуск')
                _spike_cooldown[key] = now; continue
            if direction == 'short' and not SPIKE_ALLOW_SHORT:
                log.info(f'[SPIKE] {coin}/{market}: сигнал на ПАМПЕ (Δ{chg:+.1f}%) — шорты отключены -> пропуск')
                _spike_cooldown[key] = now; continue
            side = SPIKE_COIN_SIDE.get(coin)
            if side == 'short_only' and direction == 'long':
                log.info(f'[SPIKE] {coin}/{market}: LONG запрещён (short_only) -> пропуск')
                _spike_cooldown[key] = now; continue
            if side == 'long_only' and direction == 'short':
                log.info(f'[SPIKE] {coin}/{market}: SHORT запрещён (long_only) -> пропуск')
                _spike_cooldown[key] = now; continue
            if not _wl:
                _age_ok, _age_d = _spike_age_ok(coin, market)     # первая дневная свеча своей биржи, кэш
                if not _age_ok:
                    log.warning(f'[SPIKE] {coin}/{market}: ПРОПУСК входа (молодая) — '
                                f'монете {_age_d:.0f}д < {SPIKE_AGE_BLOCK_D}д по первой свече')
                    _spike_cooldown[key] = now; continue
                if _age_d is not None and _age_d < SPIKE_AGE_SHORT_D and direction == 'long':
                    log.info(f'[SPIKE] {coin}/{market}: LONG запрещён — монете {_age_d:.0f}д '
                             f'({SPIKE_AGE_BLOCK_D}-{SPIKE_AGE_SHORT_D}д: только шорт) -> пропуск')
                    _spike_cooldown[key] = now; continue
            _liq_ok, _v24, _oi, _lr = _spike_liquidity_ok(coin, market)
            if not _liq_ok:
                log.warning(f'[SPIKE] {coin}/{market}: ПРОПУСК входа (риск делистинга) — {_lr} '
                            f'[оборот24ч ${_v24:,.0f}, OI ${_oi:,.0f}]')
                _spike_cooldown[key] = now; continue
            if SPIKE_MIN_DAILY_CANDLES:                     # страховка возраста: мало дневных свечей = свежий листинг
                _dc = _spike_pctl_closes(coin, market)
                if _dc is not None and len(_dc) < SPIKE_MIN_DAILY_CANDLES:
                    log.warning(f'[SPIKE] {coin}/{market}: ПРОПУСК входа (свежий листинг) — дневных закрытий '
                                f'{len(_dc)} < {SPIKE_MIN_DAILY_CANDLES} (кэш возраста считал монету старой)')
                    _spike_cooldown[key] = now; continue
            if direction == 'long' and SPIKE_LOW_PCTL:      # дно собственного диапазона -> не лонгуем (SOPH-кейс)
                _thr = _spike_low_pctl_price(coin, market)
                if _thr:
                    _cnd = _spike_candles(coin, market)
                    _px_now = _cnd[-1][2] if _cnd else 0
                    if _px_now and _px_now < _thr:
                        log.warning(f'[SPIKE] {coin}/{market}: ПРОПУСК входа (дно диапазона) — цена {_px_now:.8g} '
                                    f'ниже {SPIKE_LOW_PCTL}-го перцентиля {_thr:.8g} за {SPIKE_LOW_PCTL_DAYS}д')
                        _spike_cooldown[key] = now; continue
            if direction == 'long' and SPIKE_HIGH_PCTL:     # верх диапазона -> не покупаем откат сдувающегося пампа
                _thr_hi = _spike_high_pctl_price(coin, market)
                if _thr_hi:
                    _cnd = _spike_candles(coin, market)
                    _px_now = _cnd[-1][2] if _cnd else 0
                    if _px_now and _px_now > _thr_hi:
                        log.warning(f'[SPIKE] {coin}/{market}: ПРОПУСК входа (верх диапазона) — цена {_px_now:.8g} '
                                    f'выше {SPIKE_HIGH_PCTL}-го перцентиля {_thr_hi:.8g} за {SPIKE_LOW_PCTL_DAYS}д')
                        _spike_cooldown[key] = now; continue
            _floor_ok, _cur, _hmax, _pct = _spike_above_floor(coin, market)
            if not _floor_ok:
                log.warning(f'[SPIKE] {coin}/{market}: ПРОПУСК входа (упал от пика) — '
                            f'цена {_cur:.8g} = {_pct:.1f}% от макс {_hmax:.8g} (порог {SPIKE_MIN_PCT_OF_ATH:.0f}%)')
                _spike_cooldown[key] = now; continue
            if _soft:
                log.info(f'[SPIKE] {coin}/{market}: МЯГКИЙ сигнал ×{ratio:.1f} Δ{chg:+.1f}% '
                         f'(порог ×{SPIKE_SOFT_VOL_MULT}/{SPIKE_SOFT_CHG_PCT}%, свободно ног {_free}) -> цель {SPIKE_SOFT_TP_PCT}%')
                _spike_tp_override[(coin, market)] = SPIKE_SOFT_TP_PCT; _spike_soft_save()
            _spike_open(coin, market, direction, ratio, chg)
            if _soft:
                with _spike_lock:
                    _opened = (coin, market) in _spike_positions
                if not _opened:                                   # вход не исполнился -> снимаем цель 1%
                    _spike_tp_override.pop((coin, market), None); _spike_soft_save()
            if SPIKE_MAX_POSITIONS:
                with _spike_lock:
                    full = len(_spike_positions) >= SPIKE_MAX_POSITIONS
                if full:
                    return                   # достигнут лимит позиций -> ждём закрытия


def _spike_loop():
    log.info(f'[SPIKE] старт: рынки={SPIKE_MARKETS} {SPIKE_MAX_COINS or "все"}/рынок '
             f'×{SPIKE_VOL_MULT} TP={SPIKE_TP_PCT}% DRY_RUN={SPIKE_DRY_RUN}')
    while not _spike_stop.is_set():
        try:
            spike_scan_once()
        except Exception as e:
            log.warning(f'[SPIKE] loop: {e}')
        _spike_stop.wait(SPIKE_POLL_SEC)
    log.info('[SPIKE] стоп')


def start_spike_monitor():
    global _spike_thread
    if _spike_thread and _spike_thread.is_alive():
        print('Spike-монитор уже работает'); return
    _spike_stop.clear()
    _spike_ban_load()
    _spike_soft_load()
    _spike_ws_start()
    _spike_recover_positions()          # подхватить висящие позиции -> доторговать
    _spike_startup_report_once()        # старт-отчёт PnL за 24ч/72ч/7д на почту
    _spike_age_warmup()                 # возраст всего юниверсума по первой свече (в фоне, кэш в файл)
    _spike_thread = threading.Thread(target=_spike_loop, daemon=True, name='spike')
    _spike_thread.start()
    print(f'Spike-монитор запущен (свой универсум по рынкам, DRY_RUN={SPIKE_DRY_RUN})')


def stop_spike_monitor():
    _spike_stop.set()
    print('Spike-монитор остановлен (TP-ордера остаются на бирже)')


def spike_status():
    with _spike_lock:
        if not _spike_positions:
            print('Открытых spike-сделок нет.'); return
        for (coin, market), p in _spike_positions.items():
            age = (time.time() - p['opened']) / 60
            print(f'  {coin}/{market} {p["direction"]} @{p.get("entry_px")} -> TP@{p.get("tp_px")} ({age:.1f} мин)')


def spike_run_foreground():
    """Блокирующий запуск в ТЕКУЩЕЙ ячейке — видно вживую. Стоп: кнопка прерывания (■)."""
    _spike_ban_load()       # восстановить недельные баны из файла
    _spike_martban_load()   # + счётчик мартингейл-банов и перманентный список
    _spike_ws_start()       # поднять WS-тики активной позиции (точный стоп) + самодиагностика в лог
    _spike_recover_positions()          # подхватить висящие позиции -> доторговать
    _spike_startup_report_once()        # старт-отчёт PnL за 24ч/72ч/7д на почту
    _spike_age_warmup()                 # возраст всего юниверсума по первой свече (в фоне, кэш в файл)
    uni = _spike_get_universe()
    print('Универсум: ' + ', '.join(f'{m}={len(c)}' for m, c in uni.items())
          + f' | ×{SPIKE_VOL_MULT} Δ{SPIKE_PRICE_CHG_PCT}%/{SPIKE_PRICE_WIN_MIN}м {"fade" if SPIKE_FADE else "momentum"}'
          + f' TP={SPIKE_TP_PCT}% SL={"выкл" if not SPIKE_STOP_PCT else str(SPIKE_STOP_PCT)+chr(37)}'
          + f' таймаут={SPIKE_MAX_HOLD_SEC//60}м бан-после-стопа={SPIKE_STOP_BAN_SEC//86400}д DRY_RUN={SPIKE_DRY_RUN}'
          + (f' возраст>{SPIKE_MIN_AGE_DAYS}д' if SPIKE_MIN_AGE_DAYS else '')
          + (f' акции/металлы={len(SPIKE_STOCK_METAL_BLOCK)}' if SPIKE_STOCK_METAL_BLOCK else '')
          + (f' | блок-лист={sorted(SPIKE_COIN_BLOCKLIST)} стороны={SPIKE_COIN_SIDE}' if SPIKE_COIN_BLOCKLIST or SPIKE_COIN_SIDE else ''), flush=True)
    # ── ДИАГНОСТИКА WHITELIST: сколько пар реально торгуемы прямо сейчас ──
    if SPIKE_WHITELIST_ON:
        _now0 = time.time(); _active = []; _blocked = []
        for (_cn, _mk), (_tp, _sl) in sorted(SPIKE_COIN_PARAMS.items()):
            _in_uni = _cn in (uni.get(_mk) or [])
            _blk = _cn in _spike_blacklist            # авто-бан для whitelist игнорируется -> не блокиратор
            if _in_uni and not _blk:
                _was = ' (был бан, снят)' if _spike_ban.get(_cn, 0) > _now0 else ''
                _active.append(f'{_cn}/{_mk} TP{_tp}/SL{_sl}{_was}')
            else:
                _why = []
                if not _in_uni: _why.append('нет в юниверсе')
                if _blk:        _why.append('делистинг-блэклист')
                _blocked.append(f'{_cn}/{_mk}(' + ', '.join(_why) + ')')
        print(f'WHITELIST ON: торгуемых сейчас {len(_active)} из {len(SPIKE_COIN_PARAMS)}', flush=True)
        print('  активны: ' + (', '.join(_active) if _active else '— НИ ОДНОЙ (сигналов не будет!)'), flush=True)
        if _blocked:
            print('  недоступны: ' + ', '.join(_blocked), flush=True)
    else:
        print('WHITELIST OFF: торгуется весь юниверс', flush=True)
    n = 0; last_hb = time.time()
    try:
        while True:
            n += 1
            try:
                spike_scan_once()
            except Exception as e:
                import traceback
                print(f'[{time.strftime("%H:%M:%S")}] ОШИБКА цикл {n}: {type(e).__name__}: {e}', flush=True)
                print('   ' + traceback.format_exc().strip().replace('\n', '\n   '), flush=True)
                log.warning(f'[SPIKE] foreground scan: {type(e).__name__}: {e}')
            now = time.time()
            with _spike_lock:
                occ = bool(_spike_positions)
            if not occ and SPIKE_HEARTBEAT_MIN and now - last_hb >= SPIKE_HEARTBEAT_MIN * 60:
                last_hb = now
                print(f'[{time.strftime("%H:%M:%S")}] жив, открытых позиций нет, жду сигнал', flush=True)
            time.sleep(SPIKE_POLL_SEC)
    except KeyboardInterrupt:
        print('Остановлено (прерывание).')


print(f'Spike-scalper загружен | ×{SPIKE_VOL_MULT} TP={SPIKE_TP_PCT}% DRY_RUN={SPIKE_DRY_RUN}')
print('Видно вживую:  spike_run_foreground()   (блокирует ячейку, стоп — кнопка ■)')
print('В фоне (молча): start_spike_monitor() | stop_spike_monitor() | spike_status()')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ДОПОЛНЕНИЕ к спайк-боту:
#   (1) сверка PnL КАЖДОЙ закрытой сделки с реальными данными биржи
#   (2) e-mail отчёт раз в 3 часа (реальный PnL с бирж + сверка + открытые)
# Переиспользует bybit_get / bitget_get (signed) из API-клиента и log.
# Решения портированы из арбитражного бота (__54_): send_email + closed-pnl.
# ══════════════════════════════════════════════════════════════════════════
import os, ssl, smtplib, threading
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
from email.utils import formatdate
from datetime import datetime

# ── e-mail конфиг (Gmail App Password; можно переопределить через окружение) ──
EMAIL_ENABLED  = True
EMAIL_FROM     = os.environ.get('EMAIL_FROM',     '')
EMAIL_TO       = os.environ.get('EMAIL_TO',       '')
EMAIL_PASSWORD = os.environ.get('EMAIL_PASSWORD', '')   # без пробелов
SMTP_HOST, SMTP_PORT = 'smtp.gmail.com', 465
SPIKE_REPORT_EVERY_SEC = 3 * 3600        # отчёт на почту раз в 3 часа
SPIKE_DAILY_REPORT_SEC = 86400           # повтор PnL-отчёта с бирж (24ч/72ч/7д) каждые сутки после запуска

_spike_email_lock = threading.Lock()
_spike_email_n    = 0

def spike_send_email(subject: str, body: str, attachments=None) -> bool:
    """Отправка через Gmail SMTP_SSL. attachments: [(имя.png, bytes), ...]. Ошибка -> warning, бот продолжает."""
    global _spike_email_n
    if not EMAIL_ENABLED or not EMAIL_PASSWORD:
        log.info('[SPIKE] e-mail выключен или пуст пароль -> skip'); return False
    if attachments:
        msg = MIMEMultipart()
        msg.attach(MIMEText(body, _charset='utf-8'))
        for _name, _data in attachments:
            try:
                _img = MIMEImage(_data, _subtype='png')
                _img.add_header('Content-Disposition', 'attachment', filename=_name)
                msg.attach(_img)
            except Exception as _e:
                log.debug(f'email attach {_name}: {_e}')
    else:
        msg = MIMEText(body, _charset='utf-8')
    subject = subject[8:] if subject.startswith('[spike] ') else subject   # без задвоения префикса
    msg['Subject'] = f'[spike] {subject}'
    msg['From']    = EMAIL_FROM
    msg['To']      = EMAIL_TO
    msg['Date']    = formatdate(localtime=True)
    try:
        with _spike_email_lock:
            ctx = ssl.create_default_context()
            with smtplib.SMTP_SSL(SMTP_HOST, SMTP_PORT, context=ctx, timeout=15) as s:
                s.login(EMAIL_FROM, EMAIL_PASSWORD)
                s.sendmail(EMAIL_FROM, [EMAIL_TO], msg.as_string())
            _spike_email_n += 1
        log.info(f'[SPIKE] e-mail отправлен ({_spike_email_n}): {subject[:70]}')
        return True
    except Exception as e:
        log.warning(f'[SPIKE] e-mail ОШИБКА: {type(e).__name__}: {e}')
        return False

# ── реальный реализованный PnL С БИРЖИ (net, включает комиссии) ──
def spike_real_pnl_bb(coin, since_ts, until_ts=None):
    """Bybit /v5/position/closed-pnl по символу -> сумма closedPnl + avgExitPrice."""
    if until_ts is None: until_ts = time.time()
    try:
        items = bybit_get('/v5/position/closed-pnl',
                          {'category':'linear','symbol':f'{coin}USDT',
                           'startTime':int(since_ts*1000),'endTime':int(until_ts*1000),'limit':50})
        pnl=0.0; n=0; exitpx=None
        for it in items:
            try: pnl += float(it.get('closedPnl',0) or 0); n+=1
            except (TypeError,ValueError): pass
            if exitpx is None and it.get('avgExitPrice'):
                try: exitpx=float(it['avgExitPrice'])
                except (TypeError,ValueError): pass
        return {'pnl':round(pnl,4),'count':n,'exit':exitpx,'error':None}
    except Exception as e:
        return {'pnl':0.0,'count':0,'exit':None,'error':str(e)[:200]}

def _bg_fills_raw(params):
    """Сырой signed GET fill-history -> (fillList, raw_json). Данные в data.fillList.
    Тело читаем ДАЖЕ при HTTP 400 — там настоящая причина (подпись/права/параметр),
    raise_for_status её прячет за \'400 Bad Request\'."""
    from urllib.parse import urlencode
    path='/api/v2/mix/order/fill-history'
    r=BITGET_SESSION.get(BITGET_BASE+path+'?'+urlencode(params),
                         headers=_bitget_auth('GET', path, params), timeout=10)
    try:
        j=r.json()
    except Exception:
        raise ValueError(f'Bitget fill-history HTTP {r.status_code}: {r.text[:200]}')
    code=str(j.get('code','0'))
    if code not in ('0','00000'):
        raise ValueError(f'Bitget fill-history [{code}]: {j.get("msg")} (HTTP {r.status_code})')
    d=j.get('data',{}) or {}
    if isinstance(d, list):                       # на всякий случай
        return d, j
    fills=d.get('fillList') or d.get('list') or []
    return fills, j

def spike_bg_diag():
    """Быстрая диагностика Bitget-доступов: дёргаем подписанные эндпоинты и печатаем СЫРОЙ ответ.
    Сразу видно, подпись/права/UID — без ожидания письма."""
    from urllib.parse import urlencode
    tests=[('/api/v2/mix/account/accounts', {'productType':'USDT-FUTURES'}),
           ('/api/v2/spot/account/info', {}),
           ('/api/v2/mix/order/fill-history', {'productType':'USDT-FUTURES','limit':'10'})]
    for path,params in tests:
        try:
            url=BITGET_BASE+path+(('?'+urlencode(params)) if params else '')
            r=BITGET_SESSION.get(url, headers=_bitget_auth('GET', path, params), timeout=10)
            print(f'{path}\n   HTTP {r.status_code}: {r.text[:240]}\n')
        except Exception as e:
            print(f'{path}\n   EXC {type(e).__name__}: {e}\n')

def _bg_realized_pnl(since_ts, until_ts, symbol=None, max_pages=8):
    """Сумма profit по fill-history за окно. Пагинация по endId с дедупликацией по tradeId
    (защита от неверного курсора). symbol=None -> весь аккаунт."""
    base={'productType':'USDT-FUTURES','startTime':str(int(since_ts*1000)),
          'endTime':str(int(until_ts*1000)),'limit':'100'}
    if symbol: base['symbol']=symbol
    seen={}; endId=None; pages=0; last=None; err=None
    try:
        while pages<max_pages:
            params=dict(base)
            if endId: params['idLessThan']=endId
            fills, last=_bg_fills_raw(params); pages+=1
            new=0
            for it in fills:
                fid=it.get('tradeId') or it.get('execId') or it.get('orderId')
                key=fid or f'{it.get("cTime")}_{it.get("price")}_{it.get("baseVolume")}'
                if key not in seen: seen[key]=it; new+=1
            if (not fills) or new==0 or len(fills)<100: break
            endId=(last.get('data',{}) or {}).get('endId') or fills[-1].get('tradeId')
            if not endId: break
    except Exception as e:
        err=str(e)[:200]
    total=0.0; n=0
    for it in seen.values():
        pr=it.get('profit', it.get('execPnl'))
        if pr in (None,'','0','0.0','0.00'): continue
        try: total+=float(pr); n+=1
        except (TypeError,ValueError): pass
    return {'pnl':round(total,4),'count':n,'pages':pages,'error':err,'raw':last}

def spike_real_pnl_bg(coin, since_ts, until_ts=None):
    """Реальный реализованный PnL по символу с Bitget (data.fillList.profit)."""
    if until_ts is None: until_ts = time.time()
    r=_bg_realized_pnl(since_ts, until_ts, symbol=f'{coin}USDT')
    if r['count']==0 and not r['error'] and r.get('raw') is not None:
        body=str(r['raw'])[:300].replace(chr(10),' ')
        log.info(f'[PNL-CHECK] bg fill-history пусто по {coin}USDT (стр.{r["pages"]}): {body}')
    return {'pnl':r['pnl'],'count':r['count'],'exit':None,'error':r['error']}

def spike_real_pnl(coin, market, since_ts, until_ts=None):
    return (spike_real_pnl_bb if market=='bb_fut' else spike_real_pnl_bg)(coin, since_ts, until_ts)

# ── аккаунт-широкий PnL за окно (для отчёта, без фильтра по символу) ──
def spike_window_pnl_bb(since_ts, until_ts=None):
    if until_ts is None: until_ts=time.time()
    try:
        items=bybit_get('/v5/position/closed-pnl',
                        {'category':'linear','startTime':int(since_ts*1000),
                         'endTime':int(until_ts*1000),'limit':100})
        pnl=sum(float(it.get('closedPnl',0) or 0) for it in items)
        return {'pnl':round(pnl,4),'count':len(items),'error':None}
    except Exception as e:
        return {'pnl':0.0,'count':0,'error':str(e)[:200]}

def spike_window_pnl_bg(since_ts, until_ts=None):
    """Аккаунт-широкий реальный PnL Bitget за окно (data.fillList.profit, с пагинацией)."""
    if until_ts is None: until_ts=time.time()
    r=_bg_realized_pnl(since_ts, until_ts, symbol=None)
    return {'pnl':r['pnl'],'count':r['count'],'error':r['error']}

# ── журнал сверок «оценка бота vs реал биржи» (копится для 3-час. отчёта) ──
_spike_pnl_audit  = []
_spike_audit_lock = threading.Lock()

def _spike_verify_pnl(coin, market, opened_ts, est_usdt, tag, delay=6.0):
    """В фоне: дать бирже зарегистрировать закрытие, взять реальный PnL, сравнить с оценкой."""
    def _job():
        time.sleep(delay)
        info=None
        for _ in range(3):                      # закрытие может появиться не сразу
            info = spike_real_pnl(coin, market, opened_ts-5)
            if info.get('count'): break
            time.sleep(4.0)
        real   = info.get('pnl')  if (info and not info.get('error') and info.get('count')) else None
        exitpx = info.get('exit') if info else None
        with _spike_audit_lock:
            _spike_pnl_audit.append({'ts':time.time(),'coin':coin,'market':market,'tag':tag,
                                     'est':est_usdt,'real':real,'exit':exitpx,
                                     'err':(info or {}).get('error')})
        if real is None:
            log.warning(f'[PNL-CHECK] {coin}/{market} {tag}: реал недоступен '
                        f'({(info or {}).get("error")}) | оценка={est_usdt:+.4f} USDT')
        else:
            d=real-est_usdt; ex=f' exit@{exitpx:.8g}' if exitpx else ''
            log.info(f'[PNL-CHECK] {coin}/{market} {tag}: оценка={est_usdt:+.4f} / '
                     f'РЕАЛ={real:+.4f} USDT (Δ{d:+.4f}){ex}')
    threading.Thread(target=_job, daemon=True, name=f'pnlchk-{coin}').start()

# ── 3-часовой отчёт ──
_spike_session_start = time.time()
_spike_report_last   = time.time()

def _spike_fmt_dur(sec):
    h=int(sec//3600); m=int((sec%3600)//60); return f'{h}ч{m:02d}м'

def _spike_build_report(since, now):
    wb=spike_window_pnl_bb(since, now); wg=spike_window_pnl_bg(since, now)
    real_total=(wb['pnl'] if not wb['error'] else 0)+(wg['pnl'] if not wg['error'] else 0)
    with _spike_audit_lock:
        aud=[a for a in _spike_pnl_audit if a['ts']>=since]
    est_sum =sum(a['est']  for a in aud if a['est']  is not None)
    real_sum=sum(a['real'] for a in aud if a['real'] is not None)
    with _spike_lock:
        opens=[(c,m,p) for (c,m),p in _spike_positions.items()]
    bans=sum(1 for ts in _spike_ban.values() if ts>now)
    h0=datetime.fromtimestamp(since).strftime('%m-%d %H:%M'); h1=datetime.fromtimestamp(now).strftime('%H:%M')
    subj=f'{h0}-{h1} | сделок={len(aud)} | реал PnL={real_total:+.3f} USDT'
    L=[f'ОТЧЁТ СПАЙК-БОТА  {h0} — {h1}   (uptime {_spike_fmt_dur(now-_spike_session_start)})',
       '='*54, '',
       'РЕАЛЬНЫЙ PnL С БИРЖ ЗА ОКНО (closed-pnl / fill-history):',
       f'  Bybit : {wb["pnl"]:+.4f} USDT  ({wb["count"]} закр.)'+(f'  ОШИБКА: {wb["error"]}' if wb['error'] else ''),
       f'  Bitget: {wg["pnl"]:+.4f} USDT  ({wg["count"]} филлов)'+(f'  ОШИБКА: {wg["error"]}' if wg['error'] else ''),
       f'  ИТОГО : {real_total:+.4f} USDT', '']
    if aud:
        L.append(f'СВЕРКА ПО СДЕЛКАМ ({len(aud)}): оценка бота vs реал биржи')
        for a in aud:
            r=f'{a["real"]:+.4f}' if a['real'] is not None else 'н/д'
            d=f'{a["real"]-a["est"]:+.4f}' if a['real'] is not None else '—'
            L.append(f'  {a["coin"]}/{a["market"]} {a["tag"]}: оц {a["est"]:+.4f} / реал {r} (Δ{d})')
        L.append(f'  СУММА: оценка {est_sum:+.4f} / реал {real_sum:+.4f} USDT')
    else:
        L.append('Закрытых сделок за окно нет.')
    L.append('')
    if opens:
        L.append(f'ОТКРЫТО СЕЙЧАС ({len(opens)}):')
        for c,m,p in opens:
            L.append(f'  {c}/{m} {p["direction"]} вход@{p.get("entry_px")} '
                     f'TP@{p.get("tp_px")} SL@{p.get("sl_px")} ({(now-p["opened"])/60:.0f}м)')
    else:
        L.append('Открытых позиций нет.')
    L.append(''); L.append(f'монет под баном: {bans} | писем за сессию: {_spike_email_n}')
    return subj, '\n'.join(L)

def _spike_maybe_report():
    """Вызывается каждый цикл скана; шлёт письмо не чаще раза в SPIKE_REPORT_EVERY_SEC."""
    global _spike_report_last
    now=time.time()
    if now - _spike_report_last < SPIKE_REPORT_EVERY_SEC: return
    since=_spike_report_last; _spike_report_last=now
    try:
        subj, body=_spike_build_report(since, now)
        threading.Thread(target=spike_send_email, args=(subj, body), daemon=True).start()
    except Exception as e:
        log.warning(f'[SPIKE] отчёт не собран: {type(e).__name__}: {e}')

_spike_daily_report_last = time.time()
def _spike_maybe_daily_report():
    """Каждые SPIKE_DAILY_REPORT_SEC (сутки) повторяет отчёт с бирж: PnL 24ч/72ч/7д + список сделок."""
    global _spike_daily_report_last
    now = time.time()
    if now - _spike_daily_report_last < SPIKE_DAILY_REPORT_SEC:
        return
    _spike_daily_report_last = now
    def _job():
        try: spike_startup_report(send=True, tag='СУТКИ')
        except Exception as e: log.warning(f'[SPIKE] суточный отчёт не собран: {type(e).__name__}: {e}')
    threading.Thread(target=_job, daemon=True, name='spike-daily').start()

def spike_report_now():
    """Ручной отчёт прямо сейчас (за окно с последнего автоотчёта)."""
    subj, body=_spike_build_report(_spike_report_last, time.time())
    print(subj); print(body)
    return spike_send_email(subj, body)

def spike_test_email():
    """Проверка SMTP без ожидания 3 часов."""
    return spike_send_email('тест', 'Проверка связи: SMTP работает.')


# ── СТАРТ-ОТЧЁТ: реальный PnL с бирж за 24ч и 72ч (с пагинацией по времени) ──
def _spike_pnl_chunked(which, since_ts, until_ts, chunk_sec=24*3600):
    """Сумма реального PnL за [since,until], нарезая на чанки -> обход лимита 100/запрос."""
    total=0.0; cnt=0; err=None; t=since_ts
    while t < until_ts:
        t2=min(t+chunk_sec, until_ts)
        r=(spike_window_pnl_bb if which=='bb' else spike_window_pnl_bg)(t, t2)
        if r['error']: err=r['error']
        total+=r['pnl']; cnt+=r['count']; t=t2
    return {'pnl':round(total,4),'count':cnt,'error':err}

def _spike_bb_trades(since_ts, until_ts):
    """Список закрытых сделок Bybit за окно: [{sym,side,pnl,ts}]. По суточным чанкам."""
    out=[]; t=since_ts
    while t<until_ts:
        t2=min(t+24*3600, until_ts)
        try:
            items=bybit_get('/v5/position/closed-pnl',
                            {'category':'linear','startTime':int(t*1000),'endTime':int(t2*1000),'limit':100})
            for it in items:
                try: pnl=float(it.get('closedPnl',0) or 0)
                except (TypeError,ValueError): pnl=0.0
                out.append({'sym':str(it.get('symbol','')),'side':str(it.get('side','')),'pnl':pnl,
                            'ts':int(float(it.get('updatedTime') or it.get('createdTime') or 0))})
        except Exception as e:
            out.append({'sym':'(ошибка Bybit)','side':'','pnl':0.0,'ts':0,'err':str(e)[:80]})
        t=t2
    return out

def _spike_bg_trades(since_ts, until_ts):
    """Список закрытых сделок Bitget за окно: [{sym,side,pnl,ts}] (закрывающие филлы, profit!=0)."""
    out=[]; t=since_ts
    while t<until_ts:
        t2=min(t+24*3600, until_ts); endId=None; pages=0; seen=set()
        while pages<8:
            params={'productType':'USDT-FUTURES','startTime':str(int(t*1000)),
                    'endTime':str(int(t2*1000)),'limit':'100'}
            if endId: params['idLessThan']=endId
            try:
                fills,raw=_bg_fills_raw(params)
            except Exception as e:
                out.append({'sym':'(ошибка Bitget)','side':'','pnl':0.0,'ts':0,'err':str(e)[:80]}); fills=[]; raw={}; break
            pages+=1; new=0
            for it in fills:
                fid=it.get('tradeId') or it.get('execId')
                if fid and fid in seen: continue
                if fid: seen.add(fid)
                new+=1
                pr=it.get('profit', it.get('execPnl'))
                if pr in (None,'','0','0.0','0.00'): continue
                try: pnl=float(pr)
                except (TypeError,ValueError): continue
                out.append({'sym':str(it.get('symbol','')),'side':str(it.get('side','')),'pnl':pnl,
                            'ts':int(float(it.get('cTime') or it.get('createdTime') or 0))})
            if (not fills) or new==0 or len(fills)<100: break
            endId=(raw.get('data',{}) or {}).get('endId') or (fills[-1].get('tradeId') if fills else None)
            if not endId: break
        t=t2
    return out

def spike_trades_window(hours=72):
    """Сделки с обеих бирж за последние N часов, отсортированы (новые сверху)."""
    now=time.time(); since=now-hours*3600
    tr=[]
    for r in _spike_bb_trades(since, now): r['ex']='bb'; tr.append(r)
    for r in _spike_bg_trades(since, now): r['ex']='bg'; tr.append(r)
    tr.sort(key=lambda x: x.get('ts',0), reverse=True)
    return tr

def spike_pnl_windows(windows=(('24ч',86400),('72ч',3*86400),('7д',7*86400))):
    """Реальный PnL по каждому окну с обеих бирж."""
    now=time.time(); out={}
    for label,sec in windows:
        bb=_spike_pnl_chunked('bb', now-sec, now)
        bg=_spike_pnl_chunked('bg', now-sec, now)
        tot=(bb['pnl'] if not bb['error'] else 0)+(bg['pnl'] if not bg['error'] else 0)
        out[label]={'bb':bb,'bg':bg,'total':round(tot,4)}
    return now, out

_spike_startup_sent=False
def spike_startup_report(send=True, list_hours=72, max_rows=60, tag='СТАРТ'):
    """Отчёт: суммы за 24ч/72ч/7д + СПИСОК сделок с PnL -> в лог и на почту (старт и раз в сутки)."""
    now, win = spike_pnl_windows()
    L=[f'{tag}-ОТЧЁТ СПАЙК-БОТА  {datetime.fromtimestamp(now).strftime("%Y-%m-%d %H:%M")}',
       '='*54, '', 'РЕАЛЬНЫЙ PnL С БИРЖ (closed-pnl / fill-history, net с комиссией):']
    for label,d in win.items():
        bb=d['bb']; bg=d['bg']
        L.append(f'  за {label}:')
        L.append(f'      Bybit : {bb["pnl"]:+.4f} USDT ({bb["count"]} закр.)' + (f'  ОШИБКА:{bb["error"]}' if bb['error'] else ''))
        L.append(f'      Bitget: {bg["pnl"]:+.4f} USDT ({bg["count"]} филлов)' + (f'  ОШИБКА:{bg["error"]}' if bg['error'] else ''))
        L.append(f'      ИТОГО : {d["total"]:+.4f} USDT')
    # --- список сделок с PnL ---
    try:
        trades=spike_trades_window(list_hours)
    except Exception as e:
        trades=[]; L.append(f'\n(не удалось получить список сделок: {type(e).__name__}: {e})')
    real=[t for t in trades if not t.get('err')]
    errs=[t for t in trades if t.get('err')]
    s=sum(t['pnl'] for t in real)
    L.append(''); L.append(f'СДЕЛКИ ЗА {list_hours}ч: {len(real)} шт, сумма PnL={s:+.4f} USDT')
    if real:
        L.append('  дата/время        биржа монета            сторона      PnL USDT')
        for t in real[:max_rows]:
            tm=datetime.fromtimestamp(t['ts']/1000).strftime('%m-%d %H:%M') if t['ts'] else '  --  '
            ex='Bybit ' if t['ex']=='bb' else 'Bitget'
            L.append(f'  {tm}   {ex}  {t["sym"][:16]:16} {t["side"][:10]:10} {t["pnl"]:+.4f}')
        if len(real)>max_rows:
            L.append(f'  ... ещё {len(real)-max_rows} сделок (показаны последние {max_rows})')
    else:
        L.append('  (закрытых сделок за окно нет)')
    for e in errs[:4]:
        L.append(f'  ! {e["sym"]}: {e.get("err")}')
    body='\n'.join(L)
    subj=f'{tag} | 24ч={win["24ч"]["total"]:+.3f} | 72ч={win["72ч"]["total"]:+.3f} | 7д={win["7д"]["total"]:+.3f} USDT | сделок{list_hours}ч={len(real)}'
    for ln in body.split('\n'): log.info('[SPIKE] '+ln)
    if send: return spike_send_email(subj, body)
    return True

def _spike_startup_report_once():
    """Шлёт старт-отчёт один раз за запуск бота (в фоне, не блокирует торговлю)."""
    global _spike_startup_sent
    if _spike_startup_sent: return
    _spike_startup_sent=True
    def _job():
        try: spike_startup_report(send=True)
        except Exception as e: log.warning(f'[SPIKE] старт-отчёт не собран: {type(e).__name__}: {e}')
    threading.Thread(target=_job, daemon=True, name='spike-startup').start()



# ══════════════════════════════════════════════════════════════════════════
# ПОСТ-АНАЛИЗ TP/SL ПО ЗАПИСАННЫМ ТИКАМ: для убыточных сделок пересчитываем
# оптимальные TP/SL по реальному пути цены этой пары. Подсказка, НЕ автозамена.
# Путь цены пишется в _spike_fast_stop (state _spike_path в торговой ячейке).
# ══════════════════════════════════════════════════════════════════════════
SPIKE_HIST_FILE       = 'spike_trade_hist.json'
SPIKE_SUGGEST_FILE    = 'spike_tpsl_suggest.json'
SPIKE_HIST_MAX        = 40           # хранить последних N сделок на (монета|рынок)
SPIKE_TPSL_MIN_TRADES = 5            # минимум сделок с тиками для подсказки
SPIKE_TPSL_STORE_PTS  = 400          # точек пути сохраняем на сделку
SPIKE_TPSL_TP_GRID    = [0.8,1.0,1.2,1.5,2.0,2.5,3.0]
SPIKE_TPSL_SL_GRID    = [2.0,3.0,4.0,4.5,5.0,6.0,8.0]
SPIKE_TPSL_AUTOAPPLY  = False        # (legacy) общий автоприменитель — не используется
SPIKE_POSTCLOSE_SEC      = 1800      # дозапись цены ПОСЛЕ закрытия, сек (30 мин)
SPIKE_POSTCLOSE_POLL_SEC = 6         # шаг REST-опроса цены после закрытия, сек
SPIKE_TPSL_AUTOAPPLY_ON_LOSS = False # РЕЖИМ ЛОГИРОВАНИЯ: ничего не применяем, только пишем новые TP/SL в лог по всем сделкам
SPIKE_APPLIED_FILE    = 'spike_tpsl_applied.json'  # применённые авто-TP/SL (переживают рестарт)

_spike_trade_hist   = {}             # 'coin|market' -> list[rec]
_spike_tpsl_suggest = {}             # 'coin|market' -> подсказка
_spike_hist_lock    = threading.Lock()

def _spike_hist_save():
    try:
        with open(SPIKE_HIST_FILE,'w') as f: json.dump(_spike_trade_hist,f)
    except Exception as e: log.debug(f'[TPSL] hist save: {e}')

def _spike_hist_load():
    global _spike_trade_hist
    try:
        with open(SPIKE_HIST_FILE) as f: _spike_trade_hist=json.load(f)
        tot=sum(len(v) for v in _spike_trade_hist.values())
        log.info(f'[TPSL] история путей загружена: {tot} сделок по {len(_spike_trade_hist)} парам')
    except FileNotFoundError: pass
    except Exception as e: log.warning(f'[TPSL] hist load: {e}')

def _spike_suggest_save():
    try:
        with open(SPIKE_SUGGEST_FILE,'w') as f: json.dump(_spike_tpsl_suggest,f)
    except Exception as e: log.debug(f'[TPSL] suggest save: {e}')

def _spike_downsample(xs, k):
    if len(xs)<=k: return list(xs)
    step=len(xs)/k
    return [xs[int(i*step)] for i in range(k)]

def _spike_path_outcome(prices, entry, direction, tp, sl):
    """Проиграть путь цены: что раньше — TP или SL. Тики тоньше 1-мин баров -> точнее свечного бэктеста."""
    if not prices or not entry: return 0.0
    for px in prices:
        if direction=='short': fav=(entry-px)/entry*100.0; adv=(px-entry)/entry*100.0
        else:                  fav=(px-entry)/entry*100.0; adv=(entry-px)/entry*100.0
        if adv>=sl: return -sl
        if fav>=tp: return tp
    last=prices[-1]
    return ((entry-last)/entry if direction=='short' else (last-entry)/entry)*100.0

def _spike_apply_tpsl(coin, market, tp, sl):
    """Применить новые TP/SL к паре в рантайме (мутируем SPIKE_COIN_PARAMS) + сохранить на диск."""
    try:
        SPIKE_COIN_PARAMS[(coin, market)] = (tp, sl)
    except Exception as e:
        log.warning(f'[TPSL] не удалось применить {coin}/{market}: {e}'); return False
    try:
        with _spike_hist_lock:
            applied={}
            try:
                with open(SPIKE_APPLIED_FILE) as f: applied=json.load(f)
            except FileNotFoundError: pass
            except Exception: applied={}
            applied[f'{coin}|{market}']={'tp':tp,'sl':sl,'ts':time.time()}
            with open(SPIKE_APPLIED_FILE,'w') as f: json.dump(applied,f)
    except Exception as e:
        log.debug(f'[TPSL] applied save: {e}')
    return True

def _spike_applied_load():
    """Подтянуть ранее применённые авто-TP/SL поверх SPIKE_COIN_PARAMS при старте."""
    try:
        with open(SPIKE_APPLIED_FILE) as f: applied=json.load(f)
    except FileNotFoundError: return
    except Exception as e: log.warning(f'[TPSL] applied load: {e}'); return
    n=0
    for key,v in applied.items():
        try:
            coin,market=key.split('|')
            SPIKE_COIN_PARAMS[(coin,market)]=(float(v['tp']),float(v['sl'])); n+=1
        except Exception: pass
    if n: log.info(f'[TPSL] применённые авто-TP/SL восстановлены из файла: {n} пар')

def _spike_analyze_tpsl(coin, market, apply_if_loss=False):
    """Грид-поиск TP/SL по записанным путям пары. При убытке (apply_if_loss) — применить сразу;
    при плюсе — только показать оптимум в логе."""
    key=f'{coin}|{market}'
    with _spike_hist_lock:
        paths=[h for h in _spike_trade_hist.get(key,[]) if h.get('prices')]
    if len(paths)<SPIKE_TPSL_MIN_TRADES:
        log.info(f'[TPSL] {coin}/{market}: мало сделок с тиками ({len(paths)}/{SPIKE_TPSL_MIN_TRADES}) — '
                 f'{"автоприменение отложено" if apply_if_loss else "анализ позже"}')
        return None
    fee=SPIKE_FEE_PCT
    try: cur_tp,cur_sl=_spike_pct(coin,market)
    except Exception: cur_tp,cur_sl=(1.5,4.5)
    def expect(tp,sl):
        return sum(_spike_path_outcome(h['prices'],h['entry'],h['dir'],tp,sl)-fee for h in paths)/len(paths)
    cur_e=expect(cur_tp,cur_sl); best=(cur_tp,cur_sl,cur_e)
    for tp in SPIKE_TPSL_TP_GRID:
        for sl in SPIKE_TPSL_SL_GRID:
            if sl<=tp: continue
            e=expect(tp,sl)
            if e>best[2]: best=(tp,sl,e)
    improved=(best[2]>cur_e+1e-9) and (best[0]!=cur_tp or best[1]!=cur_sl)
    wins=sum(1 for h in paths if h.get('net',0)>0); losses=len(paths)-wins
    head=(f'[TPSL] {coin}/{market} N={len(paths)} ({wins}+/{losses}-) | '
          f'текущий TP{cur_tp}/SL{cur_sl} эксп={cur_e:+.3f}% | '
          f'лучший TP{best[0]}/SL{best[1]} эксп={best[2]:+.3f}%')
    applied=False
    if improved and apply_if_loss and SPIKE_TPSL_AUTOAPPLY_ON_LOSS:
        applied=_spike_apply_tpsl(coin, market, best[0], best[1])
    if applied:
        log.warning(head + f' -> УБЫТОК: ПРИМЕНЕНО TP{best[0]}/SL{best[1]} (было TP{cur_tp}/SL{cur_sl})')
    elif improved:
        log.warning(head + (' -> РЕКОМЕНДАЦИЯ сменить (плюс: не применяю)' if not apply_if_loss
                            else ' -> РЕКОМЕНДАЦИЯ сменить (автоприменение выкл)'))
    else:
        log.warning(head + ' -> текущие близки к оптимуму')
    sug={'coin':coin,'market':market,'n':len(paths),'wins':wins,'losses':losses,
         'cur':[cur_tp,cur_sl],'cur_exp':round(cur_e,4),'best':[best[0],best[1]],
         'best_exp':round(best[2],4),'improved':improved,'applied':applied,'ts':time.time()}
    with _spike_hist_lock:
        _spike_tpsl_suggest[key]=sug; _spike_suggest_save()
    if improved:
        verb=('ПРИМЕНЕНО (был убыток)' if applied else
              ('подсказка по плюсовой сделке — применять вручную' if not apply_if_loss else
               'подсказка (автоприменение выкл)'))
        body=(head+f'\n\n{coin}/{market}: {verb}.\nN={len(paths)} сделок ({wins} плюс / {losses} минус).\n'
              f'Текущие TP{cur_tp}/SL{cur_sl} -> эксп {cur_e:+.3f}%/сделку.\n'
              f'Лучшие по истории TP{best[0]}/SL{best[1]} -> эксп {best[2]:+.3f}%.\n\n'
              f'ВНИМАНИЕ: выборка мала, риск подгонки под прошлое.')
        threading.Thread(target=spike_send_email, args=(f'TP/SL {coin}/{market}'+(' ПРИМЕНЕНО' if applied else ''), body), daemon=True).start()
    return sug

def _spike_postclose_job(coin, market, entry, direction, net_pct, tag, path_pts):
    """30 мин дозаписи цены ПОСЛЕ закрытия (REST), затем финализация истории + анализ TP/SL."""
    pts=list(path_pts)                       # цены ВО ВРЕМЯ позиции (из WS)
    t_end=time.time()+SPIKE_POSTCLOSE_SEC
    while time.time()<t_end:
        try:
            if _spike_stop.is_set(): break
        except Exception: pass
        time.sleep(SPIKE_POSTCLOSE_POLL_SEC)
        try:
            bid,ask=_spike_book(coin, market)
            px=((bid+ask)/2) if (bid and ask) else (bid or ask)
        except Exception:
            px=None
        if px: pts.append(float(px))
    # MFE/MAE по ПОЛНОМУ пути (позиция + 30 мин после)
    mfe=mae=0.0
    if entry:
        for px in pts:
            if direction=='short': fav=(entry-px)/entry*100; adv=(px-entry)/entry*100
            else:                  fav=(px-entry)/entry*100; adv=(entry-px)/entry*100
            if fav>mfe: mfe=fav
            if adv>mae: mae=adv
    rec={'ts':time.time(),'tag':tag,'dir':direction,'entry':entry,'net':round(net_pct,3),
         'mfe':round(mfe,3),'mae':round(mae,3),'pts':len(pts),'post':True,
         'prices':_spike_downsample(pts, SPIKE_TPSL_STORE_PTS)}
    key=f'{coin}|{market}'
    with _spike_hist_lock:
        lst=_spike_trade_hist.setdefault(key,[]); lst.append(rec)
        if len(lst)>SPIKE_HIST_MAX: del lst[0:len(lst)-SPIKE_HIST_MAX]
        _spike_hist_save()
    log.info(f'[TPSL] путь(+{SPIKE_POSTCLOSE_SEC//60}мин) {coin}/{market} {tag}: '
             f'MFE={mfe:.2f}% MAE={mae:.2f}% net={net_pct:+.2f}% точек={len(pts)}')
    _spike_analyze_tpsl(coin, market, apply_if_loss=(net_pct<0))

def _spike_record_close(coin, market, pos, net_pct, est_usdt, tag):
    """На закрытии снять путь и запустить фоновую дозапись 30 мин + последующий анализ."""
    if pos.get('dry'): return
    try:
        with _spike_path_lock:
            path=[p for _,p in _spike_path.pop((coin,market), [])]
    except Exception:
        path=[]
    entry=pos.get('entry_px') or 0; d=pos.get('direction','long')
    threading.Thread(target=_spike_postclose_job,
                     args=(coin, market, entry, d, net_pct, tag, path),
                     daemon=True, name=f'postclose-{coin}').start()
    log.info(f'[TPSL] {coin}/{market} {tag}: дозапись цены {SPIKE_POSTCLOSE_SEC//60} мин после закрытия запущена')

def spike_tpsl_report():
    """Показать накопленные подсказки TP/SL по всем парам."""
    if not _spike_tpsl_suggest:
        print(f'Подсказок пока нет (нужно >={SPIKE_TPSL_MIN_TRADES} сделок с тиками и хотя бы один минус по паре).'); return
    for key,s in sorted(_spike_tpsl_suggest.items()):
        flag=' <- СМЕНИТЬ' if s.get('improved') else ''
        print(f'{key}: N={s["n"]} ({s["wins"]}+/{s["losses"]}-) текущий TP{s["cur"][0]}/SL{s["cur"][1]} '
              f'эксп={s["cur_exp"]:+.3f}% | лучший TP{s["best"][0]}/SL{s["best"][1]} эксп={s["best_exp"]:+.3f}%{flag}')

_spike_hist_load()
_spike_applied_load()


# ── письмо при нехватке средств на вход (с троттлингом, иначе спам при пустом балансе) ──
SPIKE_NOBAL_EMAIL_EVERY_SEC = 900     # не чаще раза в 15 мин
_spike_nobal_last = 0.0
_spike_nobal_lock = threading.Lock()

def _spike_notify_no_balance(coin, market, direction):
    global _spike_nobal_last
    now=time.time()
    with _spike_nobal_lock:
        if now - _spike_nobal_last < SPIKE_NOBAL_EMAIL_EVERY_SEC:
            return
        _spike_nobal_last=now
    bal=None
    try:
        bal = get_bb_balance() if market=='bb_fut' else get_bg_balance()
    except Exception:
        bal=None
    bal_s = f'{bal:.4f} USDT' if isinstance(bal,(int,float)) else 'н/д'
    try: need=f'{HALF_USDT:.2f} USDT'
    except Exception: need='?'
    ex='Bybit' if market=='bb_fut' else 'Bitget'
    subj=f'НЕТ ДЕНЕГ: пропущен вход {coin}/{market}'
    body=(f'Спайк-бот не смог открыть позицию из-за нехватки средств.\n\n'
          f'Монета: {coin}/{market} ({direction})\nБиржа: {ex}\n'
          f'Доступно на бирже: {bal_s}\nНужно ~{need} на вход.\n\n'
          f'Сигнал пропущен. Пополни баланс {ex} или уменьши SPIKE_USDT.\n'
          f'(письма о нехватке шлются не чаще раза в {SPIKE_NOBAL_EMAIL_EVERY_SEC//60} мин)')
    threading.Thread(target=spike_send_email, args=(subj, body), daemon=True).start()
    log.warning(f'[SPIKE] НЕТ ДЕНЕГ на вход {coin}/{market} (доступно {bal_s}) -> письмо отправляется')


# ── ВОССТАНОВЛЕНИЕ ПОЗИЦИЙ ПРИ СТАРТЕ: подхватить висящие открытые позиции и доторговать ──
def _spike_recover_positions():
    """Опросить биржи на открытые позиции и зарегистрировать висящие (вайтлист) в _spike_positions,
    чтобы продолжить ими управлять после рестарта (TP + безубыток/12ч). Вне вайтлиста — только предупреждаем."""
    if SPIKE_DRY_RUN or MODE == 'SIMULATION':
        return []
    found = []
    # --- Bybit: все открытые linear-позиции ---
    try:
        rows = bybit_get('/v5/position/list', {'category': 'linear', 'settleCoin': 'USDT'})
        rows = rows if isinstance(rows, list) else (rows.get('list', []) if isinstance(rows, dict) else [])
        for r in rows:
            if float(r.get('size', 0) or 0) <= 0:
                continue
            sym = str(r.get('symbol', '')); coin = sym[:-4] if sym.endswith('USDT') else sym
            d = 'long' if str(r.get('side', '')).lower() == 'buy' else 'short'
            found.append(('bb_fut', coin, d, float(r.get('avgPrice', 0) or 0),
                          float(r.get('size', 0) or 0), int(r.get('createdTime', 0) or 0)))
    except Exception as e:
        log.warning(f'[SPIKE] восстановление: запрос Bybit-позиций не удался: {e}')
    # --- Bitget: все открытые USDT-FUTURES позиции ---
    try:
        dp = bitget_get('/api/v2/mix/position/all-position',
                        {'productType': 'USDT-FUTURES', 'marginCoin': 'USDT'}, signed=True)
        prows = dp.get('list', []) if isinstance(dp, dict) else (dp if isinstance(dp, list) else [])
        for r in prows:
            if float(r.get('total', 0) or 0) <= 0:
                continue
            sym = str(r.get('symbol', '')); coin = sym[:-4] if sym.endswith('USDT') else sym
            d = 'long' if str(r.get('holdSide', '')).lower() == 'long' else 'short'
            entry = float(r.get('openPriceAvg', 0) or r.get('averageOpenPrice', 0) or r.get('openAvgPrice', 0) or 0)
            found.append(('bg_fut', coin, d, entry,
                          float(r.get('total', 0) or 0), int(r.get('cTime', 0) or 0)))
    except Exception as e:
        log.warning(f'[SPIKE] восстановление: запрос Bitget-позиций не удался: {e}')

    now = time.time(); recovered = []; skipped = []
    for market, coin, d, entry, qty, ct in found:
        with _spike_lock:
            if (coin, market) in _spike_positions:      # уже ведём
                continue
        if SPIKE_WHITELIST_ON and (coin, market) not in SPIKE_COIN_PARAMS:
            skipped.append(f'{coin}/{market} {d}')        # не наша (напр. арб) -> не трогаем
            continue
        if not entry:
            log.warning(f'[SPIKE] восстановление {coin}/{market}: нет цены входа -> пропуск'); continue
        _spike_ensure_req(coin)        # ВАЖНО: загрузить спеки (price_place/qty_step) ДО постановки TP —
                                       # иначе при старте дефолт 5 знаков -> у OPG 45115 (тик 0.0001), у др. 110017
        opened = (ct / 1000.0) if ct else now
        # персистентность: состояние из файла — рестарт не обнуляет мартингейл и не теряет oid
        _sv = _spike_pos_load().get(f'{coin}|{market}') or {}
        if _sv.get('direction') != d:
            _sv = {}
        tp_off = _spike_pct(coin, market)[0] / 100.0
        tp_px = entry * (1 + tp_off) if d == 'long' else entry * (1 - tp_off)
        tp_oid = None; server_ok = False
        if _sv.get('tp_oid'):
            tp_oid = _sv['tp_oid']; tp_px = _sv.get('tp_px') or tp_px   # свой живой ордер: без дублей/40786
            server_ok = bool(_sv.get('server_tpsl'))
        else:
            # переустановить TP: в breakeven — GTC reduce-only лимит (как при открытии); в tpsl — серверный
            try:
                if SPIKE_EXIT_MODE == 'breakeven':
                    tp_oid, tp_px = _spike_tp(coin, market, d, qty, entry)
                elif market == 'bb_fut':
                    tp_px2, _sl2, server_ok = _spike_server_tpsl(coin, market, d, entry, qty)
                    tp_px = tp_px2 or tp_px
                else:
                    tp_oid, tp_px = _spike_tp(coin, market, d, qty, entry)
            except Exception as e:
                log.warning(f'[SPIKE] восстановление: TP {coin}/{market} не поставлен ({e}) — пойдёт безубыток')
        with _spike_lock:
            _spike_positions[(coin, market)] = {'coin': coin, 'market': market, 'direction': d,
                                                'entry_px': entry, 'entry0': _sv.get('entry0') or entry,
                                                'qty': qty, 'tp_oid': tp_oid,
                                                'tp_px': tp_px, 'sl_px': None,
                                                'rungs': int(_sv.get('rungs') or 1),
                                                'opened': float(_sv.get('opened') or opened),
                                                '_be_ts': _sv.get('_be_ts'), '_be_gtc': bool(_sv.get('_be_gtc')),
                                                '_last_rung_ts': _sv.get('_last_rung_ts'),
                                                'server_tpsl': server_ok,
                                                'recovered': True}
            _spike_cooldown[coin] = now
        if _sv:
            log.warning(f'[SPIKE] [STATE] {coin}/{market}: из файла — ступень {int(_sv.get("rungs") or 1)}/3, '
                        f'entry0={_sv.get("entry0") or entry}, oid={"свой" if _sv.get("tp_oid") else "новый"}')
        _spike_pos_save()
        age_min = (now - opened) / 60
        recovered.append(f'{coin}/{market} {d} @{entry} qty={qty} возраст={age_min:.0f}мин TP@{tp_px:.8g}')
        log.warning(f'[SPIKE] ВОССТАНОВЛЕНА {coin}/{market} {d} вход@{entry} qty={qty} '
                    f'возраст={age_min:.0f}мин -> доторговываем (TP@{tp_px:.8g}, далее безубыток/12ч)')

    if recovered:
        body = ('При запуске подхвачены открытые позиции (доторговка):\n\n' +
                '\n'.join('  ' + r for r in recovered))
        if skipped:
            body += '\n\nВне вайтлиста (НЕ управляем): ' + ', '.join(skipped)
        threading.Thread(target=spike_send_email,
                         args=(f'ВОССТАНОВЛЕНО позиций: {len(recovered)}', body), daemon=True).start()
    elif skipped:
        log.info(f'[SPIKE] старт: вайтлист-позиций нет; вне вайтлиста висят: {", ".join(skipped)}')
    else:
        log.info('[SPIKE] старт: открытых позиций нет')
    return recovered

log.info(f'[SPIKE] доп-модуль загружен: PnL-сверка + e-mail отчёт раз в {SPIKE_REPORT_EVERY_SEC//3600}ч '
         f'(spike_test_email() — проверить почту, spike_report_now() — отчёт сейчас)')


In [ ]:
spike_bg_diag()

## Запуск (видно вживую; стоп — кнопка ■)

In [ ]:
spike_run_foreground()
